# Занятие 5–6. SVM, снижение размерности и кластеризация

Семинар рассчитан на два занятия: разделы 1–3 разбирают на занятии 5,
разделы 4 и 5 — на занятии 6. Семинар состоит из четырех частей.

**Метод опорных векторов** (support vector machine, SVM). Линейный
классификатор, который выбирает разделяющую прямую с наибольшей шириной
полосы между классами. Затем нелинейный вариант: ядра позволяют строить
криволинейные границы без явного построения новых признаков.

**Метод главных компонент** (principal component analysis, PCA). Переход
к направлениям наибольшей дисперсии данных. Метод выводится из задачи
максимизации дисперсии проекций и применяется к изображениям цифр
MNIST (Modified National Institute of Standards and Technology database)
размером 28 × 28 пикселей.

**Визуализация многомерных данных.** Методы t-SNE (t-distributed stochastic
neighbor embedding, стохастическое вложение соседей с распределением
Стьюдента) и UMAP (uniform manifold approximation and projection,
равномерная аппроксимация и проекция многообразия) строят двумерное
представление данных, в котором сохраняются ближайшие соседи.

**Кластеризация.** Разбиение объектов на группы без известных ответов:
метрики качества, метод k-средних (k-means), агломеративная кластеризация
(agglomerative clustering), DBSCAN
(density-based spatial clustering of applications with noise, плотностная
кластеризация с выделением шума), смесь гауссовых распределений
(Gaussian mixture model, GMM) и их
сравнение на одних наборах данных.

Весь код в ноутбуке написан, ячейки выполняют по порядку. При первом
запуске нужен интернет: набор MNIST (около 50 МБ) загружается с OpenML
и далее хранится в кэше.

## 0. Подготовка

Функция `show_table` раскрашивает числовые столбцы таблицы: лучшее значение
выделено самым ярким цветом. Для столбцов, где лучше меньшее значение
(время, индекс Дэвиса — Болдина), шкала обращена; для таблицы, где все
значения такого рода, задают `lower_is_better=True`. При `axis=None` у всей
таблицы одна шкала.

In [ ]:
import itertools
import time
import warnings
from math import comb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import umap
import umap.umap_ as umap_internal
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Circle, Ellipse, Rectangle
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.linalg import eigh
from scipy.optimize import curve_fit
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, minimum_spanning_tree
from scipy.spatial.distance import pdist, squareform
from scipy.special import logsumexp

from sklearn.cluster import DBSCAN, HDBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import fetch_openml, load_wine, make_blobs, make_circles, make_moons
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.manifold import TSNE, trustworthiness
from sklearn.metrics import (
    adjusted_rand_score, calinski_harabasz_score, completeness_score, davies_bouldin_score,
    homogeneity_score, normalized_mutual_info_score, silhouette_samples, silhouette_score, v_measure_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVC, LinearSVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
CLUSTER_COLORS = np.array(["#0072CE", "#C98A3C", "#4DAF4A", "#984EA3", "#E41A1C",
                           "#FF7F00", "#A65628", "#F781BF", "#17BECF", "#6B7280"])
SEED = 42
LOWER_IS_BETTER = ("время", "секунд", "Дэвис")


def show_table(frame, digits=3, axis=0, lower_is_better=False):
    """Таблица с заливкой: лучшее значение ярче; для времени и индекса Дэвиса — Болдина лучше меньшее."""
    numeric = list(frame.select_dtypes("number").columns)
    lower = [c for c in numeric if lower_is_better or any(key in str(c) for key in LOWER_IS_BETTER)]
    higher = [c for c in numeric if c not in lower]
    styled = frame.style.format(precision=digits, subset=numeric)
    if higher:
        styled = styled.background_gradient(cmap="Greens", subset=higher, axis=axis)
    if lower:
        styled = styled.background_gradient(cmap="Greens_r", subset=lower, axis=axis)
    return styled

## 1. Метод опорных векторов (SVM)

### Отступ и функции потерь

Используются данные о винах из третьего занятия: сорт 1 против остальных,
два стандартизованных признака — флавоноиды и интенсивность цвета. Метки
классов кодируют как $y \in \{-1, +1\}$. Линейный классификатор вычисляет
$f(x) = \langle w, x \rangle + b$ и относит объект к классу $\mathrm{sign}\, f(x)$.

**Отступ** (margin) объекта:

$$M = y \cdot \big(\langle w, x \rangle + b\big).$$

При $M > 0$ объект классифицирован верно, при $M < 0$ — с ошибкой; модуль $M$
пропорционален расстоянию до границы. Функции потерь линейных
классификаторов записывают через отступ:

- доля ошибок (misclassification rate): $[M < 0]$;
- квадратичная функция потерь (squared loss) линейной регрессии на метках: $(1 - M)^2$;
- логистическая функция потерь (logistic loss): $\log_2(1 + e^{-M})$;
- кусочно-линейная функция потерь метода опорных векторов (hinge loss): $\max(0, 1 - M)$.

In [ ]:
margin_grid = np.linspace(-2.5, 3, 500)
loss_curves = {
    "доля ошибок, [M < 0]": ((margin_grid < 0).astype(float), GREY, 2),
    "квадратичная, (1 − M)²": ((1 - margin_grid) ** 2, BLACK, 1.5),
    "логистическая, log₂(1 + e⁻ᴹ)": (np.log2(1 + np.exp(-margin_grid)), BLUE, 2),
    "hinge loss, max(0, 1 − M)": (np.maximum(0, 1 - margin_grid), OCHRE, 3),
}
plt.figure(figsize=(8.5, 4.5))
for name, (values, color, width) in loss_curves.items():
    plt.plot(margin_grid, values, color=color, lw=width, label=name)
plt.axvspan(1, 3, color=OCHRE, alpha=0.08)
plt.text(1.1, 3.3, "M ≥ 1: hinge loss\nравна нулю", color=OCHRE, fontsize=10)
plt.axvline(0, color=GREY, lw=0.8)
plt.ylim(0, 4)
plt.xlabel("отступ M")
plt.ylabel("потери на объекте")
plt.legend(loc="upper right", fontsize=9)
plt.show()

Квадратичная функция потерь возрастает и при $M > 1$, то есть штрафует
объекты, классифицированные верно с большим запасом. Логистическая функция
потерь положительна при любом $M$: каждый объект вносит ненулевой вклад.
Hinge loss равна нулю при $M \geq 1$, поэтому объекты, лежащие дальше
единичного отступа, на решение не влияют. Это свойство приводит к понятию
опорных векторов.

In [ ]:
wine = load_wine(as_frame=True).frame
y_wine = (wine["target"] == 1).astype(int).to_numpy()
X_wine_raw = wine[["flavanoids", "color_intensity"]].to_numpy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_wine_raw, y_wine, test_size=0.3, random_state=SEED, stratify=y_wine
)
wine_scaler = StandardScaler().fit(X_train_raw)
X_train = wine_scaler.transform(X_train_raw)
X_test = wine_scaler.transform(X_test_raw)
print("обучение:", X_train.shape, " тестовый сет:", X_test.shape)

In [ ]:
def plot_points(ax, X, y, alpha=0.8):
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=26, color=GREY, alpha=alpha, label="другие сорта")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=26, color=BLUE, alpha=alpha, label="сорт 1")


def boundary_line(ax, X, w, b, color=BLACK, label=None, band=False, lw=2):
    xs = np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 100)
    ax.plot(xs, -(w[0] * xs + b) / w[1], color=color, lw=lw, label=label)
    if band:
        for shift in (-1, 1):
            ax.plot(xs, -(w[0] * xs + b - shift) / w[1], color=color, lw=1, ls="--")


def margins_of(X, y, w, b):
    margins = (2 * y - 1) * (X @ w + b)
    return margins

### Разделяющая полоса наибольшей ширины

Если классы разделимы, прямых без ошибок бесконечно много. Прямая, проходящая
вблизи объектов одного из классов, начинает ошибаться при малом сдвиге
данных. Поэтому выбирают прямую с наибольшим **зазором** (margin width) — расстоянием
до ближайшего объекта обучающей выборки. На рисунке две произвольные
прямые без ошибок и прямая метода опорных векторов на двух разделимых облаках.

In [ ]:
rng_svm = np.random.default_rng(7)
cloud_a = rng_svm.normal([0, 0], 0.7, (40, 2))
cloud_b = rng_svm.normal([3, 3], 0.7, (40, 2))
X_sep = np.vstack([cloud_a, cloud_b])
y_sep = np.r_[np.zeros(40, dtype=int), np.ones(40, dtype=int)]
svm_sep = LinearSVC(C=100, max_iter=100000).fit(X_sep, y_sep)
w_sep, b_sep = svm_sep.coef_[0], svm_sep.intercept_[0]


def gap_to_nearest(w, b):
    gap = (np.abs(X_sep @ w + b) / np.linalg.norm(w)).min()
    return gap


midpoint = np.array([1.5, 1.5])
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(cloud_a[:, 0], cloud_a[:, 1], s=22, color=GREY, label="класс −1")
ax.scatter(cloud_b[:, 0], cloud_b[:, 1], s=22, color=BLUE, label="класс +1")
xs = np.linspace(-2, 5, 100)
for name, w, color in [("прямая 1", np.array([0.25, 1.0]), "#7A63B8"), ("прямая 2", np.array([1.0, 0.25]), "#C7B45A")]:
    b = -w @ midpoint
    ax.plot(xs, -(w[0] * xs + b) / w[1], color=color, lw=1.6, label=f"{name}: зазор {gap_to_nearest(w, b):.2f}")
ax.plot(xs, -(w_sep[0] * xs + b_sep) / w_sep[1], color=OCHRE, lw=2.5, label=f"SVM: зазор {gap_to_nearest(w_sep, b_sep):.2f}")
for shift in (-1, 1):
    ax.plot(xs, -(w_sep[0] * xs + b_sep - shift) / w_sep[1], color=OCHRE, lw=1, ls="--")
support_sep = margins_of(X_sep, y_sep, w_sep, b_sep) <= 1 + 1e-6
ax.scatter(X_sep[support_sep, 0], X_sep[support_sep, 1], s=140, facecolor="none", edgecolor=BLACK, lw=1.3, label="опорные векторы")
ax.set_xlim(-2, 5)
ax.set_ylim(-2, 5)
ax.set_aspect("equal")
ax.set_title("Три прямые без ошибок с разным зазором")
ax.legend(loc="upper left", fontsize=9)
plt.show()

Положение прямой метода опорных векторов определяют только объекты на краях
полосы, обведенные черным. Их называют **опорными векторами** (support vectors).

### Постановка задачи

Пара $(w, b)$ задает границу с точностью до множителя: уравнения
$\langle w, x \rangle + b = 0$ и $\langle 2w, x \rangle + 2b = 0$ описывают одну прямую.
Множитель фиксируют условием: у ближайших к границе объектов отступ равен 1.
Тогда края полосы — прямые $\langle w, x \rangle + b = \pm 1$.

Вектор $w$ перпендикулярен границе. Пусть точка $x_1$ лежит на краю
$\langle w, x \rangle + b = 1$; ее сдвигают против направления $w$ на расстояние $d$,
в точку $x_1 - d \frac{w}{\|w\|}$, и подставляют в линейную функцию:

$$\Big\langle w,\ x_1 - d\frac{w}{\|w\|} \Big\rangle + b
= \underbrace{\langle w, x_1 \rangle + b}_{= 1} - d\,\frac{\langle w, w \rangle}{\|w\|}
= 1 - d\,\|w\|.$$

Выражение равно нулю (граница) при $d = \frac{1}{\|w\|}$ и равно $-1$ (противоположный
край) при $d = \frac{2}{\|w\|}$. Ширина полосы:

$$\text{ширина} = \frac{2}{\|w\|}.$$

Наибольшая ширина соответствует наименьшей норме $\|w\|$ при условиях
$M_i \geq 1$ для всех объектов. Если классы неразделимы, условия нарушаются;
за нарушение на объекте $i$ назначают штраф $\max(0, 1 - M_i)$. В записи,
принятой в sklearn:

$$\frac{1}{2}\|w\|^2 + C\sum_{i=1}^{n}\max(0, 1 - M_i) \to \min_{w, b}.$$

Параметр $C$ — цена одного нарушения. Опорными векторами в этой постановке
называют объекты с $M_i \leq 1$: на краю полосы, внутри нее и на чужой стороне.
Только они дают ненулевой вклад в сумму.

### Влияние параметра C

На рисунке граница и полоса при четырех значениях $C$ на данных о винах.
В заголовках — ширина полосы, число опорных векторов и accuracy на тестовом сете.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9.5))
for ax, C in zip(axes.flat, [0.01, 0.1, 1, 100]):
    model = LinearSVC(C=C, max_iter=100000).fit(X_train, y_train)
    w_c, b_c = model.coef_[0], model.intercept_[0]
    support_c = margins_of(X_train, y_train, w_c, b_c) <= 1 + 1e-6
    plot_points(ax, X_train, y_train, alpha=0.35)
    boundary_line(ax, X_train, w_c, b_c, color=OCHRE, band=True)
    ax.scatter(X_train[support_c, 0], X_train[support_c, 1], s=110, facecolor="none", edgecolor=BLACK, lw=1.0)
    ax.set_ylim(-2.2, 3.2)
    ax.set_title(f"C = {C}: ширина {2 / np.linalg.norm(w_c):.2f}, опорных {support_c.sum()}, "
                 f"accuracy {model.score(X_test, y_test):.3f}", fontsize=11)
plt.tight_layout()
plt.show()

При малом $C$ слагаемое $\frac{1}{2}\|w\|^2$ преобладает: норма весов мала,
полоса широкая, почти все объекты опорные. При большом $C$ преобладает сумма
штрафов: полоса сужается, число опорных векторов уменьшается. Значение $C$
выбирают кросс-валидацией.

### Зависимость границы от опорных векторов

Модель обучают заново дважды: только на опорных векторах и на всей выборке
без одного опорного вектора с наименьшим отступом.

In [ ]:
def fit_svm_line(mask):
    model = LinearSVC(C=1, max_iter=100000).fit(X_train[mask], y_train[mask])
    line = (model.coef_[0], model.intercept_[0])
    return line


w_all, b_all = fit_svm_line(np.ones(len(y_train), dtype=bool))
margins_all = margins_of(X_train, y_train, w_all, b_all)
support = margins_all <= 1 + 1e-6
w_support, b_support = fit_svm_line(support)
without_one = np.ones(len(y_train), dtype=bool)
without_one[np.argmin(margins_all)] = False
w_without, b_without = fit_svm_line(without_one)

fig, ax = plt.subplots(figsize=(7, 5.5))
plot_points(ax, X_train, y_train, alpha=0.3)
boundary_line(ax, X_train, w_all, b_all, color=OCHRE, label="вся выборка", lw=6)
boundary_line(ax, X_train, w_support, b_support, color=BLACK, label=f"только {support.sum()} опорных из {len(y_train)}")
boundary_line(ax, X_train, w_without, b_without, color=BLUE, label="без одного опорного вектора")
ax.scatter(*X_train[np.argmin(margins_all)], s=200, facecolor="none", edgecolor=BLUE, lw=2, label="исключенный объект")
ax.set_ylim(-2.2, 3.2)
ax.set_title("Граница зависит только от опорных векторов")
ax.legend(loc="upper right", fontsize=9)
plt.show()

Граница, построенная только по опорным векторам, совпадает с исходной:
у остальных объектов $M_i > 1$, их слагаемые в функции потерь равны нулю.
Исключение одного опорного вектора поворачивает границу. У логистической
регрессии вклад объекта убывает с ростом отступа, но не обращается в ноль,
поэтому на ее границу влияет каждый объект.

## 2. Нелинейная граница: ядра

Набор «полумесяцы» состоит из двух вложенных дуг; линейная граница
их не разделяет.

In [ ]:
X_moons, y_moons = make_moons(n_samples=400, noise=0.2, random_state=SEED)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(X_moons, y_moons, test_size=0.3, random_state=SEED, stratify=y_moons)


def plot_decision(ax, model, X, y, title):
    """Области классов, граница и опорные векторы."""
    grid_x, grid_y = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                                 np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))
    scores = model.decision_function(np.c_[grid_x.ravel(), grid_y.ravel()]).reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, scores > 0, levels=1, colors=["#E5E7EB", "#CFE3F7"], alpha=0.9)
    ax.contour(grid_x, grid_y, scores, levels=[0], colors=BLACK, linewidths=1.5)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=14, color=GREY)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=14, color=BLUE)
    if hasattr(model, "support_"):
        ax.scatter(*X[model.support_].T, s=45, facecolor="none", edgecolor=BLACK, lw=0.7)
    ax.set_title(title, fontsize=10)


linear_moons = SVC(kernel="linear", C=1).fit(Xm_train, ym_train)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
plot_decision(ax, linear_moons, Xm_train, ym_train, f"Линейный SVM: accuracy на тестовом сете {linear_moons.score(Xm_test, ym_test):.3f}")
plt.show()

### Новые признаки

К признакам добавляют их произведения и степени до третьей включительно:
$x_1^2, x_1 x_2, x_2^2, x_1^3, \dots$ В новом пространстве линейный SVM
строит гиперплоскость, а в исходных координатах ей соответствует кривая.

In [ ]:
poly_moons = make_pipeline(PolynomialFeatures(degree=3), StandardScaler(), LinearSVC(C=10, max_iter=100000))
poly_moons.fit(Xm_train, ym_train)
n_poly = poly_moons[0].n_output_features_
fig, ax = plt.subplots(figsize=(6.5, 4.5))
plot_decision(ax, poly_moons, Xm_train, ym_train, f"Признаки до третьей степени, {n_poly} шт.: accuracy {poly_moons.score(Xm_test, ym_test):.3f}")
plt.show()

Число одночленов степени не выше $d$ от $p$ переменных равно $\binom{p + d}{d}$.
Для $p = 2$, $d = 3$ это $\binom{5}{3} = 10$, для $p = 784$ пикселей цифры —
$\binom{787}{3} \approx 8 \cdot 10^7$. Ядра дают тот же результат без построения
этих признаков.

### Ядровой переход (kernel trick)

Задача SVM для разделимого случая: $\frac{1}{2}\|w\|^2 \to \min$ при
$y_i(\langle w, x_i \rangle + b) \geq 1$. Условия учитывают множителями Лагранжа (Lagrange multipliers)
$\alpha_i \geq 0$:

$$L(w, b, \alpha) = \frac{1}{2}\|w\|^2 - \sum_{i} \alpha_i \big(y_i(\langle w, x_i \rangle + b) - 1\big).$$

Производные по $w$ и $b$ приравнивают к нулю:

$$\frac{\partial L}{\partial w} = w - \sum_i \alpha_i y_i x_i = 0 \;\Rightarrow\; w = \sum_i \alpha_i y_i x_i,
\qquad \frac{\partial L}{\partial b} = -\sum_i \alpha_i y_i = 0.$$

Вектор весов — линейная комбинация обучающих объектов. В оптимуме $\alpha_i > 0$
только у объектов, для которых условие выполнено как равенство, то есть
у опорных векторов. Подстановка $w$ в модель дает

$$f(x) = \langle w, x \rangle + b = \sum_i \alpha_i y_i \langle x_i, x \rangle + b.$$

Подстановка $w$ в $L$: первое слагаемое $\frac{1}{2}\|w\|^2 = \frac{1}{2}\sum_{i,j}\alpha_i\alpha_j y_i y_j \langle x_i, x_j \rangle$;
сумма $\sum_i \alpha_i y_i \langle w, x_i \rangle$ равна той же двойной сумме без множителя
$\frac{1}{2}$; слагаемое с $b$ равно $b\sum_i \alpha_i y_i = 0$. Остается задача
относительно $\alpha$:

$$\sum_i \alpha_i - \frac{1}{2}\sum_{i,j}\alpha_i \alpha_j y_i y_j \langle x_i, x_j \rangle \to \max_{\alpha \geq 0}.$$

В обучении и в предсказании объекты входят только через скалярные
произведения. Если объекты отобразить в новое пространство $\varphi(x)$,
достаточно значений $K(x, z) = \langle \varphi(x), \varphi(z) \rangle$. Функцию $K$
называют **ядром** (kernel); во многих случаях ее вычисляют без построения $\varphi$.

Пример: $x, z \in \mathbb{R}^2$ и $K(x, z) = \langle x, z \rangle^2$. Раскрытие квадрата дает

$$(x_1 z_1 + x_2 z_2)^2 = x_1^2 z_1^2 + 2 x_1 x_2 z_1 z_2 + x_2^2 z_2^2
= \big\langle (x_1^2, \sqrt{2} x_1 x_2, x_2^2),\ (z_1^2, \sqrt{2} z_1 z_2, z_2^2) \big\rangle,$$

то есть $K$ — скалярное произведение в пространстве признаков
$\varphi(x) = (x_1^2, \sqrt{2} x_1 x_2, x_2^2)$. Равенство проверяют на двух случайных точках.

In [ ]:
x_point, z_point = np.random.default_rng(0).normal(size=(2, 2))


def phi(v):
    features = np.array([v[0] ** 2, np.sqrt(2) * v[0] * v[1], v[1] ** 2])
    return features


print(f"ядро <x, z>^2:              {(x_point @ z_point) ** 2:.6f}")
print(f"скалярное произведение phi: {phi(x_point) @ phi(z_point):.6f}")

В `SVC` чаще всего используют ядра:

- полиномиальное ядро (polynomial kernel) $K(x, z) = (\gamma \langle x, z \rangle + r)^d$, соответствующее всем
  одночленам степени до $d$;
- гауссово ядро, или ядро радиальной базисной функции (radial basis function,
  RBF), $K(x, z) = \exp\big(-\gamma \|x - z\|^2\big)$, равное 1 для совпадающих точек
  и стремящееся к нулю с ростом расстояния; соответствующее пространство
  $\varphi$ бесконечномерно.

Три ядра на одних данных и одном тестовом сете.

In [ ]:
kernel_models = {
    "линейное": SVC(kernel="linear", C=1),
    "полиномиальное, степень 3": SVC(kernel="poly", degree=3, coef0=1, C=1),
    "RBF": SVC(kernel="rbf", gamma=1, C=1),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, (name, model) in zip(axes, kernel_models.items()):
    model.fit(Xm_train, ym_train)
    plot_decision(ax, model, Xm_train, ym_train,
                  f"{name}: accuracy {model.score(Xm_test, ym_test):.3f}, опорных {len(model.support_)}")
plt.tight_layout()
plt.show()

### Параметры gamma и C

Значение RBF-ядра убывает в $e$ раз на расстоянии $\frac{1}{\sqrt{\gamma}}$, поэтому $\gamma$
задает радиус влияния объекта. При малом $\gamma$ влияние распространяется
далеко и граница гладкая; при большом $\gamma$ граница огибает отдельные
объекты. Параметр $C$ имеет тот же смысл, что и в линейном случае.
На рисунке строки соответствуют значениям $C$, столбцы — значениям $\gamma$;
в заголовках accuracy на обучающей выборке и на тестовом сете.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
for row, C in enumerate([0.1, 1, 100]):
    for col, gamma in enumerate([0.1, 1, 30]):
        model = SVC(kernel="rbf", C=C, gamma=gamma).fit(Xm_train, ym_train)
        plot_decision(axes[row, col], model, Xm_train, ym_train,
                      f"C = {C}, gamma = {gamma}: обучение {model.score(Xm_train, ym_train):.2f}, тест {model.score(Xm_test, ym_test):.2f}")
plt.tight_layout()
plt.show()

При $C = 100$, $\gamma = 30$ граница огибает отдельные объекты: accuracy на обучающей
выборке близка к 1, на тестовом сете ниже — модель переобучена. При $C = 0.1$,
$\gamma = 0.1$ граница близка к прямой — модель недообучена. Пару параметров
выбирают кросс-валидацией по сетке значений.

### Масштаб признаков

RBF-ядро зависит от евклидова расстояния, поэтому признак с большим
масштабом определяет расстояние почти целиком. Второй признак набора
умножают на 100 и сравнивают модель без стандартизации и со стандартизацией.

In [ ]:
stretch = np.array([1.0, 100.0])
raw_svm = SVC(kernel="rbf", gamma=1, C=1).fit(Xm_train * stretch, ym_train)
scaled_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", gamma=1, C=1)).fit(Xm_train * stretch, ym_train)
scales = pd.DataFrame({
    "без стандартизации": [raw_svm.score(Xm_test * stretch, ym_test)],
    "со StandardScaler": [scaled_svm.score(Xm_test * stretch, ym_test)],
}, index=["accuracy на тестовом сете"])
show_table(scales, axis=None)

Без стандартизации расстояние определяется вторым признаком, и accuracy
снижается. Перед SVM с RBF-ядром признаки стандартизуют внутри `Pipeline`.

## 3. Метод главных компонент (PCA)

### Дисперсия проекций

Пусть $X$ — матрица $n \times p$ с центрированными столбцами, $u$ — вектор единичной
длины. **Проекция** (projection) объекта $x_i$ на направление $u$ — число $z_i = \langle u, x_i \rangle$.
Если требуется заменить каждый объект одним числом, выбирают направление,
вдоль которого проекции имеют наибольшую дисперсию: в этом случае сохраняется
наибольшая часть различий между объектами.

На рисунке облако из 300 точек. На первых трех панелях точки спроецированы
на прямые с углами 0°, 90° и на прямую наибольшей дисперсии; проекции
показаны оранжевым. На четвертой панели — дисперсия проекций в зависимости
от угла направления; оранжевые точки соответствуют трем первым панелям.

In [ ]:
cloud = np.random.default_rng(1).multivariate_normal([0, 0], [[3.0, 1.6], [1.6, 1.4]], size=300)
cloud = cloud - cloud.mean(axis=0)
angles = np.deg2rad(np.arange(0, 180))
directions = np.c_[np.cos(angles), np.sin(angles)]
projection_variance = ((cloud @ directions.T) ** 2).mean(axis=0)
best_angle = angles[np.argmax(projection_variance)]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.3), gridspec_kw={"width_ratios": [1, 1, 1, 1.4]})
for ax, angle in zip(axes[:3], [0.0, np.pi / 2, best_angle]):
    u = np.array([np.cos(angle), np.sin(angle)])
    projected = (cloud @ u)[:, None] * u
    ax.scatter(cloud[:, 0], cloud[:, 1], s=6, color=GREY, alpha=0.6)
    ax.plot([-6 * u[0], 6 * u[0]], [-6 * u[1], 6 * u[1]], color=BLACK, lw=1)
    ax.scatter(projected[:, 0], projected[:, 1], s=6, color=OCHRE)
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_aspect("equal")
    ax.set_title(f"угол {np.rad2deg(angle):.0f}°: дисперсия {np.var(cloud @ u):.2f}", fontsize=10)
axes[3].plot(np.rad2deg(angles), projection_variance, color=BLUE, lw=2)
for angle in [0.0, np.pi / 2, best_angle]:
    axes[3].scatter(np.rad2deg(angle), np.var(cloud @ np.array([np.cos(angle), np.sin(angle)])), color=OCHRE, s=50, zorder=3)
axes[3].set_xlabel("угол направления, градусы")
axes[3].set_ylabel("дисперсия проекций")
axes[3].set_title("Дисперсия проекций от угла", fontsize=10)
plt.tight_layout()
plt.show()

Дисперсия проекций меняется с углом гладко и имеет один максимум
на полуокружности; ортогональное ему направление дает минимум. Положение
максимума находят аналитически.

### Вывод метода

Дисперсия проекций на направление $u$:

$$\frac{1}{n}\sum_{i=1}^{n} z_i^2 = \frac{1}{n}\sum_{i=1}^{n} u^\top x_i x_i^\top u = u^\top \Big(\frac{1}{n} X^\top X\Big) u = u^\top \Sigma u,$$

где $\Sigma = \frac{1}{n} X^\top X$ — ковариационная матрица (covariance matrix) признаков. Задача:
максимизировать $u^\top \Sigma u$ при условии $u^\top u = 1$. Функция Лагранжа
$u^\top \Sigma u - \lambda (u^\top u - 1)$, ее градиент по $u$:

$$2\Sigma u - 2\lambda u = 0 \;\Rightarrow\; \Sigma u = \lambda u.$$

Оптимальное направление — собственный вектор ковариационной матрицы.
Дисперсия вдоль него $u^\top \Sigma u = u^\top \lambda u = \lambda$, поэтому выбирают собственный
вектор с наибольшим собственным значением; его называют **первой главной
компонентой** (first principal component). Вторая главная компонента — направление наибольшей дисперсии
среди векторов, ортогональных первой, и так далее: главные компоненты —
собственные векторы $\Sigma$, упорядоченные по убыванию собственных значений.

Ковариационная матрица симметрична, поэтому раскладывается как
$\Sigma = V \Lambda V^\top$, где столбцы ортогональной матрицы $V$ — собственные векторы,
$\Lambda$ — диагональная матрица собственных значений. На практике раскладывают
саму матрицу $X$: **сингулярное разложение** (singular value decomposition, SVD)
$X = U S V^\top$, где $U$ и $V$ имеют ортонормированные столбцы, $S$ — диагональная
матрица сингулярных чисел $s_1 \geq s_2 \geq \dots \geq 0$. Из него

$$X^\top X = V S U^\top U S V^\top = V S^2 V^\top \;\Rightarrow\; \Sigma = V \, \frac{S^2}{n} \, V^\top,$$

то есть матрица $V$ в обоих разложениях одна и та же, а $\lambda_j = \frac{s_j^2}{n}$.

### Разложения на примере

Пример: 8 объектов, 5 признаков. Сначала разложение ковариационной матрицы
$\Sigma = V \Lambda V^\top$, затем сингулярное разложение центрированной матрицы
$X = U S V^\top$. В клетках — значения элементов.

In [ ]:
def show_product(matrices, titles, symbols, figsize):
    """Матрицы как тепловые карты с подписями и знаками между ними."""
    widths = []
    for k, matrix in enumerate(matrices):
        widths.append(matrix.shape[1])
        if k < len(symbols):
            widths.append(0.8)
    fig, axes = plt.subplots(1, len(widths), figsize=figsize, gridspec_kw={"width_ratios": widths})
    limit = max(np.abs(m).max() for m in matrices)
    max_rows = max(m.shape[0] for m in matrices)
    position = 0
    for k, (matrix, title) in enumerate(zip(matrices, titles)):
        ax = axes[position]
        rows, cols = matrix.shape
        pad = (max_rows - rows) / 2
        ax.imshow(matrix, cmap="RdBu", vmin=-limit, vmax=limit)
        for i in range(rows):
            for j in range(cols):
                ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=7)
        ax.add_patch(Rectangle((-0.5, -0.5), cols, rows, fill=False, lw=1.2, color=BLACK))
        ax.text((cols - 1) / 2, -0.8, title, ha="center", va="bottom", fontsize=10)
        ax.set_xlim(-0.5, cols - 0.5)
        ax.set_ylim(rows - 0.5 + pad, -0.5 - pad - 0.9)
        ax.axis("off")
        position += 1
        if k < len(symbols):
            axes[position].axis("off")
            axes[position].text(0.5, 0.5, symbols[k], ha="center", va="center", fontsize=28)
            position += 1
    plt.tight_layout()
    plt.show()


X_small = np.random.default_rng(3).normal(size=(8, 5)) @ np.random.default_rng(4).normal(size=(5, 5))
X_small = X_small - X_small.mean(axis=0)
sigma_small = X_small.T @ X_small / len(X_small)
eig_values, eig_vectors = np.linalg.eigh(sigma_small)
order = np.argsort(eig_values)[::-1]
eig_values, eig_vectors = eig_values[order], eig_vectors[:, order]
show_product([sigma_small, eig_vectors, np.diag(eig_values), eig_vectors.T],
             ["Σ, 5 × 5", "V: собственные векторы по столбцам", "Λ: собственные значения", "Vᵀ"],
             ["=", "·", "·"], figsize=(16, 3.4))

In [ ]:
U_small, s_small, Vt_small = np.linalg.svd(X_small, full_matrices=False)
show_product([X_small, U_small, np.diag(s_small), Vt_small],
             ["X, 8 × 5", "U, 8 × 5", "S: сингулярные числа", "Vᵀ, 5 × 5"],
             ["=", "·", "·"], figsize=(16, 4.2))
print("собственные значения Σ:", np.round(eig_values, 3))
print("s² / n:                ", np.round(s_small ** 2 / len(X_small), 3))
print("|<строка Vᵀ из SVD, столбец V из Σ>|:", np.round(np.abs((Vt_small * eig_vectors.T).sum(axis=1)), 3))

Строки $V^\top$ из сингулярного разложения совпадают со столбцами $V$ из разложения
ковариационной матрицы с точностью до знака, $\frac{s_j^2}{n}$ совпадают с собственными
значениями. Знак компоненты не определен: векторы $u$ и $-u$ задают одно
направление.

### Снижение размерности через сингулярное разложение

Пусть $r = \min(n, p)$. Оставляют первые $k < r$ слагаемых разложения: первые $k$
столбцов $U$ (матрица $U_k$ размера $n \times k$), левый верхний блок $S_k$ размера
$k \times k$ и первые $k$ строк $V^\top$ (матрица $V_k^\top$ размера $k \times p$). Получают
матрицу ранга $k$:

$$X_k = U_k S_k V_k^\top = (U_k S_k)\, V_k^\top.$$

Схема показывает, какие блоки сомножителей входят в $X_k$.

In [ ]:
def draw_matrix(ax, x, y, width, height, name, shape, keep=None, keep_label=None):
    """Прямоугольник матрицы; keep = ('cols' | 'rows' | 'block', доля) выделяет сохраняемую часть."""
    ax.add_patch(Rectangle((x, y), width, height, fill=False, lw=2, color=BLACK))
    ax.text(x + width / 2, y + height + 0.55, name, ha="center", fontsize=15)
    ax.text(x + width / 2, y + height + 0.2, shape, ha="center", fontsize=11, color=GREY)
    if keep is None:
        ax.text(x + width / 2, y + height / 2, keep_label, ha="center", va="center", fontsize=13)
        return
    kind, share = keep
    if kind == "cols":
        part = (x, y, width * share, height)
        ax.text(x + width * share / 2, y - 0.35, "k", ha="center", fontsize=12)
    elif kind == "rows":
        part = (x, y + height * (1 - share), width, height * share)
        ax.text(x + width + 0.15, y + height * (1 - share / 2), "k", va="center", fontsize=12)
    else:
        part = (x, y + height * (1 - share), width * share, height * share)
        ax.text(x + width * share + 0.15, y + height * (1 - share / 2), "k", va="center", fontsize=12)
        ax.text(x + width * share / 2, y + height * (1 - share) - 0.35, "k", ha="center", fontsize=12)
    ax.add_patch(Rectangle(part[:2], part[2], part[3], color=OCHRE, alpha=0.35, lw=0))
    ax.add_patch(Rectangle(part[:2], part[2], part[3], fill=False, lw=1.5, color=BLACK))
    ax.text(part[0] + part[2] / 2, part[1] + part[3] / 2, keep_label, ha="center", va="center", fontsize=13)


fig, ax = plt.subplots(figsize=(13, 5))
draw_matrix(ax, 0.0, 0, 2.4, 4, "X", "n × p", keep_label="Xₖ")
ax.text(3.0, 2.8, "≈", fontsize=26, ha="center", va="center")
draw_matrix(ax, 3.6, 0, 2.4, 4, "U", "n × r", keep=("cols", 0.4), keep_label="Uₖ")
ax.text(6.6, 2.8, "·", fontsize=26, ha="center", va="center")
draw_matrix(ax, 7.2, 1.6, 2.4, 2.4, "S", "r × r", keep=("block", 0.4), keep_label="Sₖ")
ax.text(10.2, 2.8, "·", fontsize=26, ha="center", va="center")
draw_matrix(ax, 10.8, 1.6, 2.4, 2.4, "Vᵀ", "r × p", keep=("rows", 0.4), keep_label="Vₖᵀ")
ax.set_xlim(-0.3, 13.8)
ax.set_ylim(-0.8, 5.2)
ax.set_aspect("equal")
ax.axis("off")
plt.show()

Смысл сомножителей устанавливают умножением разложения справа на $V$.
Так как $V^\top V = I$,

$$X V = U S V^\top V = U S,$$

а для первых $k$ столбцов $X V_k = U_k S_k$. Строка $i$ матрицы $X V_k$ содержит скалярные
произведения объекта $x_i$ с первыми $k$ главными компонентами, то есть его
проекции. Отсюда:

- $Z = U_k S_k = X V_k$ — матрица $n \times k$ новых координат объектов, результат `transform`;
- $V_k^\top$ — матрица $k \times p$, строки которой — главные компоненты, атрибут `components_`;
- $X_k = Z V_k^\top$ — восстановление объектов по $k$ координатам, результат `inverse_transform`
  без прибавления среднего.

Ошибка приближения равна сумме квадратов отброшенных сингулярных чисел:
$\|X - X_k\|_F^2 = \sum_{j > k} s_j^2$, так как $X - X_k = \sum_{j > k} s_j u_j v_j^\top$, а слагаемые
ортогональны. По теореме Эккарта — Янга (Eckart — Young theorem) никакая матрица ранга $k$ не дает
меньшей ошибки. Для матрицы примера при $k = 2$:

In [ ]:
k_small = 2
Z_small = U_small[:, :k_small] * s_small[:k_small]
X_small_k = Z_small @ Vt_small[:k_small]
show_product([Z_small, Vt_small[:k_small], X_small_k],
             ["Z = UₖSₖ, 8 × 2", "Vₖᵀ, 2 × 5", "Xₖ, 8 × 5"],
             ["·", "="], figsize=(8, 4.2))
print(f"||X − X_k||² = {((X_small - X_small_k) ** 2).sum():.4f}")
print(f"Σ s_j² при j > k = {(s_small[k_small:] ** 2).sum():.4f}")
print("X V_k совпадает с U_k S_k:", np.allclose(X_small @ Vt_small[:k_small].T, Z_small))

Тот же результат дает класс `PCA` из sklearn. На облаке точек сравнивают три
способа; sklearn делит на $n - 1$, а не на $n$, поэтому его дисперсии умножены
на $\frac{n - 1}{n}$. Совпадение направлений проверяют модулем косинуса угла
между компонентами: для единичных векторов он равен $|\langle u, v \rangle|$.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(cloud.T @ cloud / len(cloud))
by_eig = eigenvectors[:, ::-1].T
_, singular, vt = np.linalg.svd(cloud, full_matrices=False)
sk_pca = PCA(n_components=2).fit(cloud)

pca_check = pd.DataFrame({
    "собственные значения Σ": eigenvalues[::-1],
    "SVD: s² / n": singular ** 2 / len(cloud),
    "sklearn: explained_variance_ · (n−1)/n": sk_pca.explained_variance_ * (len(cloud) - 1) / len(cloud),
    "косинус угла: SVD и sklearn": np.abs((vt * sk_pca.components_).sum(axis=1)),
    "косинус угла: SVD и собственный вектор": np.abs((vt * by_eig).sum(axis=1)),
}, index=["компонента 1", "компонента 2"])
show_table(pca_check, digits=4)

### Компоненты и объясненная дисперсия

Для набора объектов с центром $\mu$ используют следующие величины:

- **главная компонента** (principal component) $u_j \in \mathbb{R}^p$ — единичный вектор, $j$-й собственный
  вектор ковариационной матрицы;
- **координата** (score) объекта по компоненте: $z_j = \langle u_j, x - \mu \rangle$;
- **дисперсия вдоль компоненты** $\lambda_j$ — выборочная дисперсия координат $z_j$,
  она равна $j$-му собственному значению;
- **доля объясненной дисперсии** (explained variance ratio) $\frac{\lambda_j}{\sum_{l=1}^{p} \lambda_l}$.

Знаменатель равен суммарной дисперсии всех исходных признаков. Действительно,
$\sum_l \lambda_l = \mathrm{tr}\,\Lambda = \mathrm{tr}(V^\top \Sigma V) = \mathrm{tr}(\Sigma V V^\top) = \mathrm{tr}\,\Sigma = \sum_{m=1}^{p} \mathrm{Var}(x_m)$:
переход к компонентам не меняет общую дисперсию, а перераспределяет ее так,
что первые компоненты получают наибольшую долю.

### Набор MNIST

Набор содержит 70 000 изображений рукописных цифр размером 28 × 28 пикселей;
каждое изображение — вектор из 784 яркостей от 0 до 1. Используют 12 000
изображений: 10 000 для обучения и 2000 для тестового сета.

In [ ]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X_digits_all = mnist.data.astype(np.float32) / 255
y_digits_all = mnist.target.astype(int)
X_digits, _, y_digits, _ = train_test_split(X_digits_all, y_digits_all, train_size=12000, random_state=SEED, stratify=y_digits_all)
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_digits, y_digits, test_size=2000, random_state=SEED, stratify=y_digits)
print("обучение:", Xd_train.shape, " тестовый сет:", Xd_test.shape)

In [ ]:
def show_digits(images, titles, n_cols=10, size=1.3):
    n_rows = int(np.ceil(len(images) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, size * 1.2 * n_rows))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, image, title in zip(np.atleast_1d(axes).ravel(), images, titles):
        ax.imshow(np.asarray(image).reshape(28, 28), cmap="gray_r")
        ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()


show_digits(Xd_train[:20], [str(label) for label in yd_train[:20]])

Метод `fit` класса `PCA` не подбирает параметры по ошибке, как при обучении
модели. Он вычисляет вектор средних $\mu$, сингулярное разложение центрированной
матрицы и по нему компоненты и дисперсии. Эти величины вычисляют
по обучающей выборке; тестовый сет в вычислении не участвует, как
и при стандартизации, иначе часть информации о нем попадает в преобразование.

Вычисляют все 784 компоненты и строят накопленную долю объясненной
дисперсии $\frac{\lambda_1 + \dots + \lambda_k}{\lambda_1 + \dots + \lambda_{784}}$ в зависимости от $k$.

In [ ]:
pca_digits = PCA().fit(Xd_train)
cumulative = np.cumsum(pca_digits.explained_variance_ratio_)
needed = {share: int(np.searchsorted(cumulative, share) + 1) for share in [0.8, 0.9, 0.95]}

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, 785), cumulative, color=BLUE, lw=2)
for (share, k), color in zip(needed.items(), [GREY, OCHRE, BLACK]):
    plt.axvline(k, color=color, ls="--", lw=1, label=f"{share:.0%} дисперсии: {k} компонент")
plt.xscale("log")
plt.xlabel("число компонент k")
plt.ylabel("накопленная доля дисперсии")
plt.legend()
plt.show()

Девяносто процентов суммарной дисперсии приходится менее чем на сто
компонент из 784: яркости соседних пикселей сильно коррелированы, а пиксели
у краев изображения почти всегда равны нулю и имеют малую дисперсию.

Каждая компонента $u_j$ — вектор из 784 чисел, его можно изобразить
как картинку 28 × 28. На рисунке вектор средних и первые восемь компонент;
синий цвет — положительные элементы, красный — отрицательные.

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(14, 2))
axes[0].imshow(pca_digits.mean_.reshape(28, 28), cmap="gray_r")
axes[0].set_title("среднее μ", fontsize=9)
for j, ax in enumerate(axes[1:]):
    component = pca_digits.components_[j].reshape(28, 28)
    limit = np.abs(component).max()
    ax.imshow(component, cmap="RdBu", vmin=-limit, vmax=limit)
    ax.set_title(f"u{j + 1}, λ = {pca_digits.explained_variance_[j]:.1f}", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Разложение изображения по компонентам

Компоненты образуют ортонормированный базис пространства $\mathbb{R}^{784}$, поэтому
любое изображение записывается точно:

$$x = \mu + \sum_{j=1}^{784} z_j u_j, \qquad z_j = \langle u_j, x - \mu \rangle.$$

Если оставить только первые $k$ слагаемых, получают приближение
$\hat{x}_k = \mu + \sum_{j=1}^{k} z_j u_j$ — строку матрицы $X_k$ из предыдущего раздела
с прибавленным средним. Средняя по выборке квадратичная ошибка приближения
равна сумме отброшенных дисперсий $\lambda_{k+1} + \dots + \lambda_{784}$, и по теореме
Эккарта — Янга первые $k$ компонент дают наименьшую ошибку среди всех
$k$-мерных приближений.

### В чем состоит снижение размерности

Изображения $u_j$ имеют тот же размер 28 × 28, что и исходное изображение, так как
это векторы того же пространства $\mathbb{R}^{784}$. Сжатие состоит в другом: вектор
средних $\mu$ и компоненты $u_1, \dots, u_k$ вычисляют один раз по обучающей выборке,
и они общие для всех изображений. Каждое отдельное изображение описывают
только вектором координат $z = (z_1, \dots, z_k)$ — $k$ числами вместо 784.

Для всей выборки разложение записывают в матричной форме. Пусть строки
матрицы $X$ размера $n \times 784$ — изображения, $V_k$ — матрица $784 \times k$
со столбцами $u_1, \dots, u_k$. Координаты всех изображений
$Z = (X - \mathbf{1}\mu^\top) V_k$ — матрица $n \times k$, и

$$\underbrace{X - \mathbf{1}\mu^\top}_{n \times 784} \approx \underbrace{Z}_{n \times k} \cdot \underbrace{V_k^\top}_{k \times 784}.$$

Это разложение $X_k = U_k S_k V_k^\top$ из раздела о сингулярном разложении с $Z = U_k S_k$.
Строка $i$ матрицы $Z$ — координаты изображения $i$; строки $V_k^\top$ — компоненты,
одни для всех строк $Z$. На схеме выделены одно изображение и его координаты.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
row_y, row_h = 2.6, 0.35
ax.add_patch(Rectangle((0, 0), 4.8, 4, fill=False, lw=2, color=BLACK))
ax.add_patch(Rectangle((0, row_y), 4.8, row_h, color=OCHRE, alpha=0.45, lw=0))
ax.text(2.4, row_y + row_h / 2, "изображение i: 784 яркости", ha="center", va="center", fontsize=11)
ax.text(2.4, 4.55, "X − μ", ha="center", fontsize=15)
ax.text(2.4, 4.2, "n × 784", ha="center", fontsize=11, color=GREY)
ax.text(5.4, 2, "≈", fontsize=26, ha="center", va="center")

ax.add_patch(Rectangle((6.0, 0), 0.9, 4, fill=False, lw=2, color=BLACK))
ax.add_patch(Rectangle((6.0, row_y), 0.9, row_h, color=OCHRE, alpha=0.45, lw=0))
ax.text(6.45, row_y + row_h / 2, "zᵢ", ha="center", va="center", fontsize=12)
ax.text(6.45, 4.55, "Z", ha="center", fontsize=15)
ax.text(6.45, 4.2, "n × k", ha="center", fontsize=11, color=GREY)
ax.text(6.45, -0.4, "k чисел\nна изображение", ha="center", va="top", fontsize=10)
ax.text(7.5, 2, "·", fontsize=30, ha="center", va="center")

strip_names = ["u₁ᵀ", "u₂ᵀ", "…", "uₖᵀ"]
for m, name in enumerate(strip_names):
    y = 3.1 - m * 0.3
    ax.add_patch(Rectangle((8.1, y), 4.8, 0.3, color=BLUE, alpha=0.12 + 0.08 * (m % 2), lw=0))
    ax.text(10.5, y + 0.15, name, ha="center", va="center", fontsize=11)
ax.add_patch(Rectangle((8.1, 2.2), 4.8, 1.2, fill=False, lw=2, color=BLACK))
ax.text(10.5, 4.55, "Vₖᵀ", ha="center", fontsize=15)
ax.text(10.5, 4.2, "k × 784", ha="center", fontsize=11, color=GREY)
ax.text(10.5, 1.9, "компоненты: вычисляют один раз,\nобщие для всех изображений", ha="center", va="top", fontsize=10)
ax.set_xlim(-0.3, 13.2)
ax.set_ylim(-1.2, 5.0)
ax.set_aspect("equal")
ax.axis("off")
plt.show()

Для одного изображения из тестового сета при $k = 50$: метод `transform`
вычисляет $z = V_k^\top (x - \mu)$ — 50 чисел, метод `inverse_transform`
восстанавливает $\hat{x} = \mu + V_k z$ — снова 784 яркости.

In [ ]:
digit = Xd_test[0]
coordinates = pca_digits.transform(digit[None, :])[0]
k_code = 50
digit_restored = pca_digits.mean_ + coordinates[:k_code] @ pca_digits.components_[:k_code]

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6), gridspec_kw={"width_ratios": [1, 0.55, 2.6, 0.7, 1]})
axes[0].imshow(digit.reshape(28, 28), cmap="gray_r")
axes[0].set_title("x: 784 яркости", fontsize=11)
axes[2].bar(np.arange(1, k_code + 1), coordinates[:k_code], color=BLUE)
axes[2].axhline(0, color=BLACK, lw=0.8)
axes[2].set_xlabel("номер компоненты j")
axes[2].set_ylabel("координата zⱼ")
axes[2].set_title(f"z = Vₖᵀ(x − μ): {k_code} чисел", fontsize=11)
axes[4].imshow(digit_restored.reshape(28, 28), cmap="gray_r")
axes[4].set_title("x̂ = μ + Vₖz: 784 яркости", fontsize=11)
for ax, label in [(axes[1], "transform"), (axes[3], "inverse_transform")]:
    ax.annotate("", xy=(0.95, 0.5), xytext=(0.05, 0.5), arrowprops={"arrowstyle": "->", "lw": 2, "color": BLACK})
    ax.text(0.5, 0.6, label, ha="center", va="bottom", fontsize=11)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
for ax in (axes[0], axes[1], axes[3], axes[4]):
    ax.axis("off")
plt.tight_layout()
plt.show()

Объем хранения выборки из $n$ изображений без сжатия — $784\,n$ чисел, после сжатия —
$k\,n$ координат и $784\,k + 784$ чисел компонент и среднего, которые не зависят от $n$.
В таблице строки — способы хранения обучающей выборки, столбцы — число
хранимых чисел.

In [ ]:
n_images = len(Xd_train)
storage = pd.DataFrame({
    "чисел на изображение": [784, k_code],
    "общих чисел (μ и компоненты)": [0, 784 * k_code + 784],
    "чисел всего": [784 * n_images, k_code * n_images + 784 * k_code + 784],
}, index=["исходные изображения", f"PCA, k = {k_code}"])
print(f"сжатие в {storage['чисел всего'].iloc[0] / storage['чисел всего'].iloc[1]:.1f} раза")
show_table(storage, digits=0, lower_is_better=True)

Общая часть хранится один раз, поэтому при большом $n$ коэффициент сжатия
стремится к $\frac{784}{k}$. Кроме хранения, сокращается число признаков
для последующих моделей: метод k ближайших соседей (k nearest neighbors) ниже вычисляет
расстояния между векторами $z$ из 50 чисел вместо 784.

### Изображение как сумма компонент

Восстановление $\hat{x} = \mu + V_k z$ — это сумма $\mu + z_1 u_1 + \dots + z_k u_k$.
На схеме изображение из тестового сета записано как сумма вектора средних
и первых пяти компонент с коэффициентами $z_j$.

In [ ]:
n_terms = 5
parts = [(digit, "x", "gray_r"), ("=", None, None), (pca_digits.mean_, "μ", "gray_r")]
for j in range(n_terms):
    sign = "+" if coordinates[j] >= 0 else "−"
    parts += [(f"{sign} {abs(coordinates[j]):.1f} ·", None, None), (pca_digits.components_[j], f"u{j + 1}", "RdBu")]
parts += [("+ …", None, None)]

fig, axes = plt.subplots(1, len(parts), figsize=(18, 2.6), gridspec_kw={"width_ratios": [3, 1] * (len(parts) // 2)})
for ax, (content, title, cmap) in zip(axes, parts):
    ax.axis("off")
    if isinstance(content, str):
        ax.text(0.5, 0.5, content, ha="center", va="center", fontsize=13)
        continue
    if cmap == "RdBu":
        limit = np.abs(content).max()
        ax.imshow(content.reshape(28, 28), cmap=cmap, vmin=-limit, vmax=limit)
    else:
        ax.imshow(content.reshape(28, 28), cmap=cmap)
    ax.set_title(title, fontsize=11)
plt.show()

Частичные суммы $\hat{x}_k$ при росте $k$ приближаются к исходному изображению.
Для того же изображения — приближения по 2, 10, 30, 100 и 300 компонентам.

In [ ]:
restored = [pca_digits.mean_ + coordinates[:k] @ pca_digits.components_[:k] for k in [2, 10, 30, 100, 300]]
show_digits([digit] + restored, ["исходное"] + [f"k = {k}" for k in [2, 10, 30, 100, 300]], n_cols=6, size=1.8)

Средняя по тестовому сету квадратичная ошибка восстановления в зависимости
от числа компонент.

In [ ]:
test_coordinates = pca_digits.transform(Xd_test)
k_grid = [1, 2, 5, 10, 20, 50, 100, 200, 400, 784]
errors = []
for k in k_grid:
    restored_test = pca_digits.mean_ + test_coordinates[:, :k] @ pca_digits.components_[:k]
    errors.append(((Xd_test - restored_test) ** 2).sum(axis=1).mean())
plt.figure(figsize=(8, 3.6))
plt.plot(k_grid, errors, "o-", color=BLUE)
plt.xscale("log")
plt.xlabel("число компонент k")
plt.ylabel("средняя ошибка ‖x − x̂‖²")
plt.title("Ошибка восстановления на тестовом сете")
plt.show()

### Изображения на плоскости первых двух компонент

Каждое изображение обучающей выборки заменяют парой координат $(z_1, z_2)$.
На статичном рисунке точки окрашены по цифре, поверх отдельных точек
показаны сами изображения в рамке цвета их цифры. Интерактивный рисунок
ниже показывает цифру и номер изображения при наведении курсора; щелчок
по цифре в легенде скрывает или показывает ее точки.

In [ ]:
def spread_indices(points, n_marks, min_share=0.07, seed=0):
    """Номера точек, разнесенных друг от друга не меньше чем на min_share размаха."""
    span = points.max(axis=0) - points.min(axis=0)
    chosen = []
    for index in np.random.default_rng(seed).permutation(len(points)):
        if all(np.linalg.norm((points[index] - points[other]) / span) > min_share for other in chosen):
            chosen.append(index)
        if len(chosen) == n_marks:
            break
    return chosen


def plot_digit_map(ax, points, labels, images, title, n_marks=40):
    for digit_class in range(10):
        mask = labels == digit_class
        ax.scatter(points[mask, 0], points[mask, 1], s=4, color=CLUSTER_COLORS[digit_class], alpha=0.5)
    for index in spread_indices(points, n_marks):
        thumb = OffsetImage(images[index].reshape(28, 28), cmap="gray_r", zoom=0.55)
        box = AnnotationBbox(thumb, points[index], frameon=True, pad=0.1,
                             bboxprops={"edgecolor": CLUSTER_COLORS[labels[index]], "lw": 1.5})
        ax.add_artist(box)
    handles = [Line2D([], [], marker="o", ls="", color=CLUSTER_COLORS[d], label=str(d)) for d in range(10)]
    ax.legend(handles=handles, title="цифра", loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])


def interactive_digit_map(points, labels, title, axis_names=("ось 1", "ось 2")):
    frame = pd.DataFrame({axis_names[0]: points[:, 0], axis_names[1]: points[:, 1],
                          "цифра": labels.astype(str), "номер изображения": np.arange(len(labels))})
    figure = px.scatter(frame, x=axis_names[0], y=axis_names[1], color="цифра",
                        hover_data={"цифра": True, "номер изображения": True},
                        category_orders={"цифра": [str(d) for d in range(10)]},
                        color_discrete_sequence=list(CLUSTER_COLORS), opacity=0.7,
                        width=850, height=650, title=title)
    figure.update_traces(marker_size=5)
    figure.show()


plane = pca_digits.transform(Xd_train[:3000])[:, :2]
labels_plane = yd_train[:3000]
fig, ax = plt.subplots(figsize=(11, 9))
plot_digit_map(ax, plane, labels_plane, Xd_train[:3000],
               f"MNIST на плоскости первых двух главных компонент: {cumulative[1]:.0%} дисперсии")
ax.set_xlabel("z₁")
ax.set_ylabel("z₂")
plt.show()

In [ ]:
interactive_digit_map(plane, labels_plane, "MNIST на плоскости первых двух главных компонент", ("z₁", "z₂"))

Вдоль первой компоненты отделяются нули и единицы, остальные цифры
перекрываются: две компоненты из 784 сохраняют лишь малую долю дисперсии,
указанную в заголовке рисунка.

### PCA перед классификатором

Метод k ближайших соседей (k nearest neighbors, kNN) вычисляет расстояния
в 784-мерном пространстве. Изображения сжимают до $k$ компонент и сравнивают
accuracy и время предсказания на одном тестовом сете. PCA входит в `Pipeline`
и вычисляется только по обучающей выборке.

In [ ]:
pca_rows = {}
for k in [5, 10, 20, 50, 100, 200, 784]:
    steps = [KNeighborsClassifier(n_neighbors=5)] if k == 784 else [PCA(n_components=k, random_state=SEED), KNeighborsClassifier(n_neighbors=5)]
    model = make_pipeline(*steps).fit(Xd_train, yd_train)
    start = time.perf_counter()
    accuracy = model.score(Xd_test, yd_test)
    pca_rows[k] = {"accuracy": accuracy, "время предсказания, с": time.perf_counter() - start}
pca_table = pd.DataFrame(pca_rows).T.rename_axis("компонент")
show_table(pca_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(pca_table.index, pca_table["accuracy"], "o-", color=BLUE)
axes[0].set_ylabel("accuracy на тестовом сете")
axes[1].plot(pca_table.index, pca_table["время предсказания, с"], "o-", color=OCHRE)
axes[1].set_ylabel("время предсказания, с")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("число компонент; 784 — без PCA")
plt.tight_layout()
plt.show()

При 50 компонентах accuracy не ниже, чем на исходных пикселях, а предсказание
выполняется быстрее. Отброшенные компоненты имеют малую дисперсию
и описывают в основном индивидуальные особенности почерка; без них
расстояния между изображениями лучше отражают различие цифр. При пяти
компонентах сохраняется слишком малая доля дисперсии, и accuracy падает.

## 4. Визуализация многомерных данных

Задачу визуализации ставят как задачу вложения (embedding): для объектов
$x_1, \dots, x_n \in \mathbb{R}^m$ находят точки $y_1, \dots, y_n \in \mathbb{R}^2$ так, чтобы
объекты, близкие в исходном пространстве, остались близкими на плоскости.
Метод главных компонент решает другую задачу: линейная проекция сохраняет
наибольшую дисперсию, то есть крупные расстояния, а ближайших соседей
сохраняет плохо. Методы t-SNE и UMAP формулируют сохранение соседей
как задачу оптимизации по координатам $y_i$.

Обозначения раздела:

- $n$ — число объектов выборки, $m$ — размерность исходного пространства;
- $x_i \in \mathbb{R}^m$ — объекты, $y_i \in \mathbb{R}^2$ — их образы на плоскости,
  $Y$ — матрица $n \times 2$, строки которой — образы;
- $\|\cdot\|$ — евклидова норма.

Методы применяют к 2500 изображениям обучающей выборки, предварительно
сжатым PCA до 50 компонент ($m = 50$). Сжатие уменьшает время вычисления
расстояний и удаляет компоненты с малой дисперсией.

In [ ]:
vis_X, vis_y = Xd_train[:2500], yd_train[:2500]
vis_pca50 = PCA(n_components=50, random_state=SEED).fit_transform(vis_X)
print("объектов:", vis_pca50.shape[0], " признаков после PCA:", vis_pca50.shape[1])

In [ ]:
def plot_labeled_map(ax, points, labels, title):
    """Точки, окрашенные по цифре, и номер цифры в медиане ее точек."""
    for digit_class in range(10):
        mask = labels == digit_class
        ax.scatter(points[mask, 0], points[mask, 1], s=3, color=CLUSTER_COLORS[digit_class], alpha=0.6)
        center = np.median(points[mask], axis=0)
        ax.text(center[0], center[1], str(digit_class), fontsize=13, weight="bold", ha="center", va="center",
                bbox={"facecolor": "white", "alpha": 0.7, "edgecolor": "none", "pad": 1})
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

### Концентрация расстояний в пространстве высокой размерности

Перед выводом методов устанавливают, какие расстояния двумерное представление
способно сохранить. Пусть $x, z$ — независимые случайные векторы из $\mathcal{N}(0, I_m)$:
все их координаты $x_l$, $z_l$ независимы и имеют распределение $\mathcal{N}(0, 1)$. Координата
разности $x_l - z_l$ — сумма двух независимых нормальных величин, поэтому она
нормальна, имеет нулевое среднее и дисперсию

$$\mathrm{Var}(x_l - z_l) = \mathrm{Var}\,x_l + \mathrm{Var}\,z_l = 1 + 1 = 2,$$

то есть $x - z \sim \mathcal{N}(0, 2I_m)$. Координату нормируют на ее стандартное отклонение $\sqrt{2}$:
величина $\xi_l = \frac{x_l - z_l}{\sqrt{2}}$ имеет дисперсию $\frac{2}{2} = 1$, следовательно, $\xi_l \sim \mathcal{N}(0, 1)$.
Величины $\xi_1, \dots, \xi_m$ независимы, так как каждая зависит только от своей пары
координат. Подстановка $x_l - z_l = \sqrt{2}\,\xi_l$:

$$\|x - z\|^2 = \sum_{l=1}^{m} (x_l - z_l)^2 = \sum_{l=1}^{m} 2\xi_l^2.$$

Моменты стандартной нормальной величины: $E\,\xi^2 = 1$, $E\,\xi^4 = 3$, откуда
$\mathrm{Var}\,\xi^2 = E\,\xi^4 - (E\,\xi^2)^2 = 2$. Слагаемые независимы, поэтому

$$E\,\|x - z\|^2 = \sum_{l=1}^{m} 2\,E\,\xi_l^2 = 2m, \qquad
\mathrm{Var}\,\|x - z\|^2 = \sum_{l=1}^{m} 4\,\mathrm{Var}\,\xi_l^2 = 8m.$$

Относительный разброс квадрата расстояния:

$$\frac{\sqrt{\mathrm{Var}\,\|x - z\|^2}}{E\,\|x - z\|^2} = \frac{\sqrt{8m}}{2m} = \sqrt{\frac{2}{m}}.$$

С ростом размерности квадраты расстояний концентрируются около среднего:
при $m = 2$ разброс равен среднему, при $m = 50$ составляет 20% от него.

Второе проявление того же свойства — распределение объема. Объем шара
радиуса $r$ в $\mathbb{R}^m$ равен $V_m(r) = c_m r^m$, поэтому доля объема единичного шара
в слое между радиусами $1 - \delta$ и 1:

$$\frac{V_m(1) - V_m(1 - \delta)}{V_m(1)} = \frac{c_m - c_m (1 - \delta)^m}{c_m} = 1 - (1 - \delta)^m.$$

При $\delta = 0.1$ это 19% для $m = 2$ и 99.5% для $m = 50$: почти весь объем находится
у границы, и почти все точки шара удалены от центра на близкие расстояния.

На левом рисунке — относительный разброс квадратов попарных расстояний
для гауссовых данных разной размерности (точки) и формула $\sqrt{2/m}$ (линия);
горизонтальные линии — значения для изображений MNIST в пикселях и после PCA.
На правом рисунке — доля объема в слое при $\delta = 0.1$.

In [ ]:
dimensions = [2, 5, 10, 20, 50, 100, 200, 500, 784]
relative_spread = []
for m in dimensions:
    first, second = np.random.default_rng(0).normal(size=(2, 3000, m))
    squared = ((first - second) ** 2).sum(axis=1)
    relative_spread.append(squared.std() / squared.mean())
mnist_spread = {}
for name, data in [("MNIST, пиксели", Xd_train[:1000]), ("MNIST, PCA-50", vis_pca50[:1000])]:
    squared = pdist(data, "sqeuclidean")
    mnist_spread[name] = squared.std() / squared.mean()

m_grid = np.logspace(np.log10(2), np.log10(784), 200)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot(m_grid, np.sqrt(2 / m_grid), color=BLUE, lw=2, label="√(2/m)")
axes[0].scatter(dimensions, relative_spread, color=BLACK, s=30, zorder=3, label="гауссовы данные")
for (name, value), color in zip(mnist_spread.items(), [OCHRE, "#4DAF4A"]):
    axes[0].axhline(value, color=color, ls="--", lw=1.5, label=f"{name}: {value:.2f}")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("размерность m")
axes[0].set_ylabel("std / среднее квадрата расстояния")
axes[0].legend(fontsize=9)
axes[1].plot(m_grid, 1 - 0.9 ** m_grid, color=BLUE, lw=2)
axes[1].set_xscale("log")
axes[1].set_xlabel("размерность m")
axes[1].set_ylabel("доля объема в слое [0.9, 1]")
plt.tight_layout()
plt.show()
for name, value in mnist_spread.items():
    print(f"{name}: размерность гауссовых данных с тем же разбросом 2 / ratio² = {2 / value ** 2:.0f}")

Точки — эмпирические значения, они совпадают с формулой. Изображения MNIST
имеют такой же разброс, как гауссовы данные размерности около двадцати,
а не 784 или 50: изображения лежат вблизи множества существенно меньшей
размерности, чем число пикселей.

**Проблема скученности** (crowding problem). В $\mathbb{R}^m$ можно расположить $m + 1$
точку на одинаковых попарных расстояниях — вершины правильного симплекса,
на плоскости — только три. Кроме того, число объектов на расстоянии около $r$
от данного объекта растет как площадь сферы, то есть как $r^{m-1}$, а на плоскости —
как длина окружности, то есть как $r$. Поэтому объекты, умеренно удаленные
от $x_i$, на плоскости на тех же расстояниях не помещаются, и сохранить все
расстояния невозможно. Методы визуализации сохраняют упорядочение ближайших
соседей, а умеренные расстояния растягивают.

### Стохастическое вложение соседей (stochastic neighbor embedding, SNE)

Метод t-SNE получен изменением метода SNE (Хинтон и Ровейс, 2002). Для объекта $x_i$
задают распределение вероятностей на остальных объектах: объект $x_j$ выбирают
соседом с вероятностью

$$p_{j \mid i} = \frac{\exp\big(-\|x_i - x_j\|^2 / 2\sigma_i^2\big)}{\sum_{l \neq i} \exp\big(-\|x_i - x_l\|^2 / 2\sigma_i^2\big)}, \qquad p_{i \mid i} = 0.$$

Обозначают $d_{ij} = \|x_i - x_j\|^2$ и $\beta_i = \frac{1}{2\sigma_i^2}$. Тогда

$$p_{j \mid i} = \frac{e^{-\beta_i d_{ij}}}{Z_i}, \qquad Z_i = \sum_{l \neq i} e^{-\beta_i d_{il}}.$$

Это распределение Гиббса (Gibbs distribution) из статистической физики: вероятность
состояния $j$ с энергией $d_{ij}$ при обратной температуре $\beta_i$; $Z_i$ — статистическая
сумма (partition function). При высокой температуре (большое $\sigma_i$) распределение
близко к равномерному, при низкой вероятность сосредоточена на ближайших объектах.

На плоскости задают аналогичное распределение с фиксированной шириной $\sigma = 1/\sqrt{2}$:

$$q_{j \mid i} = \frac{\exp\big(-\|y_i - y_j\|^2\big)}{\sum_{l \neq i} \exp\big(-\|y_i - y_l\|^2\big)},$$

а координаты $y_i$ находят минимизацией суммы расхождений Кульбака — Лейблера
(Kullback — Leibler divergence, KL):

$$C_{\text{SNE}} = \sum_{i=1}^{n} \mathrm{KL}(P_i \,\|\, Q_i) = \sum_{i=1}^{n} \sum_{j \neq i} p_{j \mid i} \ln \frac{p_{j \mid i}}{q_{j \mid i}}.$$

**Неотрицательность расхождения.** Для распределений $P$ и $Q$ на одном множестве
по неравенству Йенсена для вогнутой функции $\ln$:

$$-\mathrm{KL}(P \,\|\, Q) = \sum_j p_j \ln \frac{q_j}{p_j} \le \ln \sum_j p_j \frac{q_j}{p_j} = \ln \sum_j q_j = \ln 1 = 0,$$

причем равенство достигается только при $q_j = p_j$ для всех $j$. Следовательно,
$\mathrm{KL}(P \,\|\, Q) \ge 0$, и минимум функции потерь соответствует совпадению распределений.

**Несимметричность.** Слагаемое $p \ln \frac{p}{q}$ при $p = 0.2$, $q = 0.001$ равно
$0.2 \ln 200 \approx 1.06$, а при $p = 0.001$, $q = 0.2$ равно $0.001 \ln 0.005 \approx -0.005$.
Разнесение близких объектов штрафуется сильно, сближение далеких почти
не штрафуется. Поэтому функция потерь сохраняет локальную структуру.

### Перплексия и подбор ширины ядра

Ширину $\sigma_i$ подбирают для каждого объекта отдельно. Энтропия распределения
$P_i = (p_{j \mid i})_j$ в битах:

$$H(P_i) = -\sum_{j} p_{j \mid i} \log_2 p_{j \mid i},$$

**перплексия** (perplexity):

$$\mathrm{Perp}(P_i) = 2^{H(P_i)}.$$

Если распределение равномерно на $k$ объектах, $p_{j \mid i} = \frac{1}{k}$, и подстановка дает

$$H = -\sum_{j=1}^{k} \frac{1}{k} \log_2 \frac{1}{k} = \log_2 k, \qquad \mathrm{Perp} = 2^{\log_2 k} = k.$$

Поэтому перплексию трактуют как эффективное число соседей: распределение
с перплексией $k$ имеет ту же неопределенность, что и равномерный выбор
из $k$ вариантов. Термин (от лат. perplexus — запутанный) пришел из теории
информации и языкового моделирования, где перплексия модели — эффективное
число равновероятных вариантов следующего слова. В статистической физике
аналогичная величина — число равновероятных микросостояний $W$ в формуле
Больцмана $S = k_B \ln W$: системе с энтропией $S$ соответствует $W = e^{S / k_B}$
равновероятных состояний, так же как распределению с энтропией $H$ бит
соответствует $2^H$ равновероятных исходов. Так как $p_{j \mid i}$ — распределение
Гиббса, перплексия равна экспоненте его энтропии в том же смысле.

**Монотонность.** Энтропию удобно вычислять в натуральных логарифмах:
$H_e = H \ln 2$, и $2^{H} = e^{H_e}$. Подстановка $\ln p_{j \mid i} = -\beta_i d_{ij} - \ln Z_i$ дает

$$H_e = -\sum_j p_{j \mid i} \big(-\beta_i d_{ij} - \ln Z_i\big) = \beta_i\,E[d] + \ln Z_i,$$

где $E[d] = \sum_j p_{j \mid i} d_{ij}$ — среднее по распределению $P_i$. Производные по $\beta_i$:

$$\frac{\partial \ln Z_i}{\partial \beta_i} = \frac{1}{Z_i} \sum_j (-d_{ij})\,e^{-\beta_i d_{ij}} = -E[d],
\qquad
\frac{\partial p_{j \mid i}}{\partial \beta_i} = \frac{-d_{ij} e^{-\beta_i d_{ij}}}{Z_i} - \frac{e^{-\beta_i d_{ij}}}{Z_i^2}\frac{\partial Z_i}{\partial \beta_i} = p_{j \mid i}\big(E[d] - d_{ij}\big),$$

$$\frac{\partial E[d]}{\partial \beta_i} = \sum_j d_{ij}\,p_{j \mid i}\big(E[d] - d_{ij}\big) = E[d]^2 - E[d^2] = -\mathrm{Var}(d).$$

Отсюда

$$\frac{\partial H_e}{\partial \beta_i} = E[d] + \beta_i\,\frac{\partial E[d]}{\partial \beta_i} + \frac{\partial \ln Z_i}{\partial \beta_i}
= E[d] - \beta_i\,\mathrm{Var}(d) - E[d] = -\beta_i\,\mathrm{Var}(d) \le 0.$$

Энтропия не возрастает с ростом $\beta_i$, а $\beta_i = \frac{1}{2\sigma_i^2}$ убывает с ростом $\sigma_i$;
следовательно, перплексия не убывает с ростом $\sigma_i$ и строго возрастает, если
расстояния до соседей не все равны. Предельные значения:

- при $\sigma_i \to 0$ вся вероятность сосредоточена на ближайшем объекте,
  $H \to 0$ и $\mathrm{Perp} \to 1$;
- при $\sigma_i \to \infty$ распределение становится равномерным на $n - 1$ объектах,
  и $\mathrm{Perp} \to n - 1$.

Поэтому для любого значения `perplexity` из интервала $(1, n - 1)$ уравнение
$\mathrm{Perp}(P_i) = $ `perplexity` имеет единственное решение $\sigma_i$, и его находят двоичным
поиском. Поиск ведут по $\ln \beta_i$ с целевым значением $H_e = \ln$ `perplexity`.
Перед вычислением экспонент из всех $d_{ij}$ вычитают наименьшее: числитель
и знаменатель $p_{j \mid i}$ умножаются на одну константу, распределение не меняется,
а переполнения и потери точности не возникает.

In [ ]:
def conditional_affinities(X, perplexity, n_steps=64):
    """Матрица условных вероятностей p_{j|i} и ширины sigma_i: двоичный поиск по ln beta_i."""
    squared = squareform(pdist(X, "sqeuclidean"))
    n = len(X)
    conditional = np.zeros((n, n))
    sigmas = np.zeros(n)
    target = np.log(perplexity)
    for i in range(n):
        others = np.arange(n) != i
        shifted = squared[i, others] - squared[i, others].min()
        low, high = -30.0, 10.0
        for _ in range(n_steps):
            log_beta = (low + high) / 2
            weights = np.exp(-np.exp(log_beta) * shifted)
            p = weights / weights.sum()
            entropy = -(p[p > 0] * np.log(p[p > 0])).sum()
            if entropy > target:
                low = log_beta
            else:
                high = log_beta
        conditional[i, others] = p
        sigmas[i] = np.sqrt(1 / (2 * np.exp(log_beta)))
    return conditional, sigmas


def perplexity_of(squared_distances, sigma):
    weights = np.exp(-(squared_distances - squared_distances.min()) / (2 * sigma ** 2))
    p = weights / weights.sum()
    entropy_bits = -(p[p > 0] * np.log2(p[p > 0])).sum()
    perplexity = 2 ** entropy_bits
    return perplexity

Свойства проверяют на игрушечных данных: 60 точек плотного облака
со стандартным отклонением 0.3 и 60 точек разреженного облака со стандартным
отклонением 1.2. Выбирают по одной точке в каждом облаке (звезды) и подбирают
для них $\sigma_i$ при `perplexity` = 10. На рисунке:

- левая и средняя панели — данные на плоскости; площадь каждой точки
  пропорциональна $p_{j \mid i}$ для выбранной точки, окружность имеет радиус $\sigma_i$,
  на левой панели плотное облако показано в увеличенном фрагменте;
- правая панель — перплексия как функция $\sigma$ для двух выбранных точек
  тех же цветов; горизонтальная линия — требуемое значение 10, вертикальные —
  найденные $\sigma_i$.

In [ ]:
rng_density = np.random.default_rng(5)
toy_points = np.vstack([rng_density.normal([0, 0], 0.3, (60, 2)), rng_density.normal([4, 0], 1.2, (60, 2))])
toy_conditional, toy_sigmas = conditional_affinities(toy_points, perplexity=10)
toy_squared = squareform(pdist(toy_points, "sqeuclidean"))
sigma_grid = np.logspace(-2.5, 1.5, 300)
chosen = [(0, "точка плотного облака", BLUE), (60, "точка разреженного облака", OCHRE)]

fig, axes = plt.subplots(1, 3, figsize=(21, 5.6), gridspec_kw={"width_ratios": [1, 1, 1.15]})
for ax, (i, name, color) in zip(axes[:2], chosen):
    views = [ax]
    if i == 0:
        zoom = ax.inset_axes([0.45, 0.5, 0.4, 0.4])
        zoom.set_xlim(toy_points[i, 0] - 0.6, toy_points[i, 0] + 0.6)
        zoom.set_ylim(toy_points[i, 1] - 0.6, toy_points[i, 1] + 0.6)
        zoom.set_xticks([])
        zoom.set_yticks([])
        zoom.set_title("увеличенный фрагмент", fontsize=9)
        ax.indicate_inset_zoom(zoom, edgecolor=GREY)
        views.append(zoom)
    for view in views:
        view.scatter(toy_points[:, 0], toy_points[:, 1], s=8 + 1500 * toy_conditional[i], color=color, alpha=0.45)
        view.scatter(*toy_points[i], s=220, color=color, marker="*", edgecolor=BLACK, lw=0.8, zorder=3)
        view.add_patch(Circle(toy_points[i], toy_sigmas[i], fill=False, color=color, lw=1.8))
        view.set_aspect("equal")
    ax.set_xlim(-2, 8)
    ax.set_ylim(-4, 4)
    ax.set_title(f"{name}: σ = {toy_sigmas[i]:.2f}", fontsize=11)

for i, name, color in chosen:
    others = np.delete(toy_squared[i], i)
    curve = [perplexity_of(others, sigma) for sigma in sigma_grid]
    axes[2].plot(sigma_grid, curve, color=color, lw=2, label=f"{name}: σ = {toy_sigmas[i]:.2f}")
    axes[2].axvline(toy_sigmas[i], color=color, ls=":", lw=1.5)
    print(f"{name}: σ = {toy_sigmas[i]:.3f}, Perp(σ) = {perplexity_of(others, toy_sigmas[i]):.3f}")
axes[2].axhline(10, color=GREY, ls="--", lw=1, label="perplexity = 10")
axes[2].axhline(len(toy_points) - 1, color=BLACK, ls="--", lw=1, label="предел n − 1 = 119")
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].set_xlabel("ширина ядра σ")
axes[2].set_ylabel("перплексия Perp(P_i)")
axes[2].set_title("Перплексия как функция σ", fontsize=11)
axes[2].legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.show()

Каждая кривая возрастает от 1 до $n - 1 = 119$, как установлено выше, и поэтому
пересекает уровень 10 ровно один раз: кривая точки плотного облака — при $\sigma = 0.11$,
кривая точки разреженного облака — при $\sigma = 0.81$. Значения различаются в семь раз:
в плотном облаке соседи близко, и десять эффективных соседей набираются
при малом $\sigma_i$, а в разреженном облаке для того же числа соседей ядро должно быть
шире; поэтому окружность радиуса $\sigma_i$ на средней панели больше, чем на левой.

На кривой точки плотного облака есть участок со значениями от 50 до 60
при $\sigma$ примерно от 0.4 до 1.2: при таких $\sigma$ распределение почти равномерно на своем
облаке из 60 точек и еще не охватывает второе облако; рост продолжается, когда
ядро достигает второго облака.

### Симметризация

Метод t-SNE заменяет $n$ условных распределений одним совместным распределением
на парах объектов:

$$p_{ij} = \frac{p_{j \mid i} + p_{i \mid j}}{2n}.$$

**Нормировка.** Каждое условное распределение в сумме дает 1, поэтому
$\sum_i \sum_{j \neq i} p_{j \mid i} = n$ и $\sum_i \sum_{j \neq i} p_{i \mid j} = n$; отсюда

$$\sum_{i \neq j} p_{ij} = \frac{n + n}{2n} = 1.$$

**Вклад каждого объекта.** Сумма сходств объекта $i$ со всеми остальными:

$$\sum_{j \neq i} p_{ij} = \frac{1}{2n}\Big(\underbrace{\sum_{j \neq i} p_{j \mid i}}_{= 1} + \sum_{j \neq i} p_{i \mid j}\Big) \ge \frac{1}{2n}.$$

Средняя сумма равна $\frac{1}{n}$, и каждый объект получает не меньше ее половины.
Если бы совместное распределение задавали одной гауссовой функцией
$p_{ij} \propto \exp(-\|x_i - x_j\|^2 / 2\sigma^2)$ с общей шириной, у выброса все $p_{ij}$ были бы
близки к нулю, и его положение на плоскости почти не влияло бы на функцию
потерь. Симметризация обеспечивает каждому объекту ненулевое влияние.

Матрицу $P$ вычисляют для 2500 изображений при `perplexity` = 30 и проверяют
эти свойства. На рисунке слева — распределение величины $n \sum_j p_{ij}$ (линии — граница
$\frac{1}{2}$ и среднее 1), справа — распределение найденных $\sigma_i$.

In [ ]:
start = time.perf_counter()
vis_conditional, vis_sigmas = conditional_affinities(vis_pca50, perplexity=30)
P_vis = (vis_conditional + vis_conditional.T) / (2 * len(vis_pca50))
affinity_seconds = time.perf_counter() - start

safe = np.where(vis_conditional > 0, vis_conditional, 1.0)
row_perplexity = 2 ** (-(vis_conditional * np.log2(safe)).sum(axis=1))
point_mass = len(P_vis) * P_vis.sum(axis=1)
print(f"время: {affinity_seconds:.1f} с")
print(f"перплексия строк: от {row_perplexity.min():.4f} до {row_perplexity.max():.4f}")
print(f"сумма всех p_ij: {P_vis.sum():.6f}")
print(f"n · Σ_j p_ij: наименьшее {point_mass.min():.3f}, граница 0.5")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].hist(point_mass, bins=50, color=BLUE)
for value, style in [(0.5, "--"), (1.0, ":")]:
    axes[0].axvline(value, color=BLACK, ls=style, lw=1)
axes[0].set_xlabel("n · Σⱼ pᵢⱼ")
axes[0].set_ylabel("число объектов")
axes[1].hist(vis_sigmas, bins=50, color=OCHRE)
axes[1].set_xlabel("σᵢ")
axes[1].set_ylabel("число объектов")
plt.tight_layout()
plt.show()

Перплексия каждой строки равна 30 с точностью двоичного поиска, сумма $p_{ij}$
равна 1, и ни один объект не получает меньше половины среднего веса.
Ширины $\sigma_i$ различаются в несколько раз: двоичный поиск приспосабливает
ядро к локальной плотности данных.

### Функция потерь t-SNE, ее градиент и оптимизация

Функция потерь t-SNE — расхождение Кульбака — Лейблера между распределением $P$
на парах объектов и распределением $Q$ на парах образов:

$$\boxed{\;C(Y) = \mathrm{KL}(P \,\|\, Q) = \sum_{i \neq j} p_{ij} \ln \frac{p_{ij}}{q_{ij}}, \qquad
q_{ij} = \frac{\big(1 + \|y_i - y_j\|^2\big)^{-1}}{\sum_{l \neq m} \big(1 + \|y_l - y_m\|^2\big)^{-1}}\;}$$

Ее градиент по образу $y_i$, выведенный ниже:

$$\boxed{\;\frac{\partial C}{\partial y_i} = 4 \sum_{j \neq i} \big(p_{ij} - q_{ij}\big)\big(1 + \|y_i - y_j\|^2\big)^{-1}(y_i - y_j)\;}$$

Координаты $Y$ находят градиентным спуском по этой формуле. Далее последовательно
рассматривают выбор функции сходства $q_{ij}$, вывод градиента, его интерпретацию
и вычисление, схему оптимизации.

**Сходство на плоскости.** t-SNE задает совместное распределение образов через
распределение Стьюдента (Student's t-distribution) с одной степенью свободы:

$$w_{ij} = \big(1 + \|y_i - y_j\|^2\big)^{-1}, \qquad q_{ij} = \frac{w_{ij}}{\sum_{l \neq m} w_{lm}}, \qquad Z = \sum_{l \neq m} w_{lm}.$$

Выбор объясняется проблемой скученности. Пусть сходство пары равно $s$.
Расстояние, при котором гауссова функция $e^{-d^2/2}$ принимает значение $s$,
находят из $e^{-d^2/2} = s$, а для функции Стьюдента — из $(1 + d^2)^{-1} = s$:

$$d_G(s) = \sqrt{-2 \ln s}, \qquad d_t(s) = \sqrt{\frac{1}{s} - 1}.$$

При $s \to 0$ величина $\frac{1}{s}$ растет быстрее, чем $\ln \frac{1}{s}$, поэтому
$\frac{d_t(s)}{d_G(s)} \to \infty$: малым сходствам на плоскости соответствуют гораздо большие
расстояния. Умеренно далекие пары, которым в исходном пространстве не хватает
места, на плоскости раздвигаются, и между группами близких объектов
образуются промежутки. Для уровня $s = 0.1$: $d_G = \sqrt{2 \ln 10} \approx 2.15$, $d_t = \sqrt{9} = 3$.

Второе свойство: при больших $d$ функция $(1 + d^2)^{-1} \approx d^{-2}$ степенная, поэтому
взаимодействие далеких групп почти не зависит от масштаба рисунка. Третье —
вычисление $w_{ij}$ не требует экспоненты.

**Вывод градиента.** От координат $Y$ зависит только $q_{kl} = \frac{w_{kl}}{Z}$. Раскрытие логарифма:

$$C = \sum_{k \neq l} p_{kl} \ln p_{kl} - \sum_{k \neq l} p_{kl} \ln w_{kl} + \ln Z \sum_{k \neq l} p_{kl}
= \text{const} - \sum_{k \neq l} p_{kl} \ln w_{kl} + \ln Z,$$

так как $\sum_{k \neq l} p_{kl} = 1$. Обозначают $d_{kl}^2 = \|y_k - y_l\|^2$; производная по $y_i$ отлична от нуля
только у слагаемых с $k = i$ или $l = i$, и $\frac{\partial d_{il}^2}{\partial y_i} = 2(y_i - y_l)$. Производные двух
вспомогательных функций:

$$\frac{\partial \ln w_{il}}{\partial y_i} = \frac{\partial}{\partial y_i}\Big(-\ln\big(1 + d_{il}^2\big)\Big) = -\frac{2(y_i - y_l)}{1 + d_{il}^2} = -2\,w_{il}\,(y_i - y_l),$$

$$\frac{\partial w_{il}}{\partial y_i} = \frac{\partial}{\partial y_i}\big(1 + d_{il}^2\big)^{-1} = -\frac{2(y_i - y_l)}{\big(1 + d_{il}^2\big)^2} = -2\,w_{il}^2\,(y_i - y_l).$$

Первое слагаемое. Пара $\{i, j\}$ входит в сумму дважды: как $(i, j)$ и как $(j, i)$, и $p_{ij} = p_{ji}$, $w_{ij} = w_{ji}$:

$$\frac{\partial}{\partial y_i}\Big(-\sum_{k \neq l} p_{kl} \ln w_{kl}\Big) = -2\sum_{j \neq i} p_{ij}\,\frac{\partial \ln w_{ij}}{\partial y_i} = 4\sum_{j \neq i} p_{ij}\,w_{ij}\,(y_i - y_j).$$

Второе слагаемое, с тем же удвоением:

$$\frac{\partial \ln Z}{\partial y_i} = \frac{1}{Z} \cdot 2\sum_{j \neq i} \frac{\partial w_{ij}}{\partial y_i} = -\frac{4}{Z}\sum_{j \neq i} w_{ij}^2\,(y_i - y_j) = -4\sum_{j \neq i} q_{ij}\,w_{ij}\,(y_i - y_j).$$

Сумма двух частей дает формулу градиента в рамке:

$$\frac{\partial C}{\partial y_i} = 4\sum_{j \neq i} p_{ij}\,w_{ij}\,(y_i - y_j) - 4\sum_{j \neq i} q_{ij}\,w_{ij}\,(y_i - y_j) = 4 \sum_{j \neq i} \big(p_{ij} - q_{ij}\big)\,w_{ij}\,(y_i - y_j).$$

**Механическая интерпретация.** Шаг градиентного спуска $y_i \leftarrow y_i - \eta\,\frac{\partial C}{\partial y_i}$
сдвигает $y_i$ по сумме сил вдоль отрезков к остальным точкам. Сила со стороны $y_j$
пропорциональна $(p_{ij} - q_{ij})\,w_{ij}$: при $p_{ij} > q_{ij}$ (пара на плоскости дальше, чем
требует $P$) точки притягиваются, при $p_{ij} < q_{ij}$ — отталкиваются. Градиент
раскладывают на притяжение $4\sum_j p_{ij} w_{ij}(y_i - y_j)$ и отталкивание
$-4\sum_j q_{ij} w_{ij}(y_i - y_j)$; второе содержит все пары и требует $O(n^2)$ операций.

**Матричная форма.** Обозначают $M = (P - Q) \circ W$ (поэлементное произведение). Тогда

$$\sum_{j} M_{ij}\,(y_i - y_j) = \Big(\sum_j M_{ij}\Big) y_i - \sum_j M_{ij}\,y_j, \qquad
\nabla_Y C = 4\big(\mathrm{diag}(M \mathbf{1})\,Y - M Y\big).$$

Квадраты расстояний вычисляют через нормы: $\|y_i - y_j\|^2 = \|y_i\|^2 + \|y_j\|^2 - 2\langle y_i, y_j \rangle$.

**Оптимизация.** Координаты находят градиентным спуском с инерцией (momentum) и адаптивными
множителями шага (gains). Обозначают $G = \nabla_Y C$, $\Delta$ — предыдущее изменение $Y$,
$g$ — матрицу множителей той же формы. Итерация:

1. множители обновляют поэлементно: $g \leftarrow g + 0.2$, если знаки $G$ и $\Delta$ различны,
   и $g \leftarrow 0.8\,g$ в противном случае, затем $g \leftarrow \max(g, 0.01)$;
2. изменение $\Delta \leftarrow \mu\,\Delta - \eta\,g \circ G$;
3. координаты $Y \leftarrow Y + \Delta$.

Различие знаков $G$ и $\Delta$ означает, что предыдущий шаг сделан против градиента
и спуск по этой координате продолжается в том же направлении, поэтому шаг
увеличивают. Совпадение знаков означает, что предыдущий шаг перешел через
минимум, и шаг уменьшают.

Параметры, принятые в sklearn:

- начальные координаты — две главные компоненты, масштабированные так, что
  стандартное отклонение первой координаты равно $10^{-4}$;
- инерция $\mu = 0.5$ первые 250 итераций и $0.8$ после;
- шаг $\eta = \max\big(\frac{n}{4\alpha}, 50\big)$, где $\alpha$ — коэффициент раннего преувеличения;
- первые 250 итераций матрицу $P$ умножают на $\alpha = 12$ (раннее преувеличение,
  early exaggeration), всего 1000 итераций.

Функции потерь и градиента реализуют ниже; формулу градиента проверяют
конечными разностями на случайных $P$ и $Y$ из шести точек:
$\frac{\partial C}{\partial y_{id}} \approx \frac{C(Y + h e_{id}) - C(Y - h e_{id})}{2h}$.

In [ ]:
def student_weights(Y):
    squared_norms = (Y ** 2).sum(axis=1)
    squared = np.maximum(squared_norms[:, None] + squared_norms[None, :] - 2 * Y @ Y.T, 0)
    W = 1 / (1 + squared)
    np.fill_diagonal(W, 0)
    return W


def tsne_kl(P, Y):
    W = student_weights(Y)
    Q = W / W.sum()
    mask = P > 0
    kl = (P[mask] * np.log(P[mask] / Q[mask])).sum()
    return kl


def tsne_gradient(P, Y):
    W = student_weights(Y)
    Q = W / W.sum()
    M = (P - Q) * W
    gradient = 4 * (M.sum(axis=1)[:, None] * Y - M @ Y)
    return gradient


rng_check = np.random.default_rng(0)
P_check = rng_check.random((6, 6))
P_check = P_check + P_check.T
np.fill_diagonal(P_check, 0)
P_check = P_check / P_check.sum()
Y_check = rng_check.normal(size=(6, 2))
numeric = np.zeros_like(Y_check)
step = 1e-6
for i in range(6):
    for d in range(2):
        shift = np.zeros_like(Y_check)
        shift[i, d] = step
        numeric[i, d] = (tsne_kl(P_check, Y_check + shift) - tsne_kl(P_check, Y_check - shift)) / (2 * step)
print("наибольшее расхождение формулы и конечных разностей:", np.abs(tsne_gradient(P_check, Y_check) - numeric).max())

### Раннее преувеличение (early exaggeration)

В начале оптимизации все образы находятся почти в одной точке, и взаимное
расположение групп еще не определено. Если группы начнут складываться сразу,
каждая — в случайном месте, части одной цифры могут оказаться по разные стороны
от группы другой цифры. Соединить их позже трудно: на пути стоят точки чужой
группы, которые отталкивают их.

Поэтому первые 250 итераций все сходства $p_{ij}$ умножают на 12. Притяжение между
соседями становится намного сильнее отталкивания, и в этот период происходит
следующее:

- объекты, соседствующие в исходном пространстве, быстро стягиваются
  в компактные группы;
- группы, между которыми мало связей, расходятся, и каждая группа движется
  как одно целое;
- между компактными группами остается свободное место, поэтому группы
  занимают взаимное расположение, не смешиваясь друг с другом.

После 250 итераций множитель убирают. Притяжение возвращается к исходной силе,
отталкивание раздвигает группы, и оптимизация уточняет положение точек внутри
групп.

Прием работает, если группы в данных выражены: соседи каждого объекта лежат
в основном в его же группе. Если окрестности объектов сильно перекрываются
между группами, усиленное притяжение стягивает все точки вместе, и группы
появляются только после отключения множителя. Оба случая показаны ниже.

### Реализация t-SNE

Функция `run_tsne` выполняет описанную оптимизацию с точным градиентом
и сохраняет промежуточные координаты и значения функции потерь. Функция
потерь всегда вычисляется по исходной матрице $P$, без множителя $\alpha$.

In [ ]:
def run_tsne(P, init, n_iter=1000, exaggeration=12.0, exaggeration_iter=250, snapshot_at=()):
    n = len(P)
    Y = init / np.std(init[:, 0]) * 1e-4
    learning_rate = max(n / exaggeration / 4, 50)
    update = np.zeros_like(Y)
    gains = np.ones_like(Y)
    history, snapshots = [], {0: Y.copy()}
    for iteration in range(n_iter):
        early = iteration < exaggeration_iter
        momentum = 0.5 if early else 0.8
        gradient = tsne_gradient(P * exaggeration if early else P, Y)
        opposite = update * gradient < 0
        gains = np.maximum(np.where(opposite, gains + 0.2, gains * 0.8), 0.01)
        update = momentum * update - learning_rate * gains * gradient
        Y = Y + update
        if (iteration + 1) % 10 == 0:
            history.append((iteration + 1, tsne_kl(P, Y)))
        if iteration + 1 in snapshot_at:
            snapshots[iteration + 1] = Y.copy()
    return Y, history, snapshots

In [ ]:
start = time.perf_counter()
own_tsne, own_history, own_snapshots = run_tsne(P_vis, PCA(n_components=2, random_state=SEED).fit_transform(vis_pca50),
                                                snapshot_at=(50, 250, 300, 500, 1000))
own_tsne_seconds = time.perf_counter() - start
print(f"1000 итераций с точным градиентом: {own_tsne_seconds:.1f} с")

Координаты на шести итерациях. Масштаб осей у каждой панели свой;
в заголовке — значение функции потерь и наибольшая по модулю координата.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10.5))
for ax, (iteration, points) in zip(axes.flat, own_snapshots.items()):
    plot_labeled_map(ax, points, vis_y, f"итерация {iteration}: KL {tsne_kl(P_vis, points):.2f}, max |y| = {np.abs(points).max():.1g}")
plt.tight_layout()
plt.show()

In [ ]:
iterations, kl_values = np.array(own_history).T
plt.figure(figsize=(8.5, 4))
plt.plot(iterations, kl_values, color=BLUE, lw=2)
plt.axvspan(0, 250, color=OCHRE, alpha=0.12, label="раннее преувеличение, α = 12")
plt.xlabel("итерация")
plt.ylabel("KL(P ‖ Q)")
plt.legend()
plt.show()

На этапе раннего преувеличения облако остается малым, а значение функции
потерь велико и почти не меняется: при малом масштабе $q_{ij}$ близки к равномерным.
На итерации 50 облако вытянуто вдоль одной линии: сначала проявляется самое
сильное различие между группами: нули оказываются на одном конце, семерки
и девятки — на другом. К итерации 250 проявляются и остальные различия, и точки собираются
в группы. После отключения множителя
отталкивание раздвигает группы, функция потерь быстро убывает, и далее
идет уточнение положения точек.

Для сравнения ту же оптимизацию выполняют на подвыборке из 500 изображений
и сравнивают состояние после 250 итераций раннего преувеличения в обоих случаях.
В заголовке — наибольшая координата и accuracy метода k ближайших соседей
на текущих координатах по кросс-валидации: она показывает, сложилась ли
структура цифр.

In [ ]:
small_conditional, _ = conditional_affinities(vis_pca50[:500], perplexity=30)
P_small = (small_conditional + small_conditional.T) / (2 * 500)
small_init = PCA(n_components=2, random_state=SEED).fit_transform(vis_pca50[:500])
small_after, _, _ = run_tsne(P_small, small_init, n_iter=250)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (name, points, labels) in zip(axes, [("n = 500", small_after, vis_y[:500]), ("n = 2500", own_snapshots[250], vis_y)]):
    accuracy = cross_val_score(KNeighborsClassifier(n_neighbors=5), points, labels, cv=5).mean()
    plot_labeled_map(ax, points, labels, f"{name}, итерация 250: max |y| = {np.abs(points).max():.1g}, accuracy kNN {accuracy:.2f}")
plt.tight_layout()
plt.show()

При $n = 500$ облако сжалось на порядки, и цифры перемешаны; при $n = 2500$ облако
выросло, и цифры уже разделены. При 500 изображениях на каждую цифру приходится
около 50 изображений, и окрестность из 30 эффективных соседей часто захватывает
изображения других цифр: группы выражены слабо, и усиленное притяжение стягивает
все точки вместе. При 2500 изображениях на цифру приходится около 250 изображений,
окрестности реже выходят за пределы своей цифры, и группы успевают сложиться.
При малом числе объектов группы появляются только после отключения преувеличения.

### t-SNE в sklearn

Результат класса `TSNE` из sklearn на тех же 2500 изображениях при `perplexity` = 30.

In [ ]:
start = time.perf_counter()
tsne_model = TSNE(n_components=2, perplexity=30, random_state=SEED).fit(vis_pca50)
tsne_points = tsne_model.embedding_
tsne_seconds = time.perf_counter() - start
fig, ax = plt.subplots(figsize=(11, 9))
plot_digit_map(ax, tsne_points, vis_y, vis_X, f"t-SNE, perplexity = 30, время {tsne_seconds:.1f} с")
plt.show()

In [ ]:
interactive_digit_map(tsne_points, vis_y, "t-SNE, perplexity = 30")

Изображения одной цифры образуют отдельные группы. Группы цифр 4 и 9
перемешаны, рядом с ними расположена группа цифры 7; соприкасаются также
группы цифр 3, 5 и 8. Эти цифры сходны по начертанию.

### Влияние перплексии

При малом значении `perplexity` распределение $P_i$ сосредоточено на нескольких
ближайших объектах, и метод сохраняет только самые локальные связи: группы
распадаются на мелкие части. При большом значении учитывается больше соседей
и сохраняется более крупная структура. Значение должно быть заметно меньше $n$:
при $\mathrm{Perp} \to n - 1$ распределения $P_i$ становятся равномерными и перестают отражать
соседство.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, perplexity in zip(axes, [5, 30, 100]):
    points = TSNE(n_components=2, perplexity=perplexity, random_state=SEED).fit_transform(vis_pca50)
    plot_labeled_map(ax, points, vis_y, f"t-SNE, perplexity = {perplexity}")
plt.tight_layout()
plt.show()

### Что t-SNE не сохраняет

Два свойства функции потерь определяют, что на рисунке t-SNE нельзя
интерпретировать:

- ширина $\sigma_i$ подбирается под фиксированную перплексию, поэтому в плотной
  и в разреженной группе распределения $P_i$ одинаково сосредоточены на 30
  эффективных соседях, и размеры групп на плоскости выравниваются;
- далекие пары имеют $p_{ij} \approx 0$ и вносят малый вклад в функцию потерь,
  поэтому расстояния между далекими группами не сохраняются.

Оба свойства проверяют на двумерных данных. В первом наборе два облака
по 300 точек со стандартными отклонениями 1 и 10. Во втором наборе три облака
по 200 точек с центрами в точках 0, 20 и 200 оси абсцисс: расстояние между вторым
и третьим облаком в 9 раз больше, чем между первым и вторым. Слева — исходные
данные, справа — результат t-SNE; в заголовках — отношение разбросов облаков
и отношение расстояний между центрами.

In [ ]:
rng_demo = np.random.default_rng(4)
sizes_data = np.vstack([rng_demo.normal([0, 0], 1.0, (300, 2)), rng_demo.normal([60, 0], 10.0, (300, 2))])
sizes_labels = np.repeat([0, 1], 300)
gaps_data = np.vstack([rng_demo.normal([x, 0], 1.0, (200, 2)) for x in (0, 20, 200)])
gaps_labels = np.repeat([0, 1, 2], 200)
sizes_map = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(sizes_data)
gaps_map = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(gaps_data)


def spread_ratio(points, labels):
    spreads = [points[labels == c].std(axis=0).mean() for c in (0, 1)]
    ratio = spreads[1] / spreads[0]
    return ratio


def gap_ratio(points, labels):
    centers = np.array([points[labels == c].mean(axis=0) for c in (0, 1, 2)])
    ratio = np.linalg.norm(centers[2] - centers[1]) / np.linalg.norm(centers[1] - centers[0])
    return ratio


fig, axes = plt.subplots(2, 2, figsize=(14, 10))
panels = [
    (axes[0, 0], sizes_data, sizes_labels, f"исходные облака: отношение разбросов {spread_ratio(sizes_data, sizes_labels):.1f}"),
    (axes[0, 1], sizes_map, sizes_labels, f"t-SNE: отношение разбросов {spread_ratio(sizes_map, sizes_labels):.1f}"),
    (axes[1, 0], gaps_data, gaps_labels, f"исходные облака: отношение расстояний {gap_ratio(gaps_data, gaps_labels):.1f}"),
    (axes[1, 1], gaps_map, gaps_labels, f"t-SNE: отношение расстояний {gap_ratio(gaps_map, gaps_labels):.1f}"),
]
for ax, points, labels, title in panels:
    ax.scatter(points[:, 0], points[:, 1], s=5, c=CLUSTER_COLORS[labels])
    ax.set_title(title, fontsize=11)
axes[0, 0].set_aspect("equal")
plt.tight_layout()
plt.show()

Разброс облаков на рисунке t-SNE одинаков, хотя в исходных данных различается
в 10 раз, а расстояния между соседними облаками на рисунке равны, хотя
в исходных данных различаются в 9 раз.
Сохраняется только то, какие точки соседствуют.

### Равномерная аппроксимация и проекция многообразия (UMAP)

Метод UMAP (uniform manifold approximation and projection; Макиннес, Хили
и Мелвилл, 2018) также сохраняет соседей, но описывает их взвешенным графом
и не нормирует сходства на всю выборку.

**Граф в исходном пространстве.** Для каждого объекта $x_i$ находят $k$ = `n_neighbors`
ближайших объектов, причем в umap-learn сам объект входит в их число: соседями
считают $k - 1$ ближайших других объектов. Обозначают $\rho_i$ — расстояние
до ближайшего соседа, $\rho(x_i, x_j)$ — расстояние до соседа $x_j$. Вес направленного
ребра:

$$w_{j \mid i} = \exp\Big(-\frac{\rho(x_i, x_j) - \rho_i}{\sigma_i}\Big),$$

где $\sigma_i$ — решение уравнения

$$f_i(\sigma) = \sum_{j \in \mathrm{kNN}(i)} \exp\Big(-\frac{\rho(x_i, x_j) - \rho_i}{\sigma}\Big) = \log_2 k.$$

**Существование и единственность решения.** Каждое слагаемое с $\rho(x_i, x_j) > \rho_i$
строго возрастает по $\sigma$ от 0 при $\sigma \to 0$ до 1 при $\sigma \to \infty$; слагаемые с $\rho(x_i, x_j) = \rho_i$
равны 1. Поэтому $f_i$ строго возрастает от числа ближайших соседей $m_i \geq 1$ до $k - 1$.
Для $k \geq 2$ выполнено $\log_2 k \leq k - 1$, и при $m_i < \log_2 k$ решение существует и единственно;
его находят двоичным поиском. Значение $\log_2 k$ предложено авторами эмпирически.

**Смысл нормировок.** Вычитание $\rho_i$ гарантирует, что ребро к ближайшему соседу
имеет вес 1, то есть каждый объект связан хотя бы с одним объектом. Деление
на $\sigma_i$ измеряет расстояния в единицах локального масштаба: в плотной области
$\sigma_i$ мало, в разреженной велико. Это соответствует предположению, что данные
распределены на многообразии равномерно в его собственной метрике
(отсюда слово uniform в названии), а различия плотности в исходном пространстве
вызваны искажением метрики. Условие на сумму играет ту же роль, что перплексия
в t-SNE, и параметр `n_neighbors` задает размер окрестности так же, как `perplexity`.

**Симметризация.** Вес $w_{j \mid i}$ трактуют как вероятность существования ребра
в направлении $i \to j$. Если события для двух направлений независимы, вероятность
того, что ребро существует хотя бы в одном направлении:

$$w_{ij} = 1 - (1 - w_{j \mid i})(1 - w_{i \mid j}) = w_{j \mid i} + w_{i \mid j} - w_{j \mid i}\, w_{i \mid j}.$$

Это вероятностная t-конорма (probabilistic t-conorm), операция нечеткого объединения
(fuzzy union) множеств. Функция `fuzzy_graph` строит граф по этим формулам,
включая нижнюю границу $\sigma_i \geq 10^{-3} \cdot \bar{\rho}_i$ из umap-learn, где $\bar{\rho}_i$ — среднее расстояние
до $k$ ближайших объектов с учетом самого объекта.

In [ ]:
def fuzzy_graph(X, n_neighbors):
    """Симметричные веса ребер графа ближайших соседей по формулам UMAP."""
    distances, neighbors = NearestNeighbors(n_neighbors=n_neighbors).fit(X).kneighbors(X)
    mean_distances = distances.mean(axis=1)
    distances, neighbors = distances[:, 1:], neighbors[:, 1:]
    target = np.log2(n_neighbors)
    directed = np.zeros((len(X), len(X)))
    rhos, sigmas = distances[:, 0], np.zeros(len(X))
    for i in range(len(X)):
        shifted = np.maximum(distances[i] - rhos[i], 0)
        low, high, sigma = 0.0, np.inf, 1.0
        for _ in range(64):
            total = np.exp(-shifted / sigma).sum()
            if abs(total - target) < 1e-5:
                break
            if total > target:
                high = sigma
                sigma = (low + high) / 2
            else:
                low = sigma
                sigma = sigma * 2 if np.isinf(high) else (low + high) / 2
        sigmas[i] = max(sigma, 1e-3 * mean_distances[i])
        directed[i, neighbors[i]] = np.exp(-shifted / sigmas[i])
    weights = directed + directed.T - directed * directed.T
    return weights, rhos, sigmas

Граф для 40 точек на спирали при `n_neighbors` = 6 (пять соседей и сама точка);
толщина ребра пропорциональна $w_{ij}$.

In [ ]:
t_curve = np.sort(np.random.default_rng(2).uniform(0, 3 * np.pi, 40))
curve_points = np.c_[t_curve * np.cos(t_curve), t_curve * np.sin(t_curve)] / 3 + np.random.default_rng(3).normal(0, 0.08, (40, 2))
curve_weights, _, _ = fuzzy_graph(curve_points, n_neighbors=6)
fig, ax = plt.subplots(figsize=(7, 6))
for i, j in zip(*np.nonzero(np.triu(curve_weights))):
    ax.plot(*curve_points[[i, j]].T, color=BLUE, lw=4 * curve_weights[i, j], alpha=0.7)
ax.scatter(curve_points[:, 0], curve_points[:, 1], s=40, color=OCHRE, zorder=3)
ax.set_aspect("equal")
ax.set_title("Взвешенный граф ближайших соседей, n_neighbors = 6")
ax.set_xticks([])
ax.set_yticks([])
plt.show()

Построенный граф сравнивают с атрибутом `graph_` класса `umap.UMAP` на спирали
и на 2500 изображениях. Обучение `umap.UMAP` на изображениях дальше используется
и для рисунков; первый вызов включает компиляцию кода библиотекой numba,
поэтому занимает больше времени.

In [ ]:
umap_curve = umap.UMAP(n_neighbors=6, random_state=SEED).fit(curve_points)
start = time.perf_counter()
umap_model = umap.UMAP(n_neighbors=15, random_state=SEED).fit(vis_pca50)
umap_seconds = time.perf_counter() - start
umap_points = umap_model.embedding_
vis_graph, vis_rhos, vis_graph_sigmas = fuzzy_graph(vis_pca50, n_neighbors=15)
print("спираль: наибольшее расхождение весов", np.abs(curve_weights - umap_curve.graph_.toarray()).max())
print("MNIST: наибольшее расхождение весов", np.abs(vis_graph - umap_model.graph_.toarray()).max())
print("MNIST: ребер в графе", int((np.triu(vis_graph) > 0).sum()), " время umap.UMAP:", round(umap_seconds, 1), "с")

Веса совпадают с umap-learn с точностью до вычислений в одинарной точности,
которые использует библиотека.

**Сходство на плоскости** для образов $y_i, y_j$:

$$v_{ij} = \big(1 + a\,\|y_i - y_j\|^{2b}\big)^{-1}.$$

При $a = b = 1$ формула совпадает с функцией Стьюдента из t-SNE. Параметры $a$ и $b$
подбирают по параметрам `min_dist` и `spread` (по умолчанию 1): функцию $v(d)$
приближают методом наименьших квадратов к кусочной функции

$$\psi(d) = \begin{cases} 1, & d < \text{min\_dist}, \\ \exp\big(-(d - \text{min\_dist}) / \text{spread}\big), & d \geq \text{min\_dist}. \end{cases}$$

Параметр `min_dist` — расстояние, до которого образы считаются совпадающими;
при малом значении образы соседей располагаются плотнее. Подбор выполняют
функцией `curve_fit` на сетке 300 точек отрезка $[0, 3]$ и сравнивают с функцией
`find_ab_params` из umap-learn.

In [ ]:
def target_similarity(d, min_dist, spread=1.0):
    psi = np.where(d < min_dist, 1.0, np.exp(-(d - min_dist) / spread))
    return psi


def umap_similarity(d, a, b):
    v = 1 / (1 + a * d ** (2 * b))
    return v


fit_grid = np.linspace(0, 3, 300)
ab_rows = {}
for min_dist in [0.1, 0.5]:
    (a_fit, b_fit), _ = curve_fit(umap_similarity, fit_grid, target_similarity(fit_grid, min_dist))
    a_lib, b_lib = umap_internal.find_ab_params(1.0, min_dist)
    ab_rows[f"min_dist = {min_dist}"] = {"a, curve_fit": a_fit, "b, curve_fit": b_fit,
                                         "a, umap-learn": a_lib, "b, umap-learn": b_lib}
show_table(pd.DataFrame(ab_rows).T, digits=4, axis=None)

In [ ]:
distance = np.linspace(0, 5, 300)
student_kernel = 1 / (1 + distance ** 2)
plt.figure(figsize=(8.5, 4.2))
for min_dist, color in [(0.1, BLUE), (0.5, OCHRE)]:
    a, b = umap_internal.find_ab_params(1.0, min_dist)
    plt.plot(distance, umap_similarity(distance, a, b), color=color, lw=2, label=f"v(d), min_dist = {min_dist}: a = {a:.2f}, b = {b:.2f}")
    plt.plot(distance, target_similarity(distance, min_dist), color=color, lw=1, ls=":", label=f"ψ(d), min_dist = {min_dist}")
plt.plot(distance, student_kernel, color=GREY, ls="--", label="a = b = 1, функция Стьюдента")
plt.xlabel("расстояние между образами d")
plt.ylabel("сходство")
plt.legend(fontsize=9)
plt.show()

**Функция потерь** — перекрестная энтропия (cross-entropy) между весами ребер
$w_{ij}$ и сходствами $v_{ij}$, если трактовать их как вероятности существования ребра:

$$C = \sum_{i < j} \Big[\, w_{ij} \ln \frac{w_{ij}}{v_{ij}} + (1 - w_{ij}) \ln \frac{1 - w_{ij}}{1 - v_{ij}} \Big].$$

Первое слагаемое велико, если соседи в графе ($w_{ij}$ близко к 1) далеки на плоскости;
второе — если не соседние в графе объекты ($w_{ij}$ близко к 0) оказались рядом.
В отличие от t-SNE, сходства не нормируют на всю выборку, и отталкивание
задано отдельным слагаемым.

**Вывод градиента.** От $Y$ зависят только $-w_{ij} \ln v_{ij}$ и $-(1 - w_{ij}) \ln(1 - v_{ij})$.
Обозначают $s = \|y_i - y_j\|^2$, тогда $v = (1 + a s^b)^{-1}$ и

$$\ln v = -\ln(1 + a s^b), \qquad \frac{\partial \ln v}{\partial s} = -\frac{a b\, s^{b-1}}{1 + a s^b};$$

$$1 - v = \frac{a s^b}{1 + a s^b}, \qquad \ln(1 - v) = \ln a + b \ln s - \ln(1 + a s^b),$$

$$\frac{\partial \ln(1 - v)}{\partial s} = \frac{b}{s} - \frac{a b\, s^{b-1}}{1 + a s^b} = \frac{b\,(1 + a s^b) - a b\, s^b}{s\,(1 + a s^b)} = \frac{b}{s\,(1 + a s^b)}.$$

С учетом $\frac{\partial s}{\partial y_i} = 2(y_i - y_j)$:

$$\frac{\partial}{\partial y_i}\big(-w_{ij} \ln v_{ij}\big) = w_{ij}\,\frac{2 a b\, s^{b-1}}{1 + a s^b}\,(y_i - y_j),$$

$$\frac{\partial}{\partial y_i}\big(-(1 - w_{ij}) \ln(1 - v_{ij})\big) = -(1 - w_{ij})\,\frac{2b}{s\,(1 + a s^b)}\,(y_i - y_j).$$

Шаг против градиента по первому слагаемому сдвигает $y_i$ к $y_j$ (притяжение),
по второму — от $y_j$ (отталкивание). Отталкивающий коэффициент неограничен
при $s \to 0$, поэтому в umap-learn знаменатель заменяют на $(0.001 + s)(1 + a s^b)$.

### Оптимизация UMAP

Полная сумма отталкивающих слагаемых содержит $O(n^2)$ пар. Метод использует
стохастический градиентный спуск с двумя видами выборки:

- **выборка ребер.** В каждую эпоху ребро $(i, j)$ используют с вероятностью
  $\frac{w_{ij}}{w_{\max}}$ и применяют притягивающий шаг без множителя $w_{ij}$. Математическое
  ожидание шага равно $\frac{w_{ij}}{w_{\max}}$, умноженному на шаг без веса, то есть
  пропорционально притягивающему слагаемому градиента;
- **отрицательная выборка** (negative sampling). Для каждого использованного ребра
  выбирают `negative_sample_rate` = 5 случайных объектов $l$ и применяют к $y_i$ отталкивающий
  шаг с $w_{il} = 0$. Для случайной пары $w_{il}$ почти всегда равно нулю, поэтому
  среднее этих шагов, умноженное на $\frac{n}{5}$, приближает сумму отталкивающих
  слагаемых по всем объектам.

Дополнительно каждую координату шага ограничивают отрезком $[-4, 4]$,
а шаг $\alpha$ уменьшают линейно от 1 до 0 за `n_epochs` эпох (500 при $n \le 10\,000$,
иначе 200). Ребра с $w_{ij} < w_{\max} / \text{n\_epochs}$ не используют: в среднем их выбрали бы
меньше одного раза. Каждая эпоха требует $O(k n)$ операций.

**Начальные координаты — спектральное вложение** (spectral embedding, Laplacian
eigenmaps). Ищут координаты $y \in \mathbb{R}^n$ (одну ось), при которых связанные сильными
ребрами объекты близки:

$$\frac{1}{2}\sum_{i, j} w_{ij}\,(y_i - y_j)^2 = y^\top L y \to \min, \qquad L = D - W, \qquad D = \mathrm{diag}\Big(\sum_j w_{ij}\Big).$$

Матрицу $L$ называют лапласианом графа (graph Laplacian). Равенство выводится
непосредственно: раскрывают квадрат $(y_i - y_j)^2$ и пользуются симметрией весов
$w_{ij} = w_{ji}$, после чего сумма распадается на $y^\top D y$ и $y^\top W y$.

Квадратичная форма мала, если $y$ почти не меняется вдоль ребер с большим весом,
то есть почти постоянна на группах сильно связанных объектов.

Без ограничений минимум равен нулю при $y = 0$, поэтому фиксируют масштаб:
$y^\top D y = 1$. Функция Лагранжа $y^\top L y - \lambda\,(y^\top D y - 1)$, ее градиент:

$$2 L y - 2\lambda D y = 0 \;\Rightarrow\; L y = \lambda D y,$$

обобщенная задача на собственные значения, и в решении $y^\top L y = \lambda\, y^\top D y = \lambda$.
Наименьшее $\lambda = 0$ дает постоянный вектор $y = c\mathbf{1}$, так как $L\mathbf{1} = D\mathbf{1} - W\mathbf{1} = 0$; он не различает
объекты. Две координаты берут из собственных векторов со вторым и третьим
наименьшими собственными значениями. В umap-learn решают эквивалентную
задачу для нормированного лапласиана $I - D^{-1/2} W D^{-1/2}$ и масштабируют результат
в квадрат со стороной 10. Если граф несвязен, нулевых собственных значений
несколько, поэтому связность проверяют.

In [ ]:
n_components_graph, _ = connected_components(vis_graph > 0)
degrees = vis_graph.sum(axis=1)
spectral_values, spectral_vectors = eigh(np.diag(degrees) - vis_graph, np.diag(degrees), subset_by_index=[0, 2])
spectral_init = spectral_vectors[:, 1:3]
spectral_init = 10 * (spectral_init - spectral_init.min(axis=0)) / (spectral_init.max(axis=0) - spectral_init.min(axis=0))
print("компонент связности графа:", n_components_graph)
print("три наименьших собственных значения:", np.round(spectral_values, 4))

fig, ax = plt.subplots(figsize=(8, 7))
accuracy = cross_val_score(KNeighborsClassifier(n_neighbors=5), spectral_init, vis_y, cv=5).mean()
plot_labeled_map(ax, spectral_init, vis_y, f"Спектральное вложение графа UMAP: accuracy kNN {accuracy:.2f}")
plt.show()

Спектральное вложение уже отделяет часть цифр: это линейная задача для того
же графа, и ее решение задает разумное начальное приближение. Функция
`run_umap` выполняет описанный стохастический градиентный спуск. Отличие
от umap-learn: шаги всех ребер одной эпохи вычисляют по координатам начала
эпохи и применяют одновременно, а не по одному.

In [ ]:
def run_umap(weights, init, a, b, n_epochs=500, negative_rate=5, seed=SEED, snapshot_at=()):
    rng = np.random.default_rng(seed)
    heads, tails = np.nonzero(weights)
    edge_weights = weights[heads, tails]
    keep = edge_weights >= edge_weights.max() / n_epochs
    heads, tails, edge_weights = heads[keep], tails[keep], edge_weights[keep]
    probability = edge_weights / edge_weights.max()
    Y = init.copy()
    snapshots = {0: Y.copy()}
    for epoch in range(n_epochs):
        alpha = 1.0 - epoch / n_epochs
        active = rng.random(len(heads)) < probability
        i, j = heads[active], tails[active]
        diff = Y[i] - Y[j]
        s = (diff ** 2).sum(axis=1)
        safe = np.where(s > 0, s, 1.0)
        attraction = np.where(s > 0, -2 * a * b * safe ** (b - 1) / (1 + a * safe ** b), 0.0)
        step = np.clip(attraction[:, None] * diff, -4, 4) * alpha
        np.add.at(Y, i, step)
        np.add.at(Y, j, -step)
        i_negative = np.repeat(i, negative_rate)
        l_negative = rng.integers(0, len(Y), len(i_negative))
        diff = Y[i_negative] - Y[l_negative]
        s = (diff ** 2).sum(axis=1)
        repulsion_coeff = np.where(s > 0, 2 * b / ((0.001 + s) * (1 + a * s ** b)), 0.0)
        step = np.clip(repulsion_coeff[:, None] * diff, -4, 4) * alpha
        np.add.at(Y, i_negative, step)
        if epoch + 1 in snapshot_at:
            snapshots[epoch + 1] = Y.copy()
    return Y, snapshots

In [ ]:
a_default, b_default = umap_internal.find_ab_params(1.0, 0.1)
start = time.perf_counter()
own_umap, umap_snapshots = run_umap(vis_graph, spectral_init, a_default, b_default, snapshot_at=(50, 200, 400, 500))
own_umap_seconds = time.perf_counter() - start
fig, axes = plt.subplots(1, 5, figsize=(24, 5))
for ax, (epoch, points) in zip(axes, umap_snapshots.items()):
    accuracy = cross_val_score(KNeighborsClassifier(n_neighbors=5), points, vis_y, cv=5).mean()
    plot_labeled_map(ax, points, vis_y, f"эпоха {epoch}: accuracy kNN {accuracy:.2f}")
plt.tight_layout()
plt.show()

В первые эпохи шаг $\alpha$ велик, и отрицательная выборка перемешивает спектральное
начальное приближение; по мере уменьшения шага группы собираются, последние
эпохи уточняют положение точек. Реализацию сравнивают с umap-learn; строки —
реализации, столбцы — метрики.

In [ ]:
own_vs_umap = pd.DataFrame({
    "реализация с нуля": {
        "надежность T(10)": trustworthiness(vis_pca50, own_umap, n_neighbors=10),
        "accuracy kNN": cross_val_score(KNeighborsClassifier(n_neighbors=5), own_umap, vis_y, cv=5).mean(),
        "время оптимизации, с": own_umap_seconds,
    },
    "umap-learn": {
        "надежность T(10)": trustworthiness(vis_pca50, umap_points, n_neighbors=10),
        "accuracy kNN": cross_val_score(KNeighborsClassifier(n_neighbors=5), umap_points, vis_y, cv=5).mean(),
        "время оптимизации, с": umap_seconds,
    },
}).T
show_table(own_vs_umap)

Метрики совпадают. Время umap-learn включает построение графа, спектральное
вложение и компиляцию. Результат umap-learn на изображениях:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
plot_digit_map(ax, umap_points, vis_y, vis_X, f"UMAP, n_neighbors = 15, время {umap_seconds:.1f} с")
plt.show()

In [ ]:
interactive_digit_map(umap_points, vis_y, "UMAP, n_neighbors = 15")

**Влияние числа соседей.** При малом `n_neighbors` граф описывает только ближайшую
окрестность: группы компактны и разнесены, их взаимное расположение
случайно. При большом значении граф связывает больше объектов, сохраняется
взаимное расположение групп, и сходные цифры сближаются: при `n_neighbors` = 100
группы цифр 3, 5 и 8, а также 4, 7 и 9 образуют связные области.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, n_neighbors in zip(axes, [5, 15, 100]):
    points = umap.UMAP(n_neighbors=n_neighbors, random_state=SEED).fit_transform(vis_pca50)
    plot_labeled_map(ax, points, vis_y, f"UMAP, n_neighbors = {n_neighbors}")
plt.tight_layout()
plt.show()

### Сравнение устройства t-SNE и UMAP

| | t-SNE | UMAP |
|---|---|---|
| сходство в исходном пространстве | гауссово ядро $p_{j \mid i}$, $\sigma_i$ по перплексии | экспонента от $\rho - \rho_i$, $\sigma_i$ по условию $\sum = \log_2 k$ |
| симметризация | среднее $\frac{p_{j \mid i} + p_{i \mid j}}{2n}$ | нечеткое объединение $a + b - ab$ |
| сходство на плоскости | $(1 + d^2)^{-1}$, нормировано на все пары | $(1 + a d^{2b})^{-1}$, без нормировки |
| функция потерь | $\mathrm{KL}(P \,\Vert\, Q)$ | перекрестная энтропия по ребрам |
| отталкивание | через нормировку $Z$ | явное слагаемое $(1 - w)\ln(1 - v)$ |
| оптимизация | градиентный спуск с инерцией по всем парам | SGD с отрицательной выборкой, $O(k n)$ на эпоху |
| начальные координаты | главные компоненты | спектральное вложение |
| новые объекты | нет | метод `transform` |

### Сравнение трех методов

Качество двумерного представления оценивают **надежностью** (trustworthiness).
Для объекта $i$ упорядочивают остальные объекты по возрастанию расстояния
в исходном пространстве; $r(i, j) \in \{1, \dots, n - 1\}$ — номер объекта $j$ в этом списке.
Пусть $U_k(i)$ — множество объектов, которые входят в $k$ ближайших соседей $i$
на плоскости, но не входят в $k$ ближайших соседей в исходном пространстве.
Каждый такой ложный сосед имеет $r(i, j) > k$ и получает штраф $r(i, j) - k$: тем больше,
чем дальше он в действительности. Надежность:

$$T(k) = 1 - \frac{2}{n k (2n - 3k - 1)} \sum_{i=1}^{n} \sum_{j \in U_k(i)} \big(r(i, j) - k\big).$$

**Вывод нормировки.** Наибольший штраф объекта $i$ получается, если все $k$ соседей
на плоскости — самые далекие объекты с номерами $n - 1, n - 2, \dots, n - k$
(при $k < n/2$ они не входят в $k$ ближайших). Их штраф:

$$\sum_{t=1}^{k} \big((n - t) - k\big) = k(n - k) - \frac{k(k + 1)}{2} = \frac{k\,(2n - 3k - 1)}{2}.$$

Для $n$ объектов наибольший суммарный штраф равен $\frac{n k (2n - 3k - 1)}{2}$, поэтому
$T(k) \in [0, 1]$: $T(k) = 1$, если ложных соседей нет, и $T(k) = 0$ в наихудшем случае.
Формулу проверяют вычислением по определению и сравнением с функцией
`trustworthiness` из sklearn для результата t-SNE.

In [ ]:
def trustworthiness_by_definition(X, Y, k):
    n = len(X)
    original = squareform(pdist(X))
    embedded = squareform(pdist(Y))
    np.fill_diagonal(original, np.inf)
    np.fill_diagonal(embedded, np.inf)
    order = np.argsort(original, axis=1)
    ranks = np.empty_like(order)
    np.put_along_axis(ranks, order, np.arange(1, n + 1)[None, :].repeat(n, axis=0), axis=1)
    original_neighbors, embedded_neighbors = order[:, :k], np.argsort(embedded, axis=1)[:, :k]
    penalty = 0
    for i in range(n):
        false_neighbors = np.setdiff1d(embedded_neighbors[i], original_neighbors[i])
        penalty += (ranks[i, false_neighbors] - k).sum()
    trust = 1 - 2 / (n * k * (2 * n - 3 * k - 1)) * penalty
    return trust


print(f"по определению: {trustworthiness_by_definition(vis_pca50, tsne_points, 10):.6f}")
print(f"sklearn:        {trustworthiness(vis_pca50, tsne_points, n_neighbors=10):.6f}")

Вторая величина — accuracy метода k ближайших соседей, обученного
на двумерных координатах, по кросс-валидации: она показывает, насколько
разделены цифры на рисунке. Три представления на одних 2500 изображениях:

In [ ]:
vis_embeddings = {
    "PCA, две компоненты": PCA(n_components=2, random_state=SEED).fit_transform(vis_pca50),
    "t-SNE, perplexity = 30": tsne_points,
    "UMAP, n_neighbors = 15": umap_points,
}
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
for ax, (name, points) in zip(axes, vis_embeddings.items()):
    plot_digit_map(ax, points, vis_y, vis_X, name, n_marks=25)
    ax.get_legend().remove()
handles = [Line2D([], [], marker="o", ls="", color=CLUSTER_COLORS[d], label=str(d)) for d in range(10)]
fig.legend(handles=handles, title="цифра", loc="center right", fontsize=9)
plt.tight_layout(rect=(0, 0, 0.96, 1))
plt.show()

В таблице строки — методы, включая обе реализации с нуля, столбцы — надежность
$T(10)$ и accuracy метода k ближайших соседей на двумерных координатах.

In [ ]:
vis_rows = {}
for name, points in {**vis_embeddings, "t-SNE, реализация с нуля": own_tsne, "UMAP, реализация с нуля": own_umap}.items():
    vis_rows[name] = {
        "надежность T(10)": trustworthiness(vis_pca50, points, n_neighbors=10),
        "accuracy kNN на плоскости": cross_val_score(KNeighborsClassifier(n_neighbors=5), points, vis_y, cv=5).mean(),
    }
show_table(pd.DataFrame(vis_rows).T)

t-SNE и UMAP сохраняют соседей почти без ошибок, а две главные компоненты
сохраняют их плохо: PCA максимизирует дисперсию, а не сохранение окрестностей.

Двумерные представления t-SNE и UMAP интерпретируют со следующими ограничениями:

- соседство точек на рисунке отражает соседство в исходном пространстве,
  так как именно его оптимизирует функция потерь;
- размеры групп и расстояния между ними на рисунке t-SNE не соответствуют
  исходным: подбор $\sigma_i$ выравнивает плотность, а функция потерь слабо зависит
  от далеких пар, что показано на облаках выше; у UMAP взаимное расположение
  групп сохраняется лучше, но также неточно;
- результат зависит от `random_state` и параметров, поэтому вывод, который
  не воспроизводится при другом начальном состоянии, не используют;
- у t-SNE нет преобразования новых объектов, поэтому его координаты
  не применяют как признаки модели; выводы, полученные по рисунку,
  проверяют на исходных данных.

## 5. Кластеризация

Задача кластеризации (clustering) — разбить объекты на группы так, чтобы объекты
внутри группы были сходны, а объекты разных групп различались; правильные
ответы неизвестны. Разбиение задают метками $c(i) \in \{1, \dots, k\}$, кластер $C_c$ —
множество объектов с меткой $c$, $n_c = |C_c|$, $\mu_c = \frac{1}{n_c}\sum_{x \in C_c} x$ — его центр.

Для изучения методов используют пять двумерных наборов из sklearn, в которых
истинное разбиение известно: сферические кластеры, вытянутые кластеры,
кластеры разной плотности, полумесяцы и кольца. Все наборы стандартизованы.

In [ ]:
X_aniso, y_aniso = make_blobs(n_samples=500, centers=3, random_state=170)
X_aniso = X_aniso @ np.array([[0.6, -0.64], [-0.41, 0.85]])
X_density, y_density = make_blobs(n_samples=500, centers=3, cluster_std=[1.0, 2.5, 0.5], random_state=170)

datasets = {
    "сферические": make_blobs(n_samples=500, centers=3, random_state=8),
    "вытянутые": (X_aniso, y_aniso),
    "разная плотность": (X_density, y_density),
    "полумесяцы": make_moons(n_samples=500, noise=0.06, random_state=SEED),
    "кольца": make_circles(n_samples=500, factor=0.5, noise=0.05, random_state=SEED),
}
datasets = {name: (StandardScaler().fit_transform(X), y) for name, (X, y) in datasets.items()}
X_balls, y_balls = datasets["сферические"]


def plot_partition(ax, X, labels, title):
    labels = np.asarray(labels)
    colors = np.where(labels >= 0, CLUSTER_COLORS[labels % len(CLUSTER_COLORS)], "#111111")
    ax.scatter(X[:, 0], X[:, 1], s=7, c=colors)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])


fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
for ax, (name, (X, y)) in zip(axes, datasets.items()):
    plot_partition(ax, X, y, f"{name}: истинное разбиение")
plt.tight_layout()
plt.show()

### Метрики качества при известном разбиении

Номера кластеров произвольны: разбиения $\{0, 0, 1\}$ и $\{1, 1, 0\}$ совпадают. Поэтому
метрики сравнивают не номера, а то, какие объекты отнесены к одному кластеру.
Обозначения: $n$ — число объектов; таблица сопряженности (contingency table) $n_{ij}$ —
число объектов истинного класса $i$, отнесенных к найденному кластеру $j$;
$a_i = \sum_j n_{ij}$ — размер класса, $b_j = \sum_i n_{ij}$ — размер кластера. Обзор метрик
с примерами — в руководстве sklearn, раздел
[Clustering performance evaluation](https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation).

**Индекс Рэнда** (Rand index, RI) — доля пар объектов, по которым два разбиения
согласованы:

$$\mathrm{RI} = \frac{a + b}{\binom{n}{2}},$$

где $a$ — число пар, отнесенных к одному кластеру в обоих разбиениях, $b$ — число пар,
отнесенных к разным кластерам в обоих разбиениях. По таблице сопряженности:

$$a = \sum_{ij}\binom{n_{ij}}{2}, \qquad b = \binom{n}{2} - \sum_i \binom{a_i}{2} - \sum_j \binom{b_j}{2} + \sum_{ij}\binom{n_{ij}}{2}.$$

Функция `rand_score`. Источник: W. M. Rand. Objective criteria for the evaluation
of clustering methods // Journal of the American Statistical Association, 1971
([doi:10.1080/01621459.1971.10482356](https://doi.org/10.1080/01621459.1971.10482356)).

Пример: шесть объектов, истинное разбиение $\{0, 0, 0, 1, 1, 1\}$, найденное
$\{0, 0, 1, 1, 2, 2\}$. На схеме объекты расположены по истинным классам
(рамки), цвет — найденный кластер; линии соединяют пары, отнесенные
к одному кластеру в найденном разбиении. В таблице все 15 пар и их статус.

In [ ]:
truth_small = np.array([0, 0, 0, 1, 1, 1])
found_small = np.array([0, 0, 1, 1, 2, 2])
toy_xy = np.array([[0, 0], [1, 0], [2, 0], [4, 0], [5, 0], [6, 0]], dtype=float)
fig, ax = plt.subplots(figsize=(8, 2.2))
for left, right in [(-0.5, 2.5), (3.5, 6.5)]:
    ax.add_patch(Rectangle((left, -0.6), right - left, 1.2, fill=False, lw=1.2, color=GREY))
for i, j in itertools.combinations(range(6), 2):
    if found_small[i] == found_small[j]:
        ax.plot(*toy_xy[[i, j]].T, color=BLACK, lw=2, zorder=1)
ax.scatter(toy_xy[:, 0], toy_xy[:, 1], s=500, c=CLUSTER_COLORS[found_small], zorder=2)
for i, (x, y) in enumerate(toy_xy):
    ax.text(x, y, str(i), ha="center", va="center", color="white", fontsize=12, weight="bold", zorder=3)
ax.text(1, 0.8, "истинный класс 0", ha="center", color=GREY)
ax.text(5, 0.8, "истинный класс 1", ha="center", color=GREY)
ax.set_xlim(-1, 7)
ax.set_ylim(-1, 1.2)
ax.axis("off")
plt.show()

In [ ]:
pair_rows = []
for i, j in itertools.combinations(range(6), 2):
    together_truth, together_found = truth_small[i] == truth_small[j], found_small[i] == found_small[j]
    if together_truth and together_found:
        status = "a: вместе в обоих"
    elif not together_truth and not together_found:
        status = "b: врозь в обоих"
    else:
        status = "не согласована"
    pair_rows.append({"пара": f"({i}, {j})", "истинное: вместе": together_truth, "найденное: вместе": together_found, "статус": status})
pairs_table = pd.DataFrame(pair_rows)
counts = pairs_table["статус"].value_counts()
rand_index = (counts.get("a: вместе в обоих", 0) + counts.get("b: врозь в обоих", 0)) / comb(6, 2)
print(counts.to_string())
print(f"RI = (a + b) / C(6, 2) = {rand_index:.3f}")
pairs_table

**Скорректированный индекс Рэнда** (adjusted Rand index, ARI) — индекс Рэнда
с поправкой на случайность: из числа пар, отнесенных к одному кластеру в обоих
разбиениях, вычитают его математическое ожидание $E$ при случайной перестановке
меток с сохранением размеров кластеров и нормируют на наибольшее значение:

$$E = \frac{\sum_i \binom{a_i}{2} \sum_j \binom{b_j}{2}}{\binom{n}{2}}, \qquad
\mathrm{ARI} = \frac{\sum_{ij}\binom{n_{ij}}{2} - E}{\frac{1}{2}\Big[\sum_i \binom{a_i}{2} + \sum_j \binom{b_j}{2}\Big] - E}.$$

$\mathrm{ARI} = 1$ при совпадении разбиений, около 0 для случайного разбиения, может
быть отрицательным. Функция `adjusted_rand_score`. Источник: L. Hubert, P. Arabie.
Comparing partitions // Journal of Classification, 1985
([doi:10.1007/BF01908075](https://doi.org/10.1007/BF01908075)).

Вычисление для примера из шести объектов:

In [ ]:
contingency_small = pd.crosstab(truth_small, found_small)
sum_cells = sum(comb(int(v), 2) for v in contingency_small.to_numpy().ravel())
sum_rows = sum(comb(int(v), 2) for v in contingency_small.sum(axis=1))
sum_cols = sum(comb(int(v), 2) for v in contingency_small.sum(axis=0))
expected = sum_rows * sum_cols / comb(6, 2)
ari_by_formula = (sum_cells - expected) / (0.5 * (sum_rows + sum_cols) - expected)
print(contingency_small)
print(f"Σ C(n_ij, 2) = {sum_cells}, Σ C(a_i, 2) = {sum_rows}, Σ C(b_j, 2) = {sum_cols}, E = {expected:.3f}")
print(f"ARI по формуле: {ari_by_formula:.4f}, adjusted_rand_score: {adjusted_rand_score(truth_small, found_small):.4f}")

**Энтропийные метрики** (entropy-based metrics). Пусть $C$ — истинный класс, $K$ — найденный
кластер случайно выбранного объекта, $n_c$, $n_k$ — размеры класса и кластера, $n_{ck}$ —
число объектов в их пересечении. Используют:

- энтропию $H(C) = -\sum_c \frac{n_c}{n} \ln \frac{n_c}{n}$ и аналогично $H(K)$;
- условную энтропию $H(C \mid K) = -\sum_{c, k} \frac{n_{ck}}{n} \ln \frac{n_{ck}}{n_k}$ и аналогично $H(K \mid C)$;
- взаимную информацию (mutual information) $I(C; K) = H(C) - H(C \mid K)$.

Метрики:

- **однородность** (homogeneity) $h = 1 - \frac{H(C \mid K)}{H(C)}$; $h = 1$, если каждый кластер
  содержит объекты одного класса; функция `homogeneity_score`;
- **полнота** (completeness) $c = 1 - \frac{H(K \mid C)}{H(K)}$; $c = 1$, если каждый класс целиком
  попал в один кластер; функция `completeness_score`;
- **V-мера** (V-measure) $V = \frac{2hc}{h + c}$ — среднее гармоническое однородности и полноты;
  функция `v_measure_score`;
- **нормированная взаимная информация** (normalized mutual information, NMI)
  $\mathrm{NMI} = \frac{I(C; K)}{\frac{1}{2}\big(H(C) + H(K)\big)}$ — в sklearn с нормировкой на среднее
  арифметическое энтропий по умолчанию она совпадает с V-мерой; функция
  `normalized_mutual_info_score`.

Источники: A. Rosenberg, J. Hirschberg. V-Measure: a conditional entropy-based
external cluster evaluation measure // EMNLP-CoNLL, 2007
([ACL Anthology D07-1043](https://aclanthology.org/D07-1043/)); N. X. Vinh, J. Epps, J. Bailey.
Information theoretic measures for clusterings comparison // Journal of Machine
Learning Research, 2010 ([JMLR 11](https://jmlr.org/papers/v11/vinh10a.html)).

### Метрики качества при неизвестном разбиении

**Силуэт** (silhouette). Для объекта $i$ из кластера $C_{c(i)}$:

$$a(i) = \frac{1}{n_{c(i)} - 1}\sum_{j \in C_{c(i)},\, j \neq i} \rho(x_i, x_j), \qquad
b(i) = \min_{c \neq c(i)} \frac{1}{n_c}\sum_{j \in C_c} \rho(x_i, x_j), \qquad
s(i) = \frac{b(i) - a(i)}{\max\big(a(i), b(i)\big)} \in [-1, 1].$$

Метрика — среднее $s(i)$ по выборке; большее значение лучше. Функции `silhouette_score`
и `silhouette_samples`. Источник: P. J. Rousseeuw. Silhouettes: a graphical aid
to the interpretation and validation of cluster analysis // Journal of Computational
and Applied Mathematics, 1987 ([doi:10.1016/0377-0427(87)90125-7](https://doi.org/10.1016/0377-0427(87)90125-7)).

**Индекс Калински — Харабаша** (Calinski — Harabasz index). Пусть $\mu$ — центр выборки,
$\mu_c$ и $n_c$ — центр и размер кластера $c$, $k$ — число кластеров;
$W = \sum_c \sum_{x \in C_c} (x - \mu_c)(x - \mu_c)^\top$ — матрица внутрикластерного рассеяния,
$B = \sum_c n_c\,(\mu_c - \mu)(\mu_c - \mu)^\top$ — матрица межкластерного рассеяния. Тогда

$$\mathrm{CH} = \frac{\mathrm{tr}\,B / (k - 1)}{\mathrm{tr}\,W / (n - k)};$$

большее значение лучше. Функция `calinski_harabasz_score`. Источник: T. Caliński,
J. Harabasz. A dendrite method for cluster analysis // Communications in Statistics, 1974
([doi:10.1080/03610927408827101](https://doi.org/10.1080/03610927408827101)).

**Индекс Дэвиса — Болдина** (Davies — Bouldin index). Пусть $s_c$ — среднее расстояние объектов
кластера $c$ до его центра $\mu_c$. Тогда

$$\mathrm{DB} = \frac{1}{k}\sum_{c=1}^{k} \max_{d \neq c} \frac{s_c + s_d}{\rho(\mu_c, \mu_d)};$$

меньшее значение лучше. Функция `davies_bouldin_score`. Источник: D. L. Davies,
D. W. Bouldin. A cluster separation measure // IEEE Transactions on Pattern Analysis
and Machine Intelligence, 1979 ([doi:10.1109/TPAMI.1979.4766909](https://doi.org/10.1109/TPAMI.1979.4766909)).

**Диаграмма силуэтов** показывает $s(i)$ всех объектов, упорядоченные внутри каждого
кластера. Слева — разбиение методом k-средних на сферических кластерах,
справа — истинное разбиение полумесяцев.

In [ ]:
def silhouette_diagram(ax, X, labels, title):
    values = silhouette_samples(X, labels)
    bottom = 0
    for cluster in np.unique(labels):
        cluster_values = np.sort(values[labels == cluster])
        ax.fill_betweenx(np.arange(bottom, bottom + len(cluster_values)), 0, cluster_values,
                         color=CLUSTER_COLORS[cluster], alpha=0.85)
        bottom += len(cluster_values) + 15
    ax.axvline(values.mean(), color=BLACK, ls="--", lw=1.2, label=f"средний силуэт {values.mean():.2f}")
    ax.set_xlim(min(values.min(), 0) - 0.05, 1)
    ax.set_yticks([])
    ax.set_xlabel("s(i)")
    ax.set_title(title, fontsize=11)
    ax.legend(loc="lower right")


X_moon_clusters, y_moon_clusters = datasets["полумесяцы"]
kmeans_balls = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(X_balls)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
silhouette_diagram(axes[0], X_balls, kmeans_balls, "сферические, метод k-средних")
silhouette_diagram(axes[1], X_moon_clusters, y_moon_clusters, "полумесяцы, истинное разбиение")
plt.tight_layout()
plt.show()

У полумесяцев истинное разбиение имеет низкий силуэт: для концов дуг
среднее расстояние до соседней дуги меньше, чем до своей.

### Метрики на примерах

На наборе сферических кластеров сравнивают пять разбиений: истинное,
истинное с переставленными номерами, разбиение со слитыми кластерами,
разбиение с разрезанными кластерами и случайное. В заголовках рисунков —
ARI и силуэт.

In [ ]:
def clustering_scores(X, y_true, labels):
    labels = np.asarray(labels)
    n_clusters = len(set(labels) - {-1})
    defined = 1 < n_clusters < len(X)
    scores = {
        "ARI": adjusted_rand_score(y_true, labels),
        "NMI": normalized_mutual_info_score(y_true, labels),
        "однородность": homogeneity_score(y_true, labels),
        "полнота": completeness_score(y_true, labels),
        "V-мера": v_measure_score(y_true, labels),
        "силуэт": silhouette_score(X, labels) if defined else np.nan,
        "Калински — Харабаш": calinski_harabasz_score(X, labels) if defined else np.nan,
        "Дэвис — Болдин": davies_bouldin_score(X, labels) if defined else np.nan,
    }
    return scores


split_halves = y_balls * 2 + (X_balls[:, 0] > np.median(X_balls[:, 0])).astype(int)
variants = {
    "истинное": y_balls,
    "номера переставлены": (y_balls + 1) % 3,
    "два кластера слиты": np.where(y_balls == 2, 1, y_balls),
    "кластеры разрезаны": split_halves,
    "случайное": np.random.default_rng(0).integers(0, 3, len(y_balls)),
}
metric_demo = pd.DataFrame({name: clustering_scores(X_balls, y_balls, labels) for name, labels in variants.items()}).T

fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
for ax, (name, labels) in zip(axes, variants.items()):
    plot_partition(ax, X_balls, labels, f"{name}\nARI {metric_demo.loc[name, 'ARI']:.2f}, силуэт {metric_demo.loc[name, 'силуэт']:.2f}")
plt.tight_layout()
plt.show()

В таблице строки — разбиения, столбцы — метрики; заливка по столбцам,
для индекса Дэвиса — Болдина ярче меньшее значение.

In [ ]:
show_table(metric_demo)

Свойства метрик на примерах:

- перестановка номеров не меняет ни одну метрику, так как метрики зависят
  только от того, какие объекты отнесены к одному кластеру;
- при слиянии двух кластеров полнота равна 1 (каждый класс целиком в одном
  кластере), а однородность снижается (слитый кластер содержит два класса);
- при разрезании кластеров, наоборот, однородность равна 1, а полнота снижается;
- у случайного разбиения ARI близок к нулю, силуэт отрицателен;
- метрики без истинного разбиения оценивают форму кластеров и максимальны
  для компактных выпуклых кластеров.

### Метод k-средних

Метод k-средних (k-means, класс `KMeans`) ищет $k$ центров $\mu_1, \dots, \mu_k$ и отнесение
объектов $c(i)$, минимизирующие **инерцию** (inertia) — сумму квадратов расстояний
объектов до центров их кластеров:

$$J = \sum_{i=1}^{n} \|x_i - \mu_{c(i)}\|^2.$$

**Оптимальный центр.** Пусть $K$ — множество объектов кластера, $\bar{x}_K = \frac{1}{|K|}\sum_{i \in K} x_i$ —
их среднее, $z \in \mathbb{R}^p$ — произвольная точка. Записывают $x_i - z = (x_i - \bar{x}_K) + (\bar{x}_K - z)$:

$$\sum_{i \in K}\|x_i - z\|^2 = \sum_{i \in K}\|x_i - \bar{x}_K\|^2 + 2\Big\langle \sum_{i \in K}(x_i - \bar{x}_K),\ \bar{x}_K - z \Big\rangle + |K|\,\|\bar{x}_K - z\|^2
= \sum_{i \in K}\|x_i - \bar{x}_K\|^2 + |K|\,\|\bar{x}_K - z\|^2,$$

так как $\sum_{i \in K}(x_i - \bar{x}_K) = 0$. Сумма минимальна при $z = \bar{x}_K$, и замена текущего центра
кластера $\mu_c$ средним $\bar{x}_K$ уменьшает ее ровно на $|K|\,\|\bar{x}_K - \mu_c\|^2$.

**Алгоритм Ллойда** (Lloyd's algorithm) чередует два шага:

1. отнесение: каждый объект относят к ближайшему центру; каждое слагаемое $J$
   при этом не возрастает;
2. пересчет: каждый центр заменяют средним его объектов; по полученному
   разложению $J$ не возрастает и уменьшается, если хотя бы один центр сдвинулся.

**Сходимость за конечное число шагов.** Разбиений $n$ объектов на $k$ групп конечное
число, не больше $k^n$. Если итерация меняет разбиение (при фиксированном правиле
выбора среди равноудаленных центров), $J$ строго убывает, поэтому разбиение
не может повториться, и алгоритм останавливается. Найденный минимум
локальный: он зависит от начальных центров, а поиск глобального минимума —
NP-трудная задача уже при $k = 2$ (Алоиз и соавторы, 2009).

**Форма кластеров.** Объект $x$ ближе к центру $\mu_a$, чем к $\mu_b$, если
$\|x - \mu_a\|^2 \le \|x - \mu_b\|^2$. Раскрытие квадратов:

$$\|x\|^2 - 2\langle x, \mu_a \rangle + \|\mu_a\|^2 \le \|x\|^2 - 2\langle x, \mu_b \rangle + \|\mu_b\|^2
\;\Longleftrightarrow\; 2\langle x, \mu_b - \mu_a \rangle \le \|\mu_b\|^2 - \|\mu_a\|^2.$$

Это полупространство; его граница — гиперплоскость, перпендикулярная $\mu_b - \mu_a$.
Подстановка $x = \frac{\mu_a + \mu_b}{2}$ дает равенство $\langle \mu_a + \mu_b, \mu_b - \mu_a \rangle = \|\mu_b\|^2 - \|\mu_a\|^2$,
то есть граница проходит через середину отрезка. Область кластера —
пересечение $k - 1$ полупространств, выпуклый многогранник (ячейка диаграммы
Вороного, Voronoi diagram), поэтому кластеры метода k-средних выпуклы.

**Сложность.** Итерация требует вычисления $nk$ расстояний в $\mathbb{R}^p$, то есть $O(nkp)$
операций.

На рисунке четыре итерации на наборе сферических кластеров; два из трех
начальных центров выбраны в одном кластере. Столбцы — номера итераций.
В верхнем ряду — шаг отнесения: фон окрашен по ближайшему центру
(ячейки Вороного), крестами отмечены текущие центры. В нижнем ряду — шаг
пересчета: серые кресты — центры до пересчета, черные — после, стрелки
показывают их перемещение. В заголовках — значение $J$ после шага.

In [ ]:
start_indices = np.r_[np.flatnonzero(y_balls == 0)[:2], np.flatnonzero(y_balls == 1)[:1]]
centers = X_balls[start_indices].copy()
grid_x, grid_y = np.meshgrid(np.linspace(X_balls[:, 0].min() - 0.5, X_balls[:, 0].max() + 0.5, 300),
                             np.linspace(X_balls[:, 1].min() - 0.5, X_balls[:, 1].max() + 0.5, 300))
grid = np.c_[grid_x.ravel(), grid_y.ravel()]
fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
for step in range(4):
    distances = ((X_balls[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    assignment = distances.argmin(axis=1)
    inertia_assigned = distances[np.arange(len(X_balls)), assignment].sum()
    regions = ((grid[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2).argmin(axis=1).reshape(grid_x.shape)
    top = axes[0, step]
    top.contourf(grid_x, grid_y, regions, levels=[-0.5, 0.5, 1.5, 2.5], colors=list(CLUSTER_COLORS[:3]), alpha=0.15)
    plot_partition(top, X_balls, assignment, f"итерация {step + 1}, отнесение: J = {inertia_assigned:.1f}")
    top.scatter(centers[:, 0], centers[:, 1], s=220, marker="X", color=BLACK)

    new_centers = np.array([X_balls[assignment == j].mean(axis=0) for j in range(3)])
    inertia_updated = ((X_balls - new_centers[assignment]) ** 2).sum()
    bottom = axes[1, step]
    plot_partition(bottom, X_balls, assignment, f"итерация {step + 1}, пересчет: J = {inertia_updated:.1f}")
    bottom.scatter(centers[:, 0], centers[:, 1], s=120, marker="X", color=GREY)
    bottom.scatter(new_centers[:, 0], new_centers[:, 1], s=220, marker="X", color=BLACK)
    for old, new in zip(centers, new_centers):
        bottom.annotate("", xy=new, xytext=old, arrowprops={"arrowstyle": "->", "color": BLACK, "lw": 1.5})
    centers = new_centers
axes[0, 0].set_ylabel("отнесение к центрам", fontsize=12)
axes[1, 0].set_ylabel("пересчет центров", fontsize=12)
plt.tight_layout()
plt.show()

**Выбор начальных центров.** При случайном выборе два начальных центра могут
оказаться в одном кластере, и алгоритм сходится к плохому локальному минимуму.
Метод **k-means++** выбирает первый центр равновероятно среди объектов, а каждый
следующий — с вероятностью, пропорциональной $D(x)^2$, квадрату расстояния
от объекта до ближайшего из уже выбранных центров. Объекты еще не покрытого
кластера имеют большое $D(x)^2$, поэтому следующий центр с высокой вероятностью
попадает в такой кластер. Для этого способа доказана оценка (Артур
и Васильвицкий, 2007): математическое ожидание инерции после выбора центров,
еще до итераций Ллойда, не превосходит $8(\ln k + 2)\,J^*$, где $J^*$ — глобальный минимум.

На наборе из десяти кластеров сравнивают инерцию после одного запуска
со случайными центрами и с k-means++, по 100 запусков каждого способа.

In [ ]:
X_ten, _ = make_blobs(n_samples=1500, centers=10, cluster_std=0.6, random_state=3)
init_inertia = {
    init: [KMeans(n_clusters=10, init=init, n_init=1, random_state=seed).fit(X_ten).inertia_ for seed in range(100)]
    for init in ["random", "k-means++"]
}
plt.figure(figsize=(8, 3.8))
for (init, values), color in zip(init_inertia.items(), [GREY, BLUE]):
    plt.hist(values, bins=30, alpha=0.7, color=color, label=f"{init}: медиана {np.median(values):.0f}")
plt.xlabel("инерция после одного запуска")
plt.ylabel("число запусков")
plt.legend()
plt.show()

При k-means++ инерция в среднем меньше: алгоритм реже останавливается в плохом
минимуме. В `KMeans` по умолчанию используют k-means++, а параметр `n_init`
задает число запусков с разными начальными центрами, из которых оставляют
результат с наименьшей инерцией.

**Выбор числа кластеров.** Наименьшая инерция не возрастает с ростом $k$: решение
с $k$ центрами плюс один центр в любом объекте дает $J$ не больше прежнего;
при $k = n$ инерция равна нулю. Поэтому строят график инерции от $k$ и выбирают
значение, после которого убывание резко замедляется (метод локтя, elbow
method), либо значение с наибольшим силуэтом. Пример — набор из четырех
кластеров.

In [ ]:
X_four, _ = make_blobs(n_samples=600, centers=4, cluster_std=1.0, random_state=11)
k_values = range(2, 10)
k_fits = [KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_four) for k in k_values]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(k_values, [fit.inertia_ for fit in k_fits], "o-", color=BLUE)
axes[0].set_ylabel("инерция")
axes[1].plot(k_values, [silhouette_score(X_four, fit.labels_) for fit in k_fits], "o-", color=OCHRE)
axes[1].set_ylabel("силуэт")
for ax in axes:
    ax.axvline(4, color=GREY, ls="--", lw=1)
    ax.set_xlabel("число кластеров k")
plt.tight_layout()
plt.show()

Излом инерции и максимум силуэта приходятся на $k = 4$. На реальных данных
эти критерии часто не совпадают, и число кластеров выбирают также
из содержательных соображений.

**Ограничения метода.** Кластеры — ячейки Вороного, а граница между двумя
кластерами — серединный перпендикуляр к отрезку между центрами, не зависящий
от разброса кластеров. На рисунке результат метода с истинным числом
кластеров на всех пяти наборах.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
for ax, (name, (X, y)) in zip(axes, datasets.items()):
    labels = KMeans(n_clusters=len(np.unique(y)), n_init=10, random_state=SEED).fit_predict(X)
    plot_partition(ax, X, labels, f"{name}: ARI {adjusted_rand_score(y, labels):.2f}")
plt.tight_layout()
plt.show()

Сферические кластеры найдены верно. Вытянутые кластеры, полумесяцы и кольца
разделены прямыми линиями неверно, так как кластеры метода выпуклы.
У кластеров разной плотности часть точек широкого кластера отнесена к соседним,
так как граница проходит посередине между центрами.

### Агломеративная кластеризация (agglomerative clustering)

Алгоритм:

1. каждый объект образует отдельный кластер; вычисляют матрицу попарных
   расстояний между кластерами;
2. находят пару кластеров с наименьшим расстоянием и объединяют ее;
3. вычисляют расстояния от нового кластера до остальных;
4. повторяют шаги 2 и 3, пока не останется один кластер, и запоминают
   расстояние каждого объединения.

Разбиение на $k$ кластеров — состояние после $n - k$ объединений. Расстояние между
кластерами $A$ и $B$ определяют одним из способов:

- метод одиночной связи (single linkage): $d(A, B) = \min_{a \in A,\, b \in B} \rho(a, b)$;
- метод полной связи (complete linkage): $d(A, B) = \max_{a \in A,\, b \in B} \rho(a, b)$;
- метод средней связи (average linkage): $d(A, B) = \frac{1}{|A|\,|B|} \sum_{a \in A} \sum_{b \in B} \rho(a, b)$;
- метод Уорда (Ward's method): $\Delta J(A, B) = J(A \cup B) - J(A) - J(B)$ — прирост инерции
  при объединении, где $J(A) = \sum_{x \in A}\|x - \mu_A\|^2$.

**Прирост инерции.** По разложению из раздела о методе k-средних для любой точки $z$
выполнено $\sum_{x \in A}\|x - z\|^2 = J(A) + |A|\,\|\mu_A - z\|^2$. Центр объединения
$\mu_{AB} = \frac{|A|\mu_A + |B|\mu_B}{|A| + |B|}$; подстановка $z = \mu_{AB}$ для $A$ и для $B$ дает
$J(A \cup B) = J(A) + |A|\,\|\mu_A - \mu_{AB}\|^2 + J(B) + |B|\,\|\mu_B - \mu_{AB}\|^2$. Кроме того,
$\mu_A - \mu_{AB} = \frac{|B|}{|A| + |B|}(\mu_A - \mu_B)$ и $\mu_B - \mu_{AB} = \frac{|A|}{|A| + |B|}(\mu_B - \mu_A)$. Подстановка:

$$\Delta J = |A|\,\|\mu_A - \mu_{AB}\|^2 + |B|\,\|\mu_B - \mu_{AB}\|^2
= \frac{|A|\,|B|^2 + |B|\,|A|^2}{(|A| + |B|)^2}\,\|\mu_A - \mu_B\|^2 = \frac{|A|\,|B|}{|A| + |B|}\,\|\mu_A - \mu_B\|^2.$$

Метод Уорда на каждом шаге выбирает объединение с наименьшим приростом
инерции, то есть жадно минимизирует критерий метода k-средних, поэтому
дает сходные разбиения.

**Формула Ланса — Уильямса** (Lance — Williams formula). Пересчет на шаге 3 не требует
исходных объектов: для всех четырех способов

$$d(A \cup B, C) = \alpha_A\,d(A, C) + \alpha_B\,d(B, C) + \beta\,d(A, B) + \gamma\,\big|d(A, C) - d(B, C)\big|$$

с коэффициентами:

| способ | $\alpha_A$ | $\alpha_B$ | $\beta$ | $\gamma$ |
|---|---|---|---|---|
| одиночная связь | $\frac{1}{2}$ | $\frac{1}{2}$ | 0 | $-\frac{1}{2}$ |
| полная связь | $\frac{1}{2}$ | $\frac{1}{2}$ | 0 | $\frac{1}{2}$ |
| средняя связь | $\frac{\lvert A \rvert}{\lvert A \rvert + \lvert B \rvert}$ | $\frac{\lvert B \rvert}{\lvert A \rvert + \lvert B \rvert}$ | 0 | 0 |
| метод Уорда, для $\delta$ | $\frac{\lvert A \rvert + \lvert C \rvert}{N}$ | $\frac{\lvert B \rvert + \lvert C \rvert}{N}$ | $-\frac{\lvert C \rvert}{N}$ | 0 |

где $N = |A| + |B| + |C|$. Вывод коэффициентов:

- одиночная связь: минимум по $(A \cup B) \times C$ равен $\min\big(d(A, C), d(B, C)\big)$, и для $x \le y$
  выполнено $\frac{x + y}{2} - \frac{|x - y|}{2} = \frac{x + y - (y - x)}{2} = x = \min(x, y)$; для полной связи знак при
  модуле дает $\max(x, y)$;
- средняя связь: сумма расстояний по $(A \cup B) \times C$ равна сумме по $A \times C$ и по $B \times C$,
  то есть $|A|\,|C|\,d(A, C) + |B|\,|C|\,d(B, C)$; деление на $(|A| + |B|)\,|C|$ дает коэффициенты таблицы;
- метод Уорда: формулу записывают для $\delta(A, B) = 2\Delta J(A, B) = \frac{2|A||B|}{|A| + |B|}\|\mu_A - \mu_B\|^2$;
  для одноэлементных кластеров $\delta = \|a - b\|^2$.

**Вывод формулы для метода Уорда.** Обозначают $u = \mu_A - \mu_C$, $w = \mu_B - \mu_C$, $N_{AB} = |A| + |B|$. Тогда
$\mu_{AB} - \mu_C = \frac{|A| u + |B| w}{N_{AB}}$, и с учетом $2\langle u, w \rangle = \|u\|^2 + \|w\|^2 - \|u - w\|^2$, $u - w = \mu_A - \mu_B$:

$$\|\mu_{AB} - \mu_C\|^2 = \frac{|A|^2\|u\|^2 + |B|^2\|w\|^2 + |A||B|\big(\|u\|^2 + \|w\|^2 - \|\mu_A - \mu_B\|^2\big)}{N_{AB}^2}
= \frac{|A|\,\|u\|^2 + |B|\,\|w\|^2}{N_{AB}} - \frac{|A||B|\,\|\mu_A - \mu_B\|^2}{N_{AB}^2}.$$

Умножение на $\frac{2 N_{AB} |C|}{N_{AB} + |C|}$ дает $\delta(A \cup B, C)$:

$$\delta(A \cup B, C) = \frac{1}{N}\Big[\,2|C||A|\,\|u\|^2 + 2|C||B|\,\|w\|^2 - |C|\,\frac{2|A||B|}{N_{AB}}\|\mu_A - \mu_B\|^2\Big].$$

По определению $\delta$: $2|C||A|\,\|u\|^2 = (|A| + |C|)\,\delta(A, C)$, $2|C||B|\,\|w\|^2 = (|B| + |C|)\,\delta(B, C)$
и $\frac{2|A||B|}{N_{AB}}\|\mu_A - \mu_B\|^2 = \delta(A, B)$. Итого

$$\delta(A \cup B, C) = \frac{(|A| + |C|)\,\delta(A, C) + (|B| + |C|)\,\delta(B, C) - |C|\,\delta(A, B)}{|A| + |B| + |C|}.$$

В SciPy высота объединения для метода Уорда равна $\sqrt{\delta} = \sqrt{2\Delta J}$.

**Сложность.** Простая реализация делает $n - 1$ шагов с поиском минимума
среди $O(n^2)$ пар: $O(n^3)$ операций и $O(n^2)$ памяти. Алгоритмы с цепочками ближайших
соседей (nearest-neighbor chain) сокращают время до $O(n^2)$; память $O(n^2)$ остается
ограничением метода.

Функция `lance_williams_heights` выполняет алгоритм по шагам 1–4 с пересчетом
по формуле Ланса — Уильямса на 30 объектах; высоты объединений сравнивают
с функцией `linkage` из SciPy. Для метода Уорда дополнительно проверяют
равенство $h^2 = 2\Delta J$ для каждого объединения.

In [ ]:
def lance_williams_heights(X, method):
    """Высоты объединений агломеративной кластеризации с пересчетом по формуле Ланса — Уильямса."""
    base = squareform(pdist(X))
    if method == "ward":
        base = base ** 2
    distance_between = {(i, j): base[i, j] for i in range(len(X)) for j in range(i + 1, len(X))}
    sizes = {i: 1 for i in range(len(X))}
    heights, next_id = [], len(X)
    while len(sizes) > 1:
        a, b = min(distance_between, key=distance_between.get)
        d_ab = distance_between[a, b]
        heights.append(np.sqrt(d_ab) if method == "ward" else d_ab)
        for c in set(sizes) - {a, b}:
            d_ac, d_bc = distance_between[min(a, c), max(a, c)], distance_between[min(b, c), max(b, c)]
            n_a, n_b, n_c = sizes[a], sizes[b], sizes[c]
            if method == "single":
                new = 0.5 * d_ac + 0.5 * d_bc - 0.5 * abs(d_ac - d_bc)
            elif method == "complete":
                new = 0.5 * d_ac + 0.5 * d_bc + 0.5 * abs(d_ac - d_bc)
            elif method == "average":
                new = (n_a * d_ac + n_b * d_bc) / (n_a + n_b)
            else:
                new = ((n_a + n_c) * d_ac + (n_b + n_c) * d_bc - n_c * d_ab) / (n_a + n_b + n_c)
            distance_between[c, next_id] = new
        distance_between = {pair: value for pair, value in distance_between.items() if a not in pair and b not in pair}
        sizes[next_id] = sizes.pop(a) + sizes.pop(b)
        next_id += 1
    return np.array(heights)


sample_30 = X_balls[np.random.default_rng(1).choice(len(X_balls), 30, replace=False)]
lw_rows = {}
for method in ["single", "complete", "average", "ward"]:
    difference = np.abs(np.sort(lance_williams_heights(sample_30, method)) - np.sort(linkage(sample_30, method=method)[:, 2])).max()
    lw_rows[method] = {"наибольшее расхождение высот с SciPy": difference}
print(pd.DataFrame(lw_rows).T.to_string())

In [ ]:
def inertia_of(points):
    inertia = ((points - points.mean(axis=0)) ** 2).sum()
    return inertia


ward_merges = linkage(sample_30, method="ward")
members = {i: [i] for i in range(len(sample_30))}
ward_errors = []
for step_index, (a, b, height, _) in enumerate(ward_merges):
    A, B = members.pop(int(a)), members.pop(int(b))
    delta_J = inertia_of(sample_30[A + B]) - inertia_of(sample_30[A]) - inertia_of(sample_30[B])
    ward_errors.append(abs(height ** 2 - 2 * delta_J))
    members[len(sample_30) + step_index] = A + B
print("наибольшее отклонение h² − 2ΔJ по всем объединениям:", max(ward_errors))

**Метод одиночной связи и минимальное остовное дерево** (minimum spanning tree, MST).
Метод одиночной связи объединяет два кластера с наименьшим расстоянием между
их точками, то есть добавляет кратчайшее ребро полного графа, соединяющее
разные кластеры. Это шаг алгоритма Краскала (Kruskal's algorithm) построения
минимального остовного дерева: ребра перебирают по возрастанию длины
и добавляют ребро, если оно соединяет разные компоненты. Следовательно:

- высоты объединений метода одиночной связи равны длинам ребер минимального
  остовного дерева;
- разбиение на $k$ кластеров — компоненты связности дерева после удаления $k - 1$
  самых длинных ребер.

Минимальное остовное дерево строят за $O(n^2)$ операций, поэтому метод одиночной
связи вычисляют быстрее остальных способов. Проверка на 30 объектах
и рисунок дерева для полумесяцев: удаление одного самого длинного ребра
(красное) разделяет дерево на две компоненты.

In [ ]:
mst_30 = minimum_spanning_tree(squareform(pdist(sample_30)))
print("наибольшее расхождение длин ребер MST и высот одиночной связи:",
      np.abs(np.sort(mst_30.data) - linkage(sample_30, method="single")[:, 2]).max())

moon_tree = minimum_spanning_tree(squareform(pdist(X_moon_clusters))).tocoo()
longest = np.argmax(moon_tree.data)
keep_edges = np.ones(len(moon_tree.data), dtype=bool)
keep_edges[longest] = False
pruned = csr_matrix((moon_tree.data[keep_edges], (moon_tree.row[keep_edges], moon_tree.col[keep_edges])), shape=moon_tree.shape)
_, tree_labels = connected_components(pruned, directed=False)
single_labels = AgglomerativeClustering(n_clusters=2, linkage="single").fit_predict(X_moon_clusters)

fig, ax = plt.subplots(figsize=(9, 5.5))
for i, j in zip(moon_tree.row[keep_edges], moon_tree.col[keep_edges]):
    ax.plot(*X_moon_clusters[[i, j]].T, color=GREY, lw=0.8)
ax.plot(*X_moon_clusters[[moon_tree.row[longest], moon_tree.col[longest]]].T, color="#E41A1C", lw=3)
ax.scatter(X_moon_clusters[:, 0], X_moon_clusters[:, 1], s=9, c=CLUSTER_COLORS[tree_labels], zorder=3)
ax.set_title(f"MST полумесяцев без самого длинного ребра: ARI с методом одиночной связи "
             f"{adjusted_rand_score(single_labels, tree_labels):.2f}, с истинным разбиением {adjusted_rand_score(y_moon_clusters, tree_labels):.2f}")
ax.set_xticks([])
ax.set_yticks([])
plt.show()

**Дендрограмма** (dendrogram) — дерево объединений: по горизонтали — объекты,
высота горизонтального отрезка равна расстоянию, при котором объединились
два кластера; разрез на заданной высоте дает разбиение. На рисунке дендрограмма
метода Уорда для 40 объектов набора сферических кластеров.

In [ ]:
sample = X_balls[np.random.default_rng(1).choice(len(X_balls), 40, replace=False)]
merges = linkage(sample, method="ward")
plt.figure(figsize=(11, 4))
dendrogram(merges, color_threshold=6, no_labels=True)
plt.axhline(6, color=OCHRE, ls="--", label="разрез: три кластера")
plt.ylabel("высота объединения √(2ΔJ), метод Уорда")
plt.legend()
plt.show()

Четыре способа определения расстояния сравнивают на двух наборах. Строки
рисунка соответствуют наборам: верхняя — полумесяцы, нижняя — кластеры
разной плотности. Столбцы соответствуют способам определения расстояния,
слева направо: одиночная, полная, средняя связь и метод Уорда. Цвет точки —
найденный кластер, в заголовке — ARI.

In [ ]:
linkage_names = {"single": "одиночная связь (single)", "complete": "полная связь (complete)",
                 "average": "средняя связь (average)", "ward": "метод Уорда (Ward)"}
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, name in enumerate(["полумесяцы", "разная плотность"]):
    X, y = datasets[name]
    for col, (method, method_name) in enumerate(linkage_names.items()):
        labels = AgglomerativeClustering(n_clusters=len(np.unique(y)), linkage=method).fit_predict(X)
        plot_partition(axes[row, col], X, labels, f"{method_name}: ARI {adjusted_rand_score(y, labels):.2f}")
    axes[row, 0].set_ylabel(name, fontsize=12)
plt.tight_layout()
plt.show()

Метод одиночной связи разделяет полумесяцы верно: самое длинное ребро
минимального остовного дерева соединяет дуги, а внутри дуг ребра короткие.
На кластерах разной плотности он дает ARI около нуля: самые длинные ребра
дерева ведут к отдельным удаленным точкам широкого кластера, и при $k = 3$
эти точки становятся отдельными кластерами, а основные кластеры остаются
соединенными. Методы полной, средней связи и Уорда учитывают все точки
кластеров, предпочитают компактные кластеры и разделяют полумесяцы неверно.

### Плотностная кластеризация с выделением шума (DBSCAN)

Метод DBSCAN (Эстер и соавторы, 1996) выделяет кластеры как области высокой
плотности. Для точки $x$ определяют ее $\varepsilon$-окрестность ($\varepsilon$-neighborhood):

$$N_\varepsilon(x) = \{x_j : \rho(x, x_j) \le \varepsilon\}.$$

Точка $x$ **ядровая** (core point), если $|N_\varepsilon(x)| \ge$ `min_samples`; сама точка входит в $N_\varepsilon(x)$.
Отношения между точками:

- $q$ **непосредственно достижима по плотности** (directly density-reachable) из $p$,
  если $p$ ядровая и $q \in N_\varepsilon(p)$;
- $q$ **достижима по плотности** (density-reachable) из $p$, если существует цепочка
  $p = p_1, p_2, \dots, p_m = q$, в которой каждая $p_{t+1}$ непосредственно достижима из $p_t$;
  все точки цепочки, кроме последней, ядровые;
- $p$ и $q$ **связаны по плотности** (density-connected), если обе достижимы
  по плотности из некоторой точки $o$.

**Кластер** — непустое множество $C$, удовлетворяющее двум условиям:

- максимальность: если $p \in C$ и $q$ достижима по плотности из $p$, то $q \in C$;
- связность: любые две точки $C$ связаны по плотности.

Неядровая точка, входящая в кластер, — **граничная** (border point); точка, не входящая
ни в один кластер, — **шумовая** (noise point), ей присваивают метку $-1$.

**Свойства.** Для ядровой точки $p$ множество точек, достижимых из нее по плотности,
является кластером, и каждый кластер получается так из любой своей ядровой
точки. Ядровые точки кластера — компонента связности графа на ядровых точках
с ребрами $\rho \le \varepsilon$; эти компоненты не зависят от порядка обхода. Граничная точка
может лежать в окрестностях ядровых точек двух кластеров; тогда ее относят
к кластеру, обход которого дошел до нее первым.

**Алгоритм:**

1. для каждой точки находят $N_\varepsilon(x)$ и отмечают ядровые точки;
2. берут ядровую точку, еще не отнесенную к кластеру, и создают новый кластер;
3. обходом в ширину добавляют в кластер все точки окрестностей ядровых точек
   кластера; окрестности граничных точек не раскрывают;
4. повторяют шаги 2 и 3, пока остаются неотнесенные ядровые точки; оставшиеся
   точки — шум.

Число кластеров заранее не задают, его определяет алгоритм. Сложность
определяется поиском окрестностей: с пространственным индексом (k-d дерево,
шаровое дерево) в среднем $O(n \log n)$, в худшем случае $O(n^2)$.

На рисунке типы точек на полумесяцах при $\varepsilon$ = 0.13 и `min_samples` = 5: ядровые точки
закрашены цветом кластера, граничные — кольца цвета кластера, шумовые —
черные крестики. Для трех точек разных типов нарисованы их $\varepsilon$-окрестности.
Ядровые точки определяют по определению через `radius_neighbors` и сравнивают
с атрибутом `core_sample_indices_`.

In [ ]:
types_eps = 0.13
neighborhoods = NearestNeighbors(radius=types_eps).fit(X_moon_clusters).radius_neighbors(X_moon_clusters, return_distance=False)
is_core = np.array([len(neighborhood) for neighborhood in neighborhoods]) >= 5
types_model = DBSCAN(eps=types_eps, min_samples=5).fit(X_moon_clusters)
types_labels = types_model.labels_
is_noise = types_labels == -1
is_border = ~is_core & ~is_noise
print("ядровые точки совпадают с core_sample_indices_:", set(np.flatnonzero(is_core)) == set(types_model.core_sample_indices_))

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(*X_moon_clusters[is_core].T, s=12, c=CLUSTER_COLORS[types_labels[is_core] % 10], label=f"ядровые: {is_core.sum()}")
ax.scatter(*X_moon_clusters[is_border].T, s=45, facecolors="none", edgecolors=CLUSTER_COLORS[types_labels[is_border] % 10],
           lw=1.3, label=f"граничные: {is_border.sum()}")
ax.scatter(*X_moon_clusters[is_noise].T, s=40, marker="x", color=BLACK, label=f"шумовые: {is_noise.sum()}")
for mask in (is_core, is_border, is_noise):
    example = np.flatnonzero(mask)[0]
    ax.add_patch(Circle(X_moon_clusters[example], types_eps, fill=False, color=BLACK, lw=1, ls="--"))
ax.set_aspect("equal")
ax.set_title(f"DBSCAN, ε = {types_eps}, min_samples = 5: кластеров {len(set(types_labels) - {-1})}")
ax.legend(loc="lower left")
ax.set_xticks([])
ax.set_yticks([])
plt.show()

При малом $\varepsilon$ разрывы между группами ядровых точек дробят дуги на несколько
кластеров. **Выбор $\varepsilon$.** Для каждой точки $x$ вычисляют расстояние $d_k(x)$ до ее $k$-го
ближайшего объекта при $k$ = `min_samples` (сама точка — первый) и строят график значений
$d_k$, упорядоченных по возрастанию. Точка ядровая тогда и только тогда, когда
$d_k(x) \le \varepsilon$, поэтому график показывает долю ядровых точек при каждом $\varepsilon$. Внутри
кластеров $d_k$ мало, у шумовых точек оно резко возрастает; $\varepsilon$ выбирают в области
излома графика.

In [ ]:
min_samples = 5
neighbor_distances, _ = NearestNeighbors(n_neighbors=min_samples).fit(X_moon_clusters).kneighbors(X_moon_clusters)
kth_distance = np.sort(neighbor_distances[:, -1])
chosen_eps = 0.2

plt.figure(figsize=(8, 3.6))
plt.plot(kth_distance, color=BLUE, lw=2)
plt.axhline(chosen_eps, color=OCHRE, ls="--", label=f"ε = {chosen_eps}: ядровых {np.mean(kth_distance <= chosen_eps):.0%}")
plt.axhline(types_eps, color=GREY, ls=":", label=f"ε = {types_eps}: ядровых {np.mean(kth_distance <= types_eps):.0%}")
plt.xlabel("номер точки после упорядочения")
plt.ylabel(f"d_k, k = {min_samples}")
plt.legend()
plt.show()

Результат DBSCAN на полумесяцах при трех значениях $\varepsilon$ и `min_samples` = 5;
шумовые точки показаны черным.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, eps in zip(axes, [0.08, chosen_eps, 0.6]):
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_moon_clusters)
    plot_partition(ax, X_moon_clusters, labels,
                   f"ε = {eps}: кластеров {len(set(labels) - {-1})}, шум {np.mean(labels == -1):.0%}, "
                   f"ARI {adjusted_rand_score(y_moon_clusters, labels):.2f}")
plt.tight_layout()
plt.show()

При $\varepsilon$ = 0.08 окрестности малы и лишь часть точек является ядровыми: дуги
распадаются на множество кластеров, значительная доля точек отнесена к шуму.
При $\varepsilon$ = 0.6 окрестности перекрываются между дугами, и все точки образуют
один кластер. Значение $\varepsilon$ = 0.2 из области излома разделяет дуги верно.

### Иерархический DBSCAN (HDBSCAN)

Одно значение $\varepsilon$ не подходит для кластеров разной плотности: при малом $\varepsilon$
разреженный кластер становится шумом, при большом плотные кластеры сливаются.
Метод HDBSCAN (hierarchical density-based spatial clustering of applications
with noise; Кампелло, Мулави и Сандер, 2013) рассматривает все значения $\varepsilon$
одновременно.

**Расстояние взаимной достижимости.** Ядровое расстояние $\mathrm{core}(x) = d_k(x)$ при $k$ = `min_samples`;
точка ядровая при данном $\varepsilon$ тогда и только тогда, когда $\mathrm{core}(x) \le \varepsilon$. Расстояние
взаимной достижимости (mutual reachability distance):

$$d_{\text{mr}}(a, b) = \max\big(\mathrm{core}(a),\ \mathrm{core}(b),\ \rho(a, b)\big).$$

Для двух точек $d_{\text{mr}}(a, b) \le \varepsilon$ тогда и только тогда, когда обе ядровые при этом $\varepsilon$
и $\rho(a, b) \le \varepsilon$. Следовательно, ядровые части кластеров DBSCAN при данном $\varepsilon$ —
компоненты связности графа с ребрами $d_{\text{mr}} \le \varepsilon$, то есть разрез на высоте $\varepsilon$
дендрограммы метода одиночной связи по расстоянию $d_{\text{mr}}$.

**Перебор значений $\varepsilon$.** Значения $\varepsilon$ не перебирают по сетке. Строят минимальное
остовное дерево графа, в котором длина ребра между точками $a$ и $b$ равна $d_{\text{mr}}(a, b)$,
и упорядочивают его $n - 1$ ребро по возрастанию длины. Разбиение меняется только
тогда, когда $\varepsilon$ достигает длины очередного ребра: это ребро соединяет две компоненты
в одну. Поэтому все значения $\varepsilon$ перебирают одним проходом по упорядоченным
ребрам — от самого короткого, ниже которого никакие две точки не соединены,
до самого длинного, выше которого все точки образуют один кластер. Последовательность
объединений — это дендрограмма, и ее разрез на высоте $\varepsilon$ дает результат DBSCAN
при этом $\varepsilon$. Построение дерева в sklearn требует $O(n^2)$ операций, поэтому HDBSCAN
работает дольше одного запуска DBSCAN, но заменяет подбор $\varepsilon$.

**Выбор кластеров.** Используют плотность $\lambda = 1/\varepsilon$. При росте $\lambda$ кластер дерева либо
делится на два дочерних, если каждый содержит не меньше `min_cluster_size` точек,
либо теряет отдельные точки, которые становятся шумом. Устойчивость кластера:

$$S(C) = \sum_{x \in C} \big(\lambda_x - \lambda_{\text{рожд}}(C)\big),$$

где $\lambda_{\text{рожд}}(C)$ — плотность, при которой кластер выделился, $\lambda_x$ — плотность, при которой
точка $x$ покинула кластер. Выбирают набор непересекающихся кластеров дерева
с наибольшей суммарной устойчивостью: снизу вверх кластер заменяют его
потомками, если сумма их устойчивостей больше. Выбранные кластеры могут
соответствовать разным $\varepsilon$, поэтому метод находит кластеры разной плотности
без параметра $\varepsilon$. На рисунке DBSCAN при $\varepsilon$ = 0.2 и HDBSCAN при `min_cluster_size` = 15
на кластерах разной плотности и на кольцах. Кольца содержат по 250 точек,
а длина внешнего кольца вдвое больше, поэтому оно вдвое реже внутреннего.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for row, name in enumerate(["разная плотность", "кольца"]):
    X, y = datasets[name]
    for col, (method_name, model) in enumerate([("DBSCAN, ε = 0.2", DBSCAN(eps=0.2, min_samples=5)),
                                                ("HDBSCAN, min_cluster_size = 15", HDBSCAN(min_cluster_size=15))]):
        labels = model.fit_predict(X)
        plot_partition(axes[row, col], X, labels,
                       f"{method_name}: кластеров {len(set(labels) - {-1})}, шум {np.mean(labels == -1):.0%}, ARI {adjusted_rand_score(y, labels):.2f}")
    axes[row, 0].set_ylabel(name, fontsize=12)
plt.tight_layout()
plt.show()

На кольцах DBSCAN с $\varepsilon$ = 0.2 разбивает разреженное внешнее кольцо на части,
а HDBSCAN выделяет оба кольца: для внешнего кольца выбран кластер с меньшей
плотностью $\lambda$, чем для внутреннего. На кластерах разной плотности HDBSCAN
находит три кластера без подбора $\varepsilon$, тогда как DBSCAN при $\varepsilon$ = 0.2 выделяет
еще два мелких фрагмента; значения ARI близки, так как оба метода относят
к шуму периферию широкого кластера.

### Смесь гауссовых распределений (Gaussian mixture model, GMM)

**Модель.** Объект порождается в два этапа: сначала выбирают компоненту
$z \in \{1, \dots, k\}$ с вероятностями $\pi_1, \dots, \pi_k$, затем $x \sim \mathcal{N}(\mu_z, \Sigma_z)$. Компонента $z$
не наблюдается — это скрытая переменная (latent variable). Плотность объекта —
сумма по значениям $z$:

$$p(x) = \sum_{j=1}^{k} \pi_j \, \mathcal{N}(x \mid \mu_j, \Sigma_j), \qquad
\mathcal{N}(x \mid \mu, \Sigma) = \frac{1}{(2\pi)^{p/2} |\Sigma|^{1/2}} \exp\Big(-\frac{1}{2}(x - \mu)^\top \Sigma^{-1} (x - \mu)\Big).$$

Линии уровня плотности компоненты — эллипсы, оси которых направлены
по собственным векторам $\Sigma_j$, а полуоси пропорциональны корням собственных значений.

**Функция правдоподобия.** Параметры $\theta = \{\pi_j, \mu_j, \Sigma_j\}$ находят максимизацией

$$\ell(\theta) = \sum_{i=1}^{n} \ln \sum_{j=1}^{k} \pi_j\,\mathcal{N}(x_i \mid \mu_j, \Sigma_j).$$

Логарифм суммы не распадается на слагаемые, и приравнивание производных
к нулю не дает явного решения. Кроме того, $\ell$ не ограничена сверху: если $\mu_j = x_i$
и $\Sigma_j = \sigma^2 I$ с $\sigma \to 0$, слагаемое объекта $i$ растет как $-p \ln \sigma$. Поэтому к ковариационным
матрицам прибавляют $10^{-6} I$ (параметр `reg_covar` в sklearn).

**EM-алгоритм** (expectation — maximization, ожидание — максимизация) находит
локальный максимум $\ell(\theta)$, чередуя два шага:

1. E-шаг: при текущих параметрах для каждого объекта вычисляют **ответственности**
   (responsibilities) — вероятности принадлежности компонентам по формуле Байеса:
   $r_{ij} = \frac{\pi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}{\sum_{l} \pi_l \mathcal{N}(x_i \mid \mu_l, \Sigma_l)}$;
2. M-шаг: при фиксированных ответственностях пересчитывают параметры как долю,
   среднее и ковариационную матрицу объектов с весами $r_{ij}$; при $N_j = \sum_i r_{ij}$:
   $\pi_j = \frac{N_j}{n}$, $\mu_j = \frac{1}{N_j}\sum_i r_{ij}\,x_i$, $\Sigma_j = \frac{1}{N_j}\sum_i r_{ij}\,(x_i - \mu_j)(x_i - \mu_j)^\top$.

На каждой итерации правдоподобие не убывает, а найденный максимум зависит
от начальных параметров. Если все ковариационные матрицы одинаковые сферические,
а ответственности заменить отнесением к ближайшему центру, шаги E и M совпадают
с шагами алгоритма Ллойда метода k-средних.

**Виды ковариационных матриц.** Параметр `covariance_type` ограничивает форму $\Sigma_j$; число
параметров ковариаций при размерности $p$:

| `covariance_type` | форма | параметров ковариаций |
|---|---|---|
| `full` | своя произвольная $\Sigma_j$ | $k\,\frac{p(p + 1)}{2}$ |
| `tied` | одна общая $\Sigma$ | $\frac{p(p + 1)}{2}$ |
| `diag` | своя диагональная $\Sigma_j$ | $k p$ |
| `spherical` | $\Sigma_j = \sigma_j^2 I$ | $k$ |

Функция `em_gaussian_mixture` выполняет EM по формулам шагов E и M. Начальные
центры — $k$ случайных объектов, ковариации единичные, доли равные. Вычисления
ведут в логарифмах: $\ln \sum_j e^{a_j}$ вычисляют как $a_{\max} + \ln \sum_j e^{a_j - a_{\max}}$ (функция `logsumexp`),
чтобы избежать потери точности при малых плотностях.

In [ ]:
def log_gaussian(X, mean, covariance):
    """ln N(x | mean, covariance) для строк X через разложение Холецкого."""
    cholesky = np.linalg.cholesky(covariance)
    solved = np.linalg.solve(cholesky, (X - mean).T)
    log_density = -0.5 * (solved ** 2).sum(axis=0) - np.log(np.diag(cholesky)).sum() - 0.5 * X.shape[1] * np.log(2 * np.pi)
    return log_density


def em_gaussian_mixture(X, k, seed, n_iter=300, tol=1e-6, reg=1e-6):
    rng = np.random.default_rng(seed)
    means = X[rng.choice(len(X), k, replace=False)]
    covariances = np.array([np.eye(X.shape[1])] * k)
    weights = np.full(k, 1 / k)
    history = [(means.copy(), covariances.copy())]
    log_likelihoods = []
    for _ in range(n_iter):
        log_joint = np.column_stack([np.log(weights[j]) + log_gaussian(X, means[j], covariances[j]) for j in range(k)])
        log_total = logsumexp(log_joint, axis=1)
        log_likelihoods.append(log_total.sum())
        responsibilities = np.exp(log_joint - log_total[:, None])
        sizes = responsibilities.sum(axis=0)
        weights = sizes / len(X)
        means = responsibilities.T @ X / sizes[:, None]
        covariances = np.array([(responsibilities[:, j, None] * (X - means[j])).T @ (X - means[j]) / sizes[j]
                                + reg * np.eye(X.shape[1]) for j in range(k)])
        history.append((means.copy(), covariances.copy()))
        if len(log_likelihoods) > 1 and log_likelihoods[-1] - log_likelihoods[-2] < tol:
            break
    labels = responsibilities.argmax(axis=1)
    return labels, history, log_likelihoods


def draw_ellipses(ax, means, covariances):
    for mean, covariance in zip(means, covariances):
        values, vectors = np.linalg.eigh(covariance)
        angle = np.degrees(np.arctan2(vectors[1, -1], vectors[0, -1]))
        for scale in (1, 2):
            width, height = 2 * scale * np.sqrt(values[::-1])
            ax.add_patch(Ellipse(mean, width, height, angle=angle, fill=False, color=BLACK, lw=1))

EM на вытянутых кластерах из двух начальных состояний. На рисунке — параметры
после 0, 1, 3 и 8 итераций и итоговые для начального состояния 2; эллипсы — линии
уровня плотности компонент на расстоянии одного и двух стандартных
отклонений, цвет точек — истинный кластер.

In [ ]:
X_ellipse, y_ellipse = datasets["вытянутые"]
em_runs = {seed: em_gaussian_mixture(X_ellipse, 3, seed) for seed in (1, 2)}
_, history_2, log_likelihood_2 = em_runs[2]
shown = [0, 1, 3, 8, len(history_2) - 1]
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
for ax, iteration in zip(axes, shown):
    plot_partition(ax, X_ellipse, y_ellipse, f"итерация {iteration}")
    draw_ellipses(ax, *history_2[iteration])
    ax.scatter(history_2[iteration][0][:, 0], history_2[iteration][0][:, 1], marker="X", s=120, color=BLACK)
plt.tight_layout()
plt.show()

In [ ]:
gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=SEED).fit(X_ellipse)
sklearn_log_likelihood = gmm.score(X_ellipse) * len(X_ellipse)
plt.figure(figsize=(8.5, 4))
for (seed, (_, _, values)), color in zip(em_runs.items(), [OCHRE, BLUE]):
    plt.plot(np.arange(len(values)), values, color=color, lw=2, label=f"начальное состояние {seed}")
plt.axhline(sklearn_log_likelihood, color=BLACK, ls="--", lw=1, label="sklearn GaussianMixture")
plt.xlabel("итерация")
plt.ylabel("логарифм правдоподобия ℓ(θ)")
plt.legend()
plt.show()

em_rows = {}
for seed, (labels, _, values) in em_runs.items():
    em_rows[f"EM с нуля, начальное состояние {seed}"] = {
        "логарифм правдоподобия": values[-1], "ARI": adjusted_rand_score(y_ellipse, labels), "итераций": len(values)}
    print(f"начальное состояние {seed}: наименьшее приращение ℓ за итерацию {np.min(np.diff(values)):.1e}")
em_rows["sklearn GaussianMixture"] = {"логарифм правдоподобия": sklearn_log_likelihood,
                                      "ARI": adjusted_rand_score(y_ellipse, gmm.predict(X_ellipse)), "итераций": gmm.n_iter_}
show_table(pd.DataFrame.from_dict(em_rows, orient="index"), digits=3)

Правдоподобие возрастает на каждой итерации: наименьшее приращение, выведенное
над таблицей, положительно. Из начального состояния 2 алгоритм сходится к тому же максимуму,
что и sklearn (sklearn инициализирует центры методом k-средних). Из начального
состояния 1 он останавливается в локальном максимуме с меньшим правдоподобием,
и разбиение хуже; поэтому EM запускают из нескольких начальных состояний
(параметр `n_init`).

Сравнение с методом k-средних на тех же данных; на правой панели — вероятность
наиболее вероятной компоненты $\max_j r_{ij}$ для каждой точки.

In [ ]:
kmeans_labels = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(X_ellipse)
gmm_labels = gmm.predict(X_ellipse)
confidence = gmm.predict_proba(X_ellipse).max(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
plot_partition(axes[0], X_ellipse, kmeans_labels, f"метод k-средних (k-means): ARI {adjusted_rand_score(y_ellipse, kmeans_labels):.2f}")
plot_partition(axes[1], X_ellipse, gmm_labels, f"смесь гауссовых распределений (GMM): ARI {adjusted_rand_score(y_ellipse, gmm_labels):.2f}")
draw_ellipses(axes[1], gmm.means_, gmm.covariances_)
points = axes[2].scatter(X_ellipse[:, 0], X_ellipse[:, 1], s=7, c=confidence, cmap="Blues", vmin=0.33, vmax=1)
plt.colorbar(points, ax=axes[2], label="наибольшая ответственность")
axes[2].set_title("вероятность наиболее вероятной компоненты", fontsize=10)
plt.tight_layout()
plt.show()

Ответственности близки к 1 почти везде; меньше они только там, где компоненты
перекрываются.

**Выбор числа компонент** выполняют по байесовскому информационному
критерию (Bayesian information criterion, BIC):

$$\mathrm{BIC} = q \ln n - 2 \ln \hat{L},$$

где $\hat{L}$ — значение функции правдоподобия обученной модели, $q$ — число ее
параметров.

**Вывод.** Модели сравнивают по маргинальному правдоподобию
$p(X \mid M) = \int p(X \mid \theta)\,p(\theta)\,d\theta$. Вблизи максимума $\hat{\theta}$ логарифм правдоподобия раскладывают
до второго порядка: $\ln p(X \mid \theta) \approx \ln \hat{L} - \frac{1}{2}(\theta - \hat{\theta})^\top H (\theta - \hat{\theta})$, где $H$ — матрица
вторых производных $-\ln p(X \mid \theta)$ в $\hat{\theta}$. Для $n$ независимых объектов $H \approx n\,I_1$, где $I_1$ —
информационная матрица Фишера одного объекта. Гауссов интеграл
$\int e^{-\frac{1}{2}u^\top H u}\,du = (2\pi)^{q/2}|H|^{-1/2}$ и $|n I_1| = n^q |I_1|$ дают (приближение Лапласа)

$$\ln p(X \mid M) \approx \ln \hat{L} + \ln p(\hat{\theta}) + \frac{q}{2}\ln 2\pi - \frac{q}{2}\ln n - \frac{1}{2}\ln|I_1|
= \ln \hat{L} - \frac{q}{2}\ln n + O(1).$$

Умножение на $-2$ и отбрасывание слагаемых, не растущих с $n$, дает BIC; меньшее
значение соответствует большему маргинальному правдоподобию. Критерий
Акаике (Akaike information criterion, AIC) $2q - 2\ln \hat{L}$ штрафует сложность слабее.

**Число параметров смеси** с полными ковариационными матрицами: $k - 1$ долей
(сумма равна 1), $kp$ координат центров и $k\,\frac{p(p + 1)}{2}$ элементов симметричных
матриц; для $p = 2$: $q = (k - 1) + 2k + 3k = 6k - 1$. Число параметров и BIC проверяют
по формулам для всех видов ковариационных матриц на наборе из четырех
кластеров при $k = 4$.

In [ ]:
p_four, k_four, n_four = X_four.shape[1], 4, len(X_four)
covariance_parameters = {"full": k_four * p_four * (p_four + 1) // 2, "tied": p_four * (p_four + 1) // 2,
                         "diag": k_four * p_four, "spherical": k_four}
bic_rows = {}
for covariance_type, n_covariance in covariance_parameters.items():
    model = GaussianMixture(n_components=k_four, covariance_type=covariance_type, random_state=SEED).fit(X_four)
    q = (k_four - 1) + k_four * p_four + n_covariance
    log_likelihood = model.score(X_four) * n_four
    bic_rows[covariance_type] = {"q по формуле": q, "q в sklearn": model._n_parameters(),
                                 "BIC по формуле": q * np.log(n_four) - 2 * log_likelihood, "BIC в sklearn": model.bic(X_four)}
print(pd.DataFrame(bic_rows).T.to_string())

BIC в зависимости от числа компонент смеси с полными ковариационными
матрицами на том же наборе:

In [ ]:
components = range(1, 9)
bic = [GaussianMixture(n_components=k, random_state=SEED).fit(X_four).bic(X_four) for k in components]
best_components = components[int(np.argmin(bic))]
plt.figure(figsize=(8, 3.6))
plt.plot(components, bic, "o-", color=BLUE)
plt.axvline(best_components, color=OCHRE, ls="--", label=f"минимум при {best_components} компонентах")
plt.xlabel("число компонент")
plt.ylabel("BIC")
plt.legend()
plt.show()

### Сравнение методов в одном сетапе

Шесть методов применяют к пяти наборам с одинаковыми параметрами: число
кластеров равно истинному, у DBSCAN $\varepsilon$ = 0.2 и `min_samples` = 5, у HDBSCAN
`min_cluster_size` = 15. Для каждой пары «набор — метод» вычисляют все метрики
и время обучения вместе с предсказанием.

In [ ]:
methods = {
    "метод k-средних (k-means)": lambda k: KMeans(n_clusters=k, n_init=10, random_state=SEED),
    "метод Уорда (Ward)": lambda k: AgglomerativeClustering(n_clusters=k, linkage="ward"),
    "метод одиночной связи (single linkage)": lambda k: AgglomerativeClustering(n_clusters=k, linkage="single"),
    "DBSCAN": lambda k: DBSCAN(eps=0.2, min_samples=5),
    "HDBSCAN": lambda k: HDBSCAN(min_cluster_size=15),
    "смесь гауссовых распределений (GMM)": lambda k: GaussianMixture(n_components=k, random_state=SEED),
}

comparison_rows, comparison_labels = [], {}
for data_name, (X, y) in datasets.items():
    for method_name, make_method in methods.items():
        start = time.perf_counter()
        labels = make_method(len(np.unique(y))).fit_predict(X)
        seconds = time.perf_counter() - start
        comparison_labels[data_name, method_name] = labels
        comparison_rows.append({"набор": data_name, "метод": method_name, **clustering_scores(X, y, labels), "время, с": seconds})
comparison = pd.DataFrame(comparison_rows)


def comparison_table(values):
    table = comparison.pivot(index="набор", columns="метод", values=values).loc[list(datasets), list(methods)]
    return table

Строки рисунка — наборы данных, столбцы — методы, в заголовке — ARI;
шумовые точки DBSCAN и HDBSCAN показаны черным.

In [ ]:
fig, axes = plt.subplots(5, 6, figsize=(21, 16))
for row, (data_name, (X, y)) in enumerate(datasets.items()):
    for col, method_name in enumerate(methods):
        labels = comparison_labels[data_name, method_name]
        title = f"ARI {adjusted_rand_score(y, labels):.2f}"
        if row == 0:
            title = f"{method_name}\n{title}"
        plot_partition(axes[row, col], X, labels, title)
    axes[row, 0].set_ylabel(data_name, fontsize=12)
plt.tight_layout()
plt.show()

**Таблица ARI.** Строки — наборы данных, столбцы — методы; значение 1 означает
совпадение с истинным разбиением, значение около 0 — согласованность
на уровне случайного разбиения. Шкала заливки общая для всей таблицы.

In [ ]:
show_table(comparison_table("ARI"), axis=None)

**Таблица силуэтов.** Строки и столбцы те же; силуэт вычислен по найденному
разбиению без использования истинных меток.

In [ ]:
show_table(comparison_table("силуэт"), axis=None)

**Таблица времени** обучения вместе с предсказанием, в миллисекундах. Строки
и столбцы те же; ярче меньшее время.

In [ ]:
show_table(comparison_table("время, с") * 1000, digits=1, axis=None, lower_is_better=True)

Выводы по таблицам:

- метод k-средних и метод Уорда верно находят только сферические кластеры:
  оба минимизируют инерцию, и их кластеры близки к ячейкам Вороного;
- метод одиночной связи верно разделяет полумесяцы и кольца, где кластеры
  разделены промежутками, и дает ARI около нуля на кластерах разной плотности
  из-за отдельных удаленных точек;
- DBSCAN находит кластеры произвольной формы, но одно значение $\varepsilon$ не подходит
  для колец разной плотности;
- HDBSCAN без параметра $\varepsilon$ дает лучший или близкий к лучшему результат на всех
  наборах, кроме вытянутых, где часть точек отнесена к шуму;
- смесь гауссовых распределений лучше всех на вытянутых кластерах и кластерах
  разной плотности, так как каждая компонента имеет собственную ковариационную
  матрицу; невыпуклые кластеры она, как и метод k-средних, не находит;
- силуэт противоречит ARI на невыпуклых кластерах: на кольцах наибольший
  силуэт у метода k-средних с ARI около нуля, а у верных разбиений методом
  одиночной связи и HDBSCAN силуэт низкий, поэтому метрики без истинного
  разбиения оценивают форму кластеров, а не соответствие задаче;
- на 500 точках все методы работают за миллисекунды, и дольше всех метод
  k-средних из-за 10 запусков (`n_init`); на больших выборках определяющей
  становится асимптотическая сложность, приведенная в итогах.

## 6. Итоги

**Метод опорных векторов** выбирает разделяющую полосу наибольшей ширины
$\frac{2}{\|w\|}$; нарушения штрафуются hinge loss с весом $C$; границу определяют
только опорные векторы.

**Ядра.** Обучение и предсказание SVM зависят от объектов только через скалярные
произведения, поэтому их заменяют ядром $K(x, z)$ — скалярным произведением
в другом пространстве. У RBF-ядра параметр $\gamma$ обратен квадрату радиуса
влияния объекта; признаки перед ним стандартизуют.

**Метод главных компонент**: главные компоненты — собственные векторы
ковариационной матрицы, упорядоченные по убыванию собственных значений;
на практике их вычисляют через сингулярное разложение $X = U S V^\top$,
новые координаты — $U_k S_k$. Доля объясненной дисперсии определяет число
сохраняемых компонент.

**Визуализация.** В пространстве высокой размерности расстояния концентрируются,
поэтому на плоскости сохраняют соседей, а не расстояния:

- t-SNE подбирает $\sigma_i$ по перплексии, симметризует сходства, на плоскости
  использует функцию Стьюдента и минимизирует $\mathrm{KL}(P \,\|\, Q)$; градиент —
  сумма сил притяжения и отталкивания; раннее преувеличение в начале
  оптимизации усиливает притяжение и формирует компактные группы;
- UMAP строит взвешенный граф ближайших соседей с нормировкой $\log_2 k$ и нечетким
  объединением, начинает со спектрального вложения и минимизирует перекрестную
  энтропию стохастическим градиентом с отрицательной выборкой;
- размеры групп и расстояния между ними на таких рисунках не интерпретируют.

**Кластеризация.** Методы различаются тем, какие группы они считают кластерами:

| метод | какие кластеры находит | число кластеров | сложность | ограничение |
|---|---|---|---|---|
| метод k-средних | выпуклые ячейки Вороного | задают | $O(nkp)$ на итерацию | невыпуклые формы, разная плотность |
| метод Уорда | компактные, как у метода k-средних | задают или по дендрограмме | $O(n^2)$ памяти | невыпуклые формы |
| метод одиночной связи | связные по цепочкам близких точек | задают или по дендрограмме | $O(n^2)$, через MST | отдельные удаленные точки |
| DBSCAN | области плотности выше заданной | определяет алгоритм | $O(n \log n)$ в среднем | одно $\varepsilon$ для разной плотности |
| HDBSCAN | устойчивые области разной плотности | определяет алгоритм | $O(n \log n)$ в среднем | `min_cluster_size` |
| смесь гауссовых распределений | эллиптические, разного размера | задают или по BIC | $O(nkp^2)$ на итерацию | невыпуклые формы, локальные максимумы |

**Метрики.** При известном разбиении — ARI, NMI, однородность, полнота, V-мера;
они сравнивают, какие объекты отнесены к одному кластеру, и ARI с NMI имеют
поправку или нормировку, делающую их сравнимыми между разбиениями. Без
истинного разбиения — силуэт и индексы Калински — Харабаша и Дэвиса — Болдина;
они оценивают компактность и разделенность кластеров и выше для выпуклых
кластеров независимо от задачи.